# Qwen3-14B-IT Local Feature Homeostasis (LFH) on Natural Questions Document

Generated from the final LFH AGNews template. This variant uses ordered `Document N:` semantic blocks for Natural Questions Document and preserves the LFH/locality, attention-write, MLP-write, and causal-intervention analysis logic.

Base/raw model variants use plain prompts without chat-template tokens. Instruction/chat variants use one native user turn containing the full ordered list.


In [ ]:

import os
import re
import json
import math
import random
from collections import defaultdict
from typing import Any, Dict, List, Optional, Tuple, Sequence, Union

import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import hf_hub_download
from scipy.stats import pearsonr


# ============================================================
# 0. Reproducibility and experiment config
# ============================================================

def set_seed(seed: int):
    print("seed:", seed)
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.enabled = False
    torch.backends.cudnn.benchmark = False
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    os.environ["PYTHONHASHSEED"] = str(seed)


set_seed(42)

# Model/SAE/task variant generated from the final LFH template.
REQUESTED_MODEL_NAME = 'Qwen3-14B-IT'
MODEL_NAME = 'Qwen/Qwen3-14B'
MODEL_SHORT_NAME = 'Qwen3-14B-IT'
TARGET_LAYER = 20
SAE_BACKEND = 'qwen_batchtopk'

SAE_REPO_ID = 'adamkarvonen/qwen3-14b-saes'
SAE_TRAINER = 0 if SAE_BACKEND == "qwen_batchtopk" else None
SAE_BASE_DIR = None
SAE_WIDTH = None
SAE_AVERAGE_L0 = None
SAE_FILENAME = 'saes_Qwen_Qwen3-14B_batch_top_k/resid_post_layer_20/trainer_0/ae.pt'
SAE_LENS_RELEASE = None
SAE_LENS_SAE_ID = None
SAE_TOPK_K = None

# Task/list configuration.
TASK_TYPE = 'nq_document'
ITEM_WORD = 'Document'
ITEM_WORD_LOWER = 'document'
ITEM_WORD_PLURAL = 'documents'
PROMPT_DATASET_NAME = 'Natural Questions Document'

# Prompt family:
#   raw: plain continuation-style prompt. Used for base models; no chat tokens are inserted.
#   gemma_chat_single_user / llama_chat_single_user / qwen_chat_single_user:
#       one native chat user turn containing the whole ordered semantic-block list.
PROMPT_FORMAT = 'qwen_chat_single_user'
PROMPT_BOS_TOKEN = ''
TOKENIZE_ADD_SPECIAL_TOKENS = False
CHAT_TEMPLATE_KWARGS = {'enable_thinking': False}
CHAT_TEMPLATE_USE_SYSTEM_ROLE = True

# High-norm-token filtering is useful for the Qwen BatchTopK SAE suite and disabled by default elsewhere.
FILTER_HIGH_NORM_TOKENS_FOR_FEATURE_EXTRACTION = True
HIGH_NORM_FILTER_MULTIPLIER = 10.0
QWEN_HIGH_NORM_FILTER_MULTIPLIER = HIGH_NORM_FILTER_MULTIPLIER

DTYPE = torch.bfloat16
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Original LFH defaults are 50 x 50 = 2500 prompts.
NUM_TEST_QUERY = 30
NUM_PREVIOUS = 50
NUM_BLOCKS_TOTAL = 8
FREQ_THRESHOLD = 0.05

# --------------------------------------------------------------------
# LFH source/target block configuration
# --------------------------------------------------------------------
LFH_TARGET_BLOCK_NUM = 8
LFH_SOURCE_BLOCK_NUM = 7
LFH_LOCALITY_SWEEP_SOURCE_BLOCK_NUMS = [7, 6, 5]


def block_num_to_index(block_num: int, total_blocks: int = NUM_BLOCKS_TOTAL) -> int:
    """Convert a 1-indexed semantic-block number to a Python list index."""
    block_num = int(block_num)
    if block_num < 1 or block_num > int(total_blocks):
        raise ValueError(f"block_num must be in [1, {total_blocks}], got {block_num}.")
    return block_num - 1


LFH_SOURCE_BLOCK_INDEX = block_num_to_index(LFH_SOURCE_BLOCK_NUM)
LFH_TARGET_BLOCK_INDEX = block_num_to_index(LFH_TARGET_BLOCK_NUM)

# --------------------------------------------------------------------
# Block measurement configuration
# --------------------------------------------------------------------
# For BFCL/NQ list prompts we do not treat the chat turn terminator as part of
# any semantic block. Block boundaries are recovered from Tool/Document headers.
BLOCK_END_TOKEN = None
BLOCK_SCORE_MODE = 'selected_offsets_from_end'
BLOCK_TOKEN_OFFSETS_FROM_END = [-1]
# BLOCK_SCORE_MODE = 'All'
# BLOCK_TOKEN_OFFSETS_FROM_END = None
BLOCK_LAST_K_TOKENS = 1

QWEN_SYSTEM_PROMPT = "You are a careful assistant."
LLAMA_SYSTEM_PROMPT = "You are a careful assistant."
SYSTEM_PROMPT = (
    f"You are a careful assistant. You will read an ordered list of {ITEM_WORD_PLURAL}. "
    f"Track the sequence of {ITEM_WORD_PLURAL} carefully."
)

NOTE_FOR_THIS_VARIANT = 'LFH Natural Questions Document variant for Qwen3-14B-IT. Base/raw variants use plain prompts without chat-template tokens; instruction/chat variants use a single native user turn containing the ordered documents list.'
print(NOTE_FOR_THIS_VARIANT)

# ============================================================
# 1. Load model, tokenizer, and SAE
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)
model.eval()


def get_model_input_device(model):
    return model.get_input_embeddings().weight.device


def get_layer_device(model, layer_idx: int):
    block = model.get_submodule(f"model.layers.{layer_idx}")
    try:
        return next(block.parameters()).device
    except StopIteration:
        return get_model_input_device(model)


MODEL_INPUT_DEVICE = get_model_input_device(model)
TARGET_LAYER_DEVICE = get_layer_device(model, TARGET_LAYER)

print("REQUESTED_MODEL_NAME:", REQUESTED_MODEL_NAME)
print("MODEL_NAME:", MODEL_NAME)
print("MODEL_SHORT_NAME:", MODEL_SHORT_NAME)
print("TARGET_LAYER:", TARGET_LAYER)
print("MODEL_INPUT_DEVICE:", MODEL_INPUT_DEVICE)
print("TARGET_LAYER_DEVICE:", TARGET_LAYER_DEVICE)
print("SAE_BACKEND:", SAE_BACKEND)
print("SAE_REPO_ID:", SAE_REPO_ID)
print("SAE_FILENAME:", SAE_FILENAME)
print("SAE_LENS_RELEASE:", SAE_LENS_RELEASE)
print("SAE_LENS_SAE_ID:", SAE_LENS_SAE_ID)
print("PROMPT_FORMAT:", PROMPT_FORMAT)
print("BLOCK_END_TOKEN:", BLOCK_END_TOKEN)


class QwenBatchTopKSAE(torch.nn.Module):
    """
    Minimal inference wrapper for Adam Karvonen's Qwen3 BatchTopK SAEs.

    The checkpoint has dictionary_learning-style keys:
      b_dec, k, threshold, decoder.weight, encoder.weight, encoder.bias

    This wrapper exposes an SAE-Lens-like interface:
      W_enc: [d_model, n_features]
      W_dec: [n_features, d_model]
      b_enc: [n_features]
      b_dec: [d_model]
      process_sae_in(x)
      activation_fn(pre)
      encode(x)
      decode(acts)
    """

    def __init__(self, state_dict, device=DEVICE, dtype=DTYPE):
        super().__init__()
        self.device = torch.device(device)
        self.dtype = dtype

        if isinstance(state_dict, dict) and "state_dict" in state_dict:
            state_dict = state_dict["state_dict"]
        if isinstance(state_dict, dict) and "model" in state_dict and isinstance(state_dict["model"], dict):
            state_dict = state_dict["model"]

        b_dec = state_dict["b_dec"].detach()
        b_enc = state_dict.get("encoder.bias", state_dict.get("b_enc")).detach()
        enc_w = state_dict.get("encoder.weight", state_dict.get("W_enc")).detach()
        dec_w = state_dict.get("decoder.weight", state_dict.get("W_dec")).detach()
        threshold = state_dict["threshold"]
        threshold = threshold.detach() if torch.is_tensor(threshold) else torch.tensor(threshold)
        k = state_dict.get("k", None)
        if torch.is_tensor(k):
            k = int(k.item())

        d_model = int(b_dec.numel())
        n_features = int(b_enc.numel())

        if enc_w.shape == (n_features, d_model):
            W_enc = enc_w.T.contiguous()
        elif enc_w.shape == (d_model, n_features):
            W_enc = enc_w.contiguous()
        else:
            raise ValueError(
                f"Unexpected encoder.weight shape {tuple(enc_w.shape)} "
                f"for d_model={d_model}, n_features={n_features}"
            )

        if dec_w.shape == (d_model, n_features):
            W_dec = dec_w.T.contiguous()
        elif dec_w.shape == (n_features, d_model):
            W_dec = dec_w.contiguous()
        else:
            raise ValueError(
                f"Unexpected decoder.weight shape {tuple(dec_w.shape)} "
                f"for d_model={d_model}, n_features={n_features}"
            )

        self.register_buffer("W_enc", W_enc.to(device=self.device, dtype=self.dtype))
        self.register_buffer("W_dec", W_dec.to(device=self.device, dtype=self.dtype))
        self.register_buffer("b_enc", b_enc.to(device=self.device, dtype=self.dtype))
        self.register_buffer("b_dec", b_dec.to(device=self.device, dtype=self.dtype))
        self.register_buffer("threshold", threshold.to(device=self.device, dtype=self.dtype))

        self.k = k
        self.d_model = d_model
        self.n_features = n_features

    def process_sae_in(self, x: torch.Tensor) -> torch.Tensor:
        return x - self.b_dec

    def process_sae_out(self, x: torch.Tensor) -> torch.Tensor:
        return x

    def encode_pre(self, x: torch.Tensor) -> torch.Tensor:
        x = x.to(device=self.device, dtype=self.dtype)
        return self.process_sae_in(x) @ self.W_enc + self.b_enc

    def activation_fn(self, pre: torch.Tensor) -> torch.Tensor:
        return torch.where(pre > self.threshold, pre, torch.zeros_like(pre))

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.activation_fn(self.encode_pre(x))

    def decode(self, acts: torch.Tensor) -> torch.Tensor:
        acts = acts.to(device=self.device, dtype=self.dtype)
        return acts @ self.W_dec + self.b_dec


class GemmaScopeJumpReLUSAE(torch.nn.Module):
    """
    Minimal inference wrapper for Gemma Scope JumpReLU residual-stream SAEs.
    """

    def __init__(self, params, device=DEVICE, dtype=DTYPE):
        super().__init__()
        self.device = torch.device(device)
        self.dtype = dtype

        def get_param(name, *aliases):
            for key in (name,) + aliases:
                if key in params:
                    value = params[key]
                    if isinstance(value, np.ndarray):
                        value = torch.from_numpy(value)
                    return value.detach() if torch.is_tensor(value) else torch.tensor(value)
            raise KeyError(f"Missing SAE parameter {name!r}. Available keys: {sorted(params.keys())}")

        W_enc = get_param("W_enc")
        W_dec = get_param("W_dec")
        b_enc = get_param("b_enc")
        b_dec = get_param("b_dec")
        threshold = get_param("threshold", "thresholds")

        if W_enc.ndim != 2 or W_dec.ndim != 2:
            raise ValueError(f"Expected 2D W_enc/W_dec, got {tuple(W_enc.shape)} and {tuple(W_dec.shape)}")

        d_model = int(W_enc.shape[0])
        n_features = int(W_enc.shape[1])

        if W_dec.shape == (d_model, n_features):
            W_dec = W_dec.T.contiguous()
        if W_dec.shape != (n_features, d_model):
            raise ValueError(f"Expected W_dec shape {(n_features, d_model)}, got {tuple(W_dec.shape)}")
        if b_enc.numel() != n_features:
            raise ValueError(f"Expected b_enc length {n_features}, got {b_enc.numel()}")
        if b_dec.numel() != d_model:
            raise ValueError(f"Expected b_dec length {d_model}, got {b_dec.numel()}")
        if threshold.numel() != n_features:
            raise ValueError(f"Expected threshold length {n_features}, got {threshold.numel()}")

        self.register_buffer("W_enc", W_enc.contiguous().to(device=self.device, dtype=self.dtype))
        self.register_buffer("W_dec", W_dec.contiguous().to(device=self.device, dtype=self.dtype))
        self.register_buffer("b_enc", b_enc.reshape(-1).contiguous().to(device=self.device, dtype=self.dtype))
        self.register_buffer("b_dec", b_dec.reshape(-1).contiguous().to(device=self.device, dtype=self.dtype))
        self.register_buffer("threshold", threshold.reshape(-1).contiguous().to(device=self.device, dtype=self.dtype))

        self.k = None
        self.d_model = d_model
        self.n_features = n_features

    def process_sae_in(self, x: torch.Tensor) -> torch.Tensor:
        # Gemma Scope folds the pre-encoder bias into the encoder parameters.
        return x

    def process_sae_out(self, x: torch.Tensor) -> torch.Tensor:
        return x

    def encode_pre(self, x: torch.Tensor) -> torch.Tensor:
        x = x.to(device=self.device, dtype=self.dtype)
        return self.process_sae_in(x) @ self.W_enc + self.b_enc

    def activation_fn(self, pre: torch.Tensor) -> torch.Tensor:
        return torch.where(pre > self.threshold, torch.relu(pre), torch.zeros_like(pre))

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.activation_fn(self.encode_pre(x))

    def decode(self, acts: torch.Tensor) -> torch.Tensor:
        acts = acts.to(device=self.device, dtype=self.dtype)
        return acts @ self.W_dec + self.b_dec


class SAELensAdapter(torch.nn.Module):
    """
    Thin adapter exposing a consistent interface for SAELens-loaded SAEs.

    This keeps the rest of the LFH notebook independent of SAELens internals:
      W_enc, W_dec, b_enc, b_dec, threshold, k, encode_pre, activation_fn.
    """

    def __init__(self, saelens_sae, cfg_dict=None, device=DEVICE, dtype=DTYPE, fallback_k=None):
        super().__init__()
        self.saelens_sae = saelens_sae
        self.cfg_dict = cfg_dict or {}
        self.device = torch.device(device)
        self.dtype = dtype

        W_enc = self._get_tensor_attr(["W_enc", "encoder_weight", "W_E"])
        W_dec = self._get_tensor_attr(["W_dec", "decoder_weight", "W_D"])
        b_enc = self._get_tensor_attr(["b_enc", "encoder_bias", "b_E"], required=False)
        b_dec = self._get_tensor_attr(["b_dec", "decoder_bias", "b_D"], required=False)
        threshold = self._get_tensor_attr(["threshold", "thresholds", "jumprelu_threshold"], required=False)

        if W_enc.ndim != 2 or W_dec.ndim != 2:
            raise ValueError(f"Expected 2D W_enc/W_dec, got {tuple(W_enc.shape)} and {tuple(W_dec.shape)}")

        # Normalize common orientations to W_enc=[d_model,n_features], W_dec=[n_features,d_model].
        if W_enc.shape[0] == W_dec.shape[1] and W_enc.shape[1] == W_dec.shape[0]:
            pass
        elif W_enc.shape[1] == W_dec.shape[1]:
            W_enc = W_enc.T.contiguous()
        elif W_enc.shape[0] == W_dec.shape[0]:
            W_dec = W_dec.T.contiguous()
        else:
            raise ValueError(f"Could not infer SAELens W_enc/W_dec orientation: {tuple(W_enc.shape)}, {tuple(W_dec.shape)}")

        d_model = int(W_enc.shape[0])
        n_features = int(W_enc.shape[1])
        if W_dec.shape != (n_features, d_model):
            raise ValueError(f"Expected W_dec shape {(n_features, d_model)}, got {tuple(W_dec.shape)}")

        if b_enc is None:
            b_enc = torch.zeros(n_features)
        if b_dec is None:
            b_dec = torch.zeros(d_model)
        if threshold is None:
            threshold = torch.zeros(n_features)

        self.register_buffer("W_enc", W_enc.detach().contiguous().to(device=self.device, dtype=self.dtype))
        self.register_buffer("W_dec", W_dec.detach().contiguous().to(device=self.device, dtype=self.dtype))
        self.register_buffer("b_enc", b_enc.detach().reshape(-1).contiguous().to(device=self.device, dtype=self.dtype))
        self.register_buffer("b_dec", b_dec.detach().reshape(-1).contiguous().to(device=self.device, dtype=self.dtype))
        self.register_buffer("threshold", threshold.detach().reshape(-1).contiguous().to(device=self.device, dtype=self.dtype))

        self.d_model = d_model
        self.n_features = n_features
        self.k = self._infer_k(fallback_k=fallback_k)

        self.saelens_sae.to(self.device)
        self.saelens_sae.eval()

    def _cfg_get(self, key, default=None):
        if isinstance(self.cfg_dict, dict) and key in self.cfg_dict:
            return self.cfg_dict[key]
        cfg = getattr(self.saelens_sae, "cfg", None)
        if isinstance(cfg, dict) and key in cfg:
            return cfg[key]
        if cfg is not None and hasattr(cfg, key):
            return getattr(cfg, key)
        return default

    def _get_tensor_attr(self, names, required=True):
        for name in names:
            if hasattr(self.saelens_sae, name):
                value = getattr(self.saelens_sae, name)
                if isinstance(value, torch.nn.Parameter):
                    value = value.data
                if torch.is_tensor(value):
                    return value.detach().cpu()
        if required:
            raise AttributeError(f"Could not find any of {names} on SAELens SAE.")
        return None

    def _infer_k(self, fallback_k=None):
        for key in ["k", "topk", "top_k", "topk_k"]:
            value = self._cfg_get(key, None)
            if value is not None:
                try:
                    return int(value)
                except Exception:
                    pass
        return int(fallback_k) if fallback_k is not None else None

    def process_sae_in(self, x: torch.Tensor) -> torch.Tensor:
        x = x.to(device=self.device, dtype=self.dtype)
        fn = getattr(self.saelens_sae, "process_sae_in", None)
        if callable(fn):
            try:
                out = fn(x)
                if isinstance(out, tuple):
                    out = out[0]
                return out.to(device=self.device, dtype=self.dtype)
            except Exception:
                pass
        return x - self.b_dec

    def process_sae_out(self, x: torch.Tensor) -> torch.Tensor:
        fn = getattr(self.saelens_sae, "process_sae_out", None)
        if callable(fn):
            try:
                out = fn(x)
                if isinstance(out, tuple):
                    out = out[0]
                return out
            except Exception:
                pass
        return x

    def encode_pre(self, x: torch.Tensor) -> torch.Tensor:
        x = x.to(device=self.device, dtype=self.dtype)
        return self.process_sae_in(x) @ self.W_enc + self.b_enc

    def _topk_relu(self, pre: torch.Tensor) -> torch.Tensor:
        if self.k is None:
            return torch.relu(pre)
        k = min(int(self.k), pre.shape[-1])
        values, indices = torch.topk(torch.relu(pre), k=k, dim=-1)
        out = torch.zeros_like(pre)
        return out.scatter(dim=-1, index=indices, src=values)

    def activation_fn(self, pre: torch.Tensor) -> torch.Tensor:
        pre = pre.to(device=self.device, dtype=self.dtype)
        fn = getattr(self.saelens_sae, "activation_fn", None)
        if callable(fn):
            try:
                out = fn(pre)
                if isinstance(out, tuple):
                    out = out[0]
                if torch.is_tensor(out) and out.shape == pre.shape:
                    return out.to(device=self.device, dtype=self.dtype)
            except Exception:
                pass
        if self.threshold.numel() == pre.shape[-1] and torch.any(self.threshold.detach().float() != 0):
            return torch.where(pre > self.threshold, torch.relu(pre), torch.zeros_like(pre))
        return self._topk_relu(pre)

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.activation_fn(self.encode_pre(x))

    def decode(self, acts: torch.Tensor) -> torch.Tensor:
        acts = acts.to(device=self.device, dtype=self.dtype)
        return acts @ self.W_dec + self.b_dec


def load_qwen_batchtopk_sae(repo_id: str, filename: str, device=DEVICE, dtype=DTYPE):
    print("Downloading/loading Qwen BatchTopK SAE checkpoint:", repo_id, filename)
    path = hf_hub_download(repo_id=repo_id, filename=filename)
    try:
        state = torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        state = torch.load(path, map_location="cpu")
    sae = QwenBatchTopKSAE(state, device=device, dtype=dtype)
    print(
        f"Loaded SAE: d_model={sae.d_model}, n_features={sae.n_features}, "
        f"k={sae.k}, threshold={float(sae.threshold.detach().float().mean().cpu()):.4f}"
    )
    return sae


def load_gemma_scope_jumprelu_sae(repo_id: str, filename: str, device=DEVICE, dtype=DTYPE):
    print("Downloading/loading Gemma Scope SAE checkpoint:", repo_id, filename)
    path = hf_hub_download(repo_id=repo_id, filename=filename)
    with np.load(path, allow_pickle=False) as params_np:
        params = {k: torch.from_numpy(params_np[k]) for k in params_np.files}
    sae = GemmaScopeJumpReLUSAE(params, device=device, dtype=dtype)
    print(
        f"Loaded SAE: d_model={sae.d_model}, n_features={sae.n_features}, "
        f"mean_threshold={float(sae.threshold.detach().float().mean().cpu()):.4f}"
    )
    return sae


def load_saelens_sae(release: str, sae_id: str, device=DEVICE, dtype=DTYPE, fallback_k=None):
    try:
        from sae_lens import SAE
    except ModuleNotFoundError as e:
        raise ModuleNotFoundError(
            "This Llama Scope variant requires sae-lens. Install it with: pip install sae-lens"
        ) from e

    print("Downloading/loading SAELens SAE:", release, sae_id)
    loaded = SAE.from_pretrained(release=release, sae_id=sae_id, device=str(device))
    if isinstance(loaded, tuple):
        if len(loaded) == 3:
            sae_obj, cfg_dict, sparsity = loaded
        elif len(loaded) == 2:
            sae_obj, cfg_dict = loaded
            sparsity = None
        else:
            sae_obj = loaded[0]
            cfg_dict = {}
            sparsity = None
    else:
        sae_obj = loaded
        cfg_dict = getattr(sae_obj, "cfg", {})
        sparsity = None

    sae = SAELensAdapter(
        saelens_sae=sae_obj,
        cfg_dict=cfg_dict,
        device=device,
        dtype=dtype,
        fallback_k=fallback_k,
    )
    print(
        f"Loaded SAE: d_model={sae.d_model}, n_features={sae.n_features}, "
        f"k={sae.k}, release={release}, sae_id={sae_id}"
    )
    return sae


def load_configured_sae():
    if SAE_BACKEND == "qwen_batchtopk":
        return load_qwen_batchtopk_sae(
            repo_id=SAE_REPO_ID,
            filename=SAE_FILENAME,
            device=DEVICE,
            dtype=DTYPE,
        )
    if SAE_BACKEND == "gemma_scope_npz":
        return load_gemma_scope_jumprelu_sae(
            repo_id=SAE_REPO_ID,
            filename=SAE_FILENAME,
            device=DEVICE,
            dtype=DTYPE,
        )
    if SAE_BACKEND == "saelens_llama_scope":
        return load_saelens_sae(
            release=SAE_LENS_RELEASE,
            sae_id=SAE_LENS_SAE_ID,
            device=DEVICE,
            dtype=DTYPE,
            fallback_k=SAE_TOPK_K,
        )
    raise ValueError(f"Unknown SAE_BACKEND={SAE_BACKEND!r}")


sae = load_configured_sae()


# ============================================================
# 2. SAE and model utilities
# ============================================================

def clear_all_hooks(model: torch.nn.Module):
    for module in model.modules():
        module._forward_hooks.clear()
        module._forward_pre_hooks.clear()
        module._backward_hooks.clear()


@torch.no_grad()
def sae_logits_before_jumprelu(
    sae,
    x: torch.Tensor,
    filter_high_norm_tokens: bool = FILTER_HIGH_NORM_TOKENS_FOR_FEATURE_EXTRACTION,
) -> torch.Tensor:
    """
    Return SAE encoder pre-activations before thresholding.

    x shape can be (..., d_model). Output shape is (..., n_features).
    """
    x_sae = x.to(device=sae.device, dtype=sae.dtype)
    hidden_pre = sae.encode_pre(x_sae)

    if filter_high_norm_tokens and x_sae.ndim >= 3:
        norms = x_sae.detach().float().norm(dim=-1)  # [B, S]
        med = norms.median(dim=1, keepdim=True).values.clamp_min(1e-6)
        high_norm_mask = norms > (HIGH_NORM_FILTER_MULTIPLIER * med)
        hidden_pre = hidden_pre.masked_fill(high_norm_mask.to(hidden_pre.device)[..., None], 0.0)

    return hidden_pre


@torch.no_grad()
def sae_activations(
    sae,
    x: torch.Tensor,
    filter_high_norm_tokens: bool = FILTER_HIGH_NORM_TOKENS_FOR_FEATURE_EXTRACTION,
) -> torch.Tensor:
    pre = sae_logits_before_jumprelu(
        sae=sae,
        x=x,
        filter_high_norm_tokens=filter_high_norm_tokens,
    )
    return sae.activation_fn(pre)


@torch.no_grad()
def get_layer_output(model, input_ids, attention_mask, layer_idx: int):
    """
    Run a forward pass and capture the output tensor of model.layers[layer_idx].
    Returns shape [batch, seq, hidden].
    """
    captured = {}

    def hook_fn(module, inputs, output):
        captured["x"] = output[0].detach() if isinstance(output, tuple) else output.detach()
        return output

    block = model.get_submodule(f"model.layers.{layer_idx}")
    handle = block.register_forward_hook(hook_fn)

    try:
        _ = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
        )
    finally:
        handle.remove()

    if "x" not in captured:
        raise RuntimeError(f"Failed to capture output from layer {layer_idx}")

    return captured["x"]


In [ ]:

# ============================================================
# 3. Natural Questions Document 8-block LFH prompt generation
# ============================================================

from typing import Any, Dict, List, Optional, Tuple, Sequence, Union

prompt_template_agnews = None  # Backward-compatible placeholder.
train_data_agnews = None       # Backward-compatible placeholder.


def normalize_text(x: Any, max_chars: Optional[int] = None) -> str:
    s = str(x).replace("\n", " ").replace("\t", " ").strip()
    s = re.sub(r"\s+", " ", s)
    if max_chars is not None and len(s) > max_chars:
        # Do not append artificial ellipses before block headers; repeated "..."
        # creates an unintended boundary cue for LFH/DPE analyses.
        truncated = s[:max_chars].rstrip()
        last_space = truncated.rfind(" ")
        if last_space >= int(0.55 * max_chars):
            truncated = truncated[:last_space].rstrip()
        s = truncated
    return s


NQ_REPO_ID = "BeIR/nq"
NQ_CORPUS_CONFIG = "corpus"
NQ_QUERIES_CONFIG = "queries"
MAX_DOCUMENT_CHARS = 520
MAX_TITLE_CHARS = 120
MIN_DOCUMENT_CHARS = 60
DOC_SHUFFLE_BUFFER_SIZE = 50_000
QUERY_SHUFFLE_BUFFER_SIZE = 20_000
_DOCUMENT_POOL_CACHE = None
_QUERY_POOL_CACHE = None


def load_beir_nq_component(config_name: str, preferred_splits: List[str], streaming: bool = True):
    errors = []
    for split in preferred_splits:
        try:
            return load_dataset(NQ_REPO_ID, config_name, split=split, streaming=streaming)
        except Exception as e:
            errors.append((split, repr(e)))
    try:
        ds = load_dataset(NQ_REPO_ID, config_name, streaming=streaming)
        if hasattr(ds, "keys"):
            first_key = list(ds.keys())[0]
            print(f"Loaded {NQ_REPO_ID}/{config_name}; using split {first_key!r}.")
            return ds[first_key]
        return ds
    except Exception as e:
        errors.append(("DatasetDict fallback", repr(e)))
    msg = "\n".join([f"  {split}: {err}" for split, err in errors])
    raise RuntimeError(f"Could not load {NQ_REPO_ID}/{config_name}. Tried:\n{msg}")


def shuffled_take_records(ds, n: int, seed: int, buffer_size: int) -> List[Dict[str, Any]]:
    if n <= 0:
        return []
    try:
        rows = list(ds.shuffle(seed=seed, buffer_size=buffer_size).take(n))
    except TypeError:
        rows = list(ds.shuffle(seed=seed).select(range(n)))
    except Exception:
        rows = []
        for row in ds:
            rows.append(row)
            if len(rows) >= max(n, min(5 * n, 50_000)):
                break
        rng = random.Random(seed)
        rng.shuffle(rows)
        rows = rows[:n]
    if len(rows) < n:
        raise ValueError(f"Requested {n} rows but only obtained {len(rows)} rows.")
    return rows


def extract_query_text(row: Dict[str, Any]) -> str:
    for key in ["text", "query", "question", "title"]:
        val = row.get(key, None)
        if val is not None and len(str(val).strip()) > 0:
            return normalize_text(val)
    return normalize_text(row)


def extract_document_text(row: Dict[str, Any]) -> str:
    for key in ["text", "contents", "document", "passage", "body"]:
        val = row.get(key, None)
        if val is not None and len(str(val).strip()) > 0:
            return normalize_text(val, max_chars=MAX_DOCUMENT_CHARS)
    return normalize_text(row, max_chars=MAX_DOCUMENT_CHARS)


def extract_document_title(row: Dict[str, Any]) -> str:
    for key in ["title", "document_title", "doc_title"]:
        val = row.get(key, None)
        if val is not None and len(str(val).strip()) > 0:
            return normalize_text(val, max_chars=MAX_TITLE_CHARS)
    return ""


def compact_document(row: Dict[str, Any], idx: int) -> str:
    title = extract_document_title(row)
    text = extract_document_text(row)
    if title:
        return f"Document {idx}:\nTitle: {title}\nText: {text}"
    return f"Document {idx}:\nText: {text}"


def filter_document_rows(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    good = []
    seen = set()
    for row in rows:
        text = extract_document_text(row)
        if len(text) < MIN_DOCUMENT_CHARS:
            continue
        sig = (extract_document_title(row), text[:160])
        if sig in seen:
            continue
        seen.add(sig)
        good.append(row)
    return good


def build_document_pool(num_needed: int, seed: int = 44) -> List[Dict[str, Any]]:
    global _DOCUMENT_POOL_CACHE
    if _DOCUMENT_POOL_CACHE is not None and len(_DOCUMENT_POOL_CACHE) >= num_needed:
        return _DOCUMENT_POOL_CACHE[:num_needed]
    print("Loading BEIR-NQ corpus stream:", NQ_REPO_ID, NQ_CORPUS_CONFIG)
    corpus = load_beir_nq_component(NQ_CORPUS_CONFIG, preferred_splits=["corpus", "train"], streaming=True)
    raw_take = max(num_needed * 2, num_needed + 1000)
    rows = shuffled_take_records(corpus, n=raw_take, seed=seed, buffer_size=DOC_SHUFFLE_BUFFER_SIZE)
    rows = filter_document_rows(rows)
    if len(rows) < num_needed:
        raise ValueError(f"Only {len(rows)} usable documents after filtering; need {num_needed}. Try lowering MIN_DOCUMENT_CHARS or increasing raw_take.")
    _DOCUMENT_POOL_CACHE = rows
    print(f"Loaded {len(rows)} usable BEIR-NQ documents.")
    return rows[:num_needed]


def build_query_pool(num_needed: int, seed: int = 43) -> List[Dict[str, Any]]:
    global _QUERY_POOL_CACHE
    if _QUERY_POOL_CACHE is not None and len(_QUERY_POOL_CACHE) >= num_needed:
        return _QUERY_POOL_CACHE[:num_needed]
    print("Loading BEIR-NQ query stream:", NQ_REPO_ID, NQ_QUERIES_CONFIG)
    queries = load_beir_nq_component(NQ_QUERIES_CONFIG, preferred_splits=["queries", "train"], streaming=True)
    rows = shuffled_take_records(queries, n=num_needed, seed=seed, buffer_size=QUERY_SHUFFLE_BUFFER_SIZE)
    _QUERY_POOL_CACHE = rows
    print(f"Loaded {len(rows)} BEIR-NQ queries.")
    return rows


def render_item_prompt_body(items: List[Dict[str, Any]], query_row: Optional[Dict[str, Any]] = None) -> str:
    question = extract_query_text(query_row) if query_row is not None else ""
    header = (
        "You are given a question and an ordered list of retrieved Wikipedia documents.\n"
        "Read the documents in order. Do not answer yet; just track the sequence of documents.\n\n"
        f"Question:\n{question}\n\n"
        "Retrieved documents:\n"
    )
    blocks = [compact_document(p, i + 1) for i, p in enumerate(items)]
    return header + "\n\n".join(blocks) + "\n"


def get_item_pool_and_query_pool(num_test_query: int, num_previous: int):
    # Need enough documents for deterministic fixed contexts plus diverse varied-source samples.
    num_docs_needed = max(5000, int(num_test_query) * (int(num_previous) + NUM_BLOCKS_TOTAL + 20))
    documents = build_document_pool(num_docs_needed, seed=44)
    queries = build_query_pool(int(num_test_query), seed=43)
    return documents, queries



def ensure_single_bos_prefix(text: str, tokenizer) -> str:
    bos = globals().get("PROMPT_BOS_TOKEN", "") or ""
    if not bos:
        return text
    if text.startswith(bos):
        return text
    return bos + text


def _manual_qwen_chatml_render(system_content: str, user_content: str) -> str:
    return (
        f"<|im_start|>system\n{system_content}<|im_end|>\n"
        f"<|im_start|>user\n{user_content}<|im_end|>\n"
    )


def render_model_prompt(tokenizer, user_content: str, system_content: Optional[str] = None):
    """Render base models as plain text and chat models as a single native user turn."""
    system_content = system_content if system_content is not None else SYSTEM_PROMPT
    prompt_format = globals().get("PROMPT_FORMAT", "raw")

    # Base/raw models: no chat template and no chat tokens.
    if prompt_format == "raw":
        plain = f"{system_content}\n\n{user_content}" if system_content else user_content
        return ensure_single_bos_prefix(plain, tokenizer)

    # Instruction/chat models: one user message containing the whole ordered list.
    rendered_user_content = user_content
    messages = []
    if prompt_format == "gemma_chat_single_user":
        # Gemma chat has no system role; fold the instruction into the user message.
        rendered_user_content = f"{system_content}\n\n{user_content}" if system_content else user_content
        messages = [{"role": "user", "content": rendered_user_content}]
    elif prompt_format in {"llama_chat_single_user", "qwen_chat_single_user"}:
        if system_content:
            messages.append({"role": "system", "content": system_content})
        messages.append({"role": "user", "content": rendered_user_content})
    else:
        raise ValueError(f"Unknown PROMPT_FORMAT={prompt_format!r}")

    try:
        rendered = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
            **globals().get("CHAT_TEMPLATE_KWARGS", {}),
        )
        return ensure_single_bos_prefix(rendered, tokenizer)
    except TypeError:
        rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        return ensure_single_bos_prefix(rendered, tokenizer)
    except Exception as e:
        print("Warning: tokenizer.apply_chat_template failed. Falling back to a manual/simple prompt.")
        print("Reason:", repr(e))
        if prompt_format == "qwen_chat_single_user":
            return ensure_single_bos_prefix(_manual_qwen_chatml_render(system_content, rendered_user_content), tokenizer)
        return ensure_single_bos_prefix(f"{system_content}\n\n{user_content}", tokenizer)


def _sample_one_excluding(rng: random.Random, population_size: int, forbidden: set):
    if population_size <= len(forbidden):
        raise ValueError("Population too small for exclusion-based sampling.")
    while True:
        idx = rng.randrange(population_size)
        if idx not in forbidden:
            return idx


def build_prompts_all_agnews(
    train_data_agnews=None,
    prompt_template_agnews=None,
    num_test_query=25,
    num_previous=25,
    varied_block_num=None,
    target_block_num=None,
    test_query_indices=None,
):
    """
    Generic LFH prompt builder for ordered Tool/Document blocks.

    For each fixed target/query index:
      - the target semantic block is fixed at block 8;
      - all non-source context blocks are fixed;
      - the configured source block is resampled across variants.
    The returned variable names keep the AGNews suffix for compatibility with
    downstream LFH analysis cells.
    """
    varied_block_num = int(LFH_SOURCE_BLOCK_NUM if varied_block_num is None else varied_block_num)
    target_block_num = int(LFH_TARGET_BLOCK_NUM if target_block_num is None else target_block_num)
    if target_block_num != NUM_BLOCKS_TOTAL:
        raise NotImplementedError(f"This template fills block {NUM_BLOCKS_TOTAL} as the fixed target; got target_block_num={target_block_num}.")
    if not (1 <= varied_block_num < target_block_num):
        raise ValueError(f"varied_block_num must be before target block {target_block_num}, got {varied_block_num}.")

    item_pool, query_pool = get_item_pool_and_query_pool(num_test_query=int(num_test_query), num_previous=int(num_previous))
    n_items_total = len(item_pool)
    source_pos = block_num_to_index(varied_block_num, total_blocks=NUM_BLOCKS_TOTAL)
    num_context_blocks = NUM_BLOCKS_TOTAL - 1

    prompts_all_agnews = {i: [] for i in range(num_test_query)}
    prompt_bodies_all_agnews = {i: [] for i in range(num_test_query)}
    varied_source_indices_by_query = {}

    if test_query_indices is None:
        if TASK_TYPE == "nq_document":
            test_query_indices = list(range(int(num_test_query)))
        else:
            rng = random.Random(42)
            test_query_indices = rng.sample(range(n_items_total), int(num_test_query))
    else:
        test_query_indices = list(test_query_indices)[:int(num_test_query)]

    for i in range(int(num_test_query)):
        if i % 10 == 0:
            print(
                f"building {PROMPT_DATASET_NAME} prompts for query {i}; "
                f"varied source block={varied_block_num}, fixed target block={target_block_num}"
            )

        rng = random.Random(10_000 + i)
        query_row = query_pool[test_query_indices[i] % len(query_pool)] if TASK_TYPE == "nq_document" else None

        # Select the fixed target item for block 8.
        if TASK_TYPE == "bfcl_tool":
            target_item_idx = int(test_query_indices[i]) % n_items_total
        else:
            target_item_idx = _sample_one_excluding(rng, n_items_total, forbidden=set())

        # Context blocks 1..7. Keep all non-source context blocks fixed.
        forbidden = {target_item_idx}
        context_indices = []
        while len(context_indices) < num_context_blocks:
            idx = _sample_one_excluding(rng, n_items_total, forbidden | set(context_indices))
            context_indices.append(idx)

        fixed_context_indices = {idx for pos, idx in enumerate(context_indices) if pos != source_pos}
        used_varied_source_indices = set()
        varied_source_indices_by_query[i] = []

        for k in range(int(num_previous)):
            if k == 0:
                source_idx = context_indices[source_pos]
            else:
                source_idx = _sample_one_excluding(
                    rng,
                    n_items_total,
                    forbidden | fixed_context_indices | used_varied_source_indices,
                )
                context_indices[source_pos] = source_idx

            used_varied_source_indices.add(source_idx)
            varied_source_indices_by_query[i].append(source_idx)

            item_indices = list(context_indices) + [target_item_idx]
            items = [item_pool[idx] for idx in item_indices]
            prompt_body = render_item_prompt_body(items, query_row=query_row)
            prompt = render_model_prompt(tokenizer=tokenizer, user_content=prompt_body, system_content=SYSTEM_PROMPT)

            prompt_bodies_all_agnews[i].append(prompt_body)
            prompts_all_agnews[i].append(prompt)

    return prompts_all_agnews, prompt_bodies_all_agnews, test_query_indices, varied_source_indices_by_query


prompts_all_agnews, prompt_bodies_all_agnews, test_query_indices, varied_source_indices_by_query = build_prompts_all_agnews(
    train_data_agnews=train_data_agnews,
    prompt_template_agnews=prompt_template_agnews,
    num_test_query=NUM_TEST_QUERY,
    num_previous=NUM_PREVIOUS,
    varied_block_num=LFH_SOURCE_BLOCK_NUM,
    target_block_num=LFH_TARGET_BLOCK_NUM,
)

# Backward-compatible alias for older downstream scratch code.
varied_previous_indices_by_query = varied_source_indices_by_query

assert len(prompts_all_agnews) == NUM_TEST_QUERY
assert len(prompts_all_agnews[0]) == NUM_PREVIOUS

print("MODEL_SHORT_NAME:", MODEL_SHORT_NAME)
print("PROMPT_DATASET_NAME:", PROMPT_DATASET_NAME)
print("PROMPT_FORMAT:", PROMPT_FORMAT)
print("LFH source block:", LFH_SOURCE_BLOCK_NUM)
print("LFH target block:", LFH_TARGET_BLOCK_NUM)
print("Example rendered prompt head:")
print(prompts_all_agnews[0][0][:1600])


In [ ]:

# ============================================================
# 4. Block token-selection helpers with selectable scoring
# ============================================================

# Matches each semantic-block header. For these variants, blocks are headed as "Document N:".
EXAMPLE_HEADER_RE = re.compile(
    rf"(?m)^{re.escape(ITEM_WORD)}\s+\d+:?\s*\r?\n",
    flags=re.IGNORECASE,
)


class TokenSelection:
    """
    Token selection for one exemplar block.

    The object behaves like a legacy (start, end) span when unpacked, so older
    plotting/logging code can still do ``s, e = selection``. Actual analyses use
    ``selection.indices`` through token_selection_to_* helpers, which allows
    non-contiguous choices such as [-1, -3] if desired.
    """

    def __init__(self, indices, *, full_start=None, full_end=None, label=None):
        indices = tuple(sorted(dict.fromkeys(int(i) for i in indices)))
        if len(indices) == 0:
            raise ValueError("TokenSelection requires at least one token index.")
        self.indices = indices
        self.start = int(indices[0])
        self.end = int(indices[-1]) + 1
        self.full_start = int(full_start) if full_start is not None else self.start
        self.full_end = int(full_end) if full_end is not None else self.end
        self.label = label

    def __iter__(self):
        # Backward-compatible unpacking: s, e = selection
        yield self.start
        yield self.end

    def __getitem__(self, item):
        return (self.start, self.end)[item]

    def __len__(self):
        # Legacy span-like length. Use n_selected for number of selected tokens.
        return 2

    @property
    def n_selected(self):
        return len(self.indices)

    @property
    def is_contiguous(self):
        return self.n_selected == (self.end - self.start)

    def __repr__(self):
        idx = list(self.indices)
        if len(idx) > 10:
            idx_repr = f"{idx[:5]} ... {idx[-5:]}"
        else:
            idx_repr = repr(idx)
        return (
            f"TokenSelection(start={self.start}, end={self.end}, "
            f"n_selected={self.n_selected}, contiguous={self.is_contiguous}, "
            f"indices={idx_repr})"
        )


def token_selection_to_indices(selection):
    """Return selected token indices as a Python list."""
    if isinstance(selection, TokenSelection):
        return list(selection.indices)
    s, e = selection
    return list(range(int(s), int(e)))


def token_selection_to_torch_indices(selection, device=None):
    """Return selected token indices as a torch.LongTensor."""
    return torch.tensor(token_selection_to_indices(selection), dtype=torch.long, device=device)


def token_selection_bounds(selection):
    """Return the legacy [start, end) envelope for logging/plotting."""
    if isinstance(selection, TokenSelection):
        return int(selection.start), int(selection.end)
    s, e = selection
    return int(s), int(e)


def token_selection_mean(x: torch.Tensor, selection, dim: int = 0):
    """Mean over selected token positions along ``dim``."""
    idx = token_selection_to_torch_indices(selection, device=x.device)
    return x.index_select(dim, idx).mean(dim=dim)


def _normalize_block_score_mode(mode):
    """Normalize user-facing BLOCK_SCORE_MODE strings."""
    mode_str = str(mode).strip()
    mode_key = mode_str.lower().replace("-", "_").replace(" ", "_")

    # User-facing full-exemplar mode. "content" is kept as a backward-compatible alias.
    if mode_key in {"all", "all_tokens", "full", "full_block", "content"}:
        return "all"

    # New exact relative-position mode.
    if mode_key in {
        "selected_offsets_from_end",
        "select_offsets_from_end",
        "token_offsets_from_end",
        "selected_token_offsets_from_end",
        "selected_tokens_from_end",
        "selected_tokens",
        "offsets_from_end",
    }:
        return "selected_offsets_from_end"

    # Legacy last-K family of modes, including "last_k_with_end_of_turn".
    if mode_key.startswith("last_k"):
        return "last_k"

    raise ValueError(
        f"Unknown BLOCK_SCORE_MODE={mode!r}. Supported modes are 'selected_offsets_from_end', 'All', and legacy last_k* modes."
    )


def _normalize_offsets_from_end(offsets):
    """
    Validate exact token offsets from the end of each exemplar span.

    Negative offsets count backward from the exemplar end:
      -1 = final token, -2 = second-to-final token, etc.
    """
    if offsets is None:
        return None
    if isinstance(offsets, (int, np.integer)):
        offsets = [int(offsets)]
    offsets = [int(x) for x in list(offsets)]
    if len(offsets) == 0:
        raise ValueError("BLOCK_TOKEN_OFFSETS_FROM_END must be non-empty when using selected-offset mode.")
    if any(x >= 0 for x in offsets):
        raise ValueError(
            "BLOCK_TOKEN_OFFSETS_FROM_END expects negative integers only: "
            "-1 is the final token, -2 is the second-to-final token, etc."
        )
    return offsets


def _legacy_last_k_to_offsets(last_k_tokens: int):
    last_k_tokens = int(last_k_tokens)
    if last_k_tokens <= 0:
        raise ValueError(f"BLOCK_LAST_K_TOKENS must be positive for legacy last-K modes, got {last_k_tokens}.")
    return [-i for i in range(1, last_k_tokens + 1)]


def _get_block_scoring_config():
    """Read scoring options, with safe defaults for older cached notebooks."""
    raw_mode = globals().get("BLOCK_SCORE_MODE", "selected_offsets_from_end")
    mode = _normalize_block_score_mode(raw_mode)
    end_token = globals().get("BLOCK_END_TOKEN", None)

    if mode == "all":
        return mode, end_token, None

    if mode == "last_k":
        # Strict backward compatibility: legacy last_k* modes are controlled by
        # BLOCK_LAST_K_TOKENS, irrespective of BLOCK_TOKEN_OFFSETS_FROM_END.
        offsets = _legacy_last_k_to_offsets(globals().get("BLOCK_LAST_K_TOKENS", 8))
    else:
        explicit_offsets = globals().get("BLOCK_TOKEN_OFFSETS_FROM_END", None)
        if explicit_offsets is not None:
            offsets = _normalize_offsets_from_end(explicit_offsets)
        else:
            # Safe fallback for notebooks configured before BLOCK_TOKEN_OFFSETS_FROM_END existed.
            offsets = _legacy_last_k_to_offsets(globals().get("BLOCK_LAST_K_TOKENS", 8))

    if len(offsets) == 0:
        raise ValueError("No token offsets selected for block scoring.")

    return "selected_offsets_from_end", end_token, offsets


def _trim_trailing_chat_tail(prompt: str, start: int, end: int) -> int:
    """Exclude final chat-template terminators / assistant-generation prompts from the last semantic block."""
    tail_markers = [
        "<end_of_turn>",
        "<|eot_id|>",
        "<|im_end|>",
        "<|start_header_id|>assistant<|end_header_id|>",
        "<|im_start|>assistant",
    ]
    candidates = []
    for marker in tail_markers:
        pos = prompt.find(marker, start, end)
        if pos >= 0:
            candidates.append(pos)
    if candidates:
        end = min(candidates)
    return end


def find_example_block_char_spans(prompt: str, expected_num_blocks: int):
    """
    Find character spans of all semantic blocks headed by ITEM_WORD, e.g. Tool 1 or Document 1.

    For list-style BFCL/NQ prompts, chat-template terminators belong to the outer user turn,
    not to the final semantic block, so they are excluded from the last block span.
    """
    _, end_token, _ = _get_block_scoring_config()

    matches = list(EXAMPLE_HEADER_RE.finditer(prompt))
    if len(matches) != expected_num_blocks:
        raise ValueError(
            f"Expected {expected_num_blocks} {globals().get('ITEM_WORD', 'Item')} blocks, but found {len(matches)}.\n"
            f"Prompt head:\n{prompt[:800]}"
        )

    spans = []
    for i, m in enumerate(matches):
        start = m.end()
        next_header_start = matches[i + 1].start() if i + 1 < len(matches) else len(prompt)

        if end_token:
            end_token_start = prompt.rfind(end_token, start, next_header_start)
            if end_token_start >= 0:
                end = end_token_start + len(end_token)
            else:
                end = next_header_start
        else:
            end = next_header_start
            if i + 1 == len(matches):
                end = _trim_trailing_chat_tail(prompt, start, end)
            while end > start and prompt[end - 1].isspace():
                end -= 1
        spans.append((start, end))

    return spans


def char_span_to_token_span(offset_mapping, char_start: int, char_end: int):
    """
    Convert character span [char_start, char_end) into token span [tok_start, tok_end).
    """
    token_ids = []
    for tok_idx, (s, e) in enumerate(offset_mapping):
        # Skip padding / special tokens / empty offsets.
        if s == 0 and e == 0:
            continue
        if e > char_start and s < char_end:
            token_ids.append(tok_idx)

    if not token_ids:
        raise ValueError(f"Could not map char span [{char_start}, {char_end}) to tokens.")

    return token_ids[0], token_ids[-1] + 1


def _get_inline_end_token_ids(end_token: str | None):
    """Token IDs for the explicit inline block terminator."""
    if not end_token:
        return []
    ids = tokenizer.encode(end_token, add_special_tokens=False)
    if not ids:
        raise ValueError(f"Could not tokenize BLOCK_END_TOKEN={end_token!r}")
    return [int(x) for x in ids]


def _ensure_span_includes_inline_end_token(prompt: str, token_span, end_token: str | None):
    """
    Some fast tokenizers report offset (0, 0) for special tokens such as
    <|im_end|> or <end_of_turn>. In that case char-offset mapping can end just
    before the explicit terminator. This helper checks token IDs and extends the
    span by the terminator sequence if needed.
    """
    if not end_token:
        return token_span

    tok_start, tok_end = token_span
    end_ids = _get_inline_end_token_ids(end_token)
    if not end_ids:
        return token_span

    enc_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    if isinstance(enc_ids[0], list):
        enc_ids = enc_ids[0]
    enc_ids = [int(x) for x in enc_ids]
    L = len(end_ids)

    if tok_end - tok_start >= L and enc_ids[tok_end - L:tok_end] == end_ids:
        return tok_start, tok_end

    if enc_ids[tok_end:tok_end + L] == end_ids:
        return tok_start, tok_end + L

    for shift in range(1, 5):
        if enc_ids[tok_end + shift:tok_end + shift + L] == end_ids:
            return tok_start, tok_end + shift + L

    return tok_start, tok_end


def _select_token_offsets_from_end(tok_start: int, tok_end: int, offsets_from_end):
    """
    Convert negative offsets from a full exemplar span into exact token indices.

    Offsets that run before tok_start are ignored. Thus [-1, -2, -3] behaves
    like the old last-3 selection, but automatically clips on very short spans.
    """
    if tok_end <= tok_start:
        raise ValueError(f"Empty token span [{tok_start}, {tok_end}) cannot be scored.")

    selected = []
    for offset in offsets_from_end:
        idx = int(tok_end) + int(offset)  # offset is negative: -1 -> tok_end - 1
        if tok_start <= idx < tok_end:
            selected.append(idx)

    if len(selected) == 0:
        raise ValueError(
            f"No selected offsets {offsets_from_end} fall inside token span [{tok_start}, {tok_end})."
        )

    return selected


def get_example_block_token_spans(prompt: str, offset_mapping, expected_num_blocks: int):
    """
    Return token selections for all Example blocks in the final rendered prompt.

    If a block terminator is configured, the terminator is included in the full
    exemplar span before token selection is applied. If not, the full span ends
    before the next Example header.

    Supported BLOCK_SCORE_MODE values:
      - "selected_offsets_from_end": select exact relative token positions from
        BLOCK_TOKEN_OFFSETS_FROM_END. For example, [-1] selects only the final
        token and [-2] selects only the second-to-final token.
      - legacy "last_k...": if BLOCK_TOKEN_OFFSETS_FROM_END is None, this is
        converted to [-1, -2, ..., -K] using BLOCK_LAST_K_TOKENS.
      - "All": return all tokens in each exemplar span.

    The function name is kept for backward compatibility; returned objects are
    TokenSelection instances that can still be unpacked as legacy (start, end)
    spans for logging/plotting.
    """
    mode, end_token, offsets_from_end = _get_block_scoring_config()
    char_spans = find_example_block_char_spans(
        prompt,
        expected_num_blocks=expected_num_blocks,
    )

    full_token_spans = [
        _ensure_span_includes_inline_end_token(
            prompt,
            char_span_to_token_span(offset_mapping, s, e),
            end_token=end_token,
        )
        for (s, e) in char_spans
    ]

    selections = []
    for block_idx, (tok_start, tok_end) in enumerate(full_token_spans, start=1):
        if mode == "all":
            indices = list(range(tok_start, tok_end))
        elif mode == "selected_offsets_from_end":
            indices = _select_token_offsets_from_end(tok_start, tok_end, offsets_from_end)
        else:
            # _get_block_scoring_config should already have validated the mode.
            raise ValueError(f"Unknown normalized BLOCK_SCORE_MODE={mode!r}")

        selections.append(
            TokenSelection(
                indices,
                full_start=tok_start,
                full_end=tok_end,
                label=f"{globals().get('ITEM_WORD', 'Item')} {block_idx}",
            )
        )

    return selections


def tokenize_prompt_for_offsets(prompt: str):
    """
    Prompts are already rendered with the intended BOS/chat special tokens, so
    add_special_tokens is disabled by default to avoid duplicating them.
    """
    enc = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=bool(globals().get("TOKENIZE_ADD_SPECIAL_TOKENS", False)),
        return_offsets_mapping=True,
        padding=False,
        truncation=False,
    )
    return enc


# Sanity check on one prompt.
_test_prompt = prompts_all_agnews[0][0]
_test_enc = tokenize_prompt_for_offsets(_test_prompt)
_test_offsets = _test_enc["offset_mapping"][0].tolist()
_test_spans = get_example_block_token_spans(
    prompt=_test_prompt,
    offset_mapping=_test_offsets,
    expected_num_blocks=NUM_BLOCKS_TOTAL,
)
print(f"{globals().get('ITEM_WORD', 'Item')} block token selections under current scoring mode:")
print("  PROMPT_FORMAT:", globals().get("PROMPT_FORMAT", "raw"))
print("  BLOCK_SCORE_MODE:", globals().get("BLOCK_SCORE_MODE", "selected_offsets_from_end"))
print("  BLOCK_END_TOKEN:", globals().get("BLOCK_END_TOKEN", None))
_printed_mode, _, _printed_offsets = _get_block_scoring_config()
if _printed_mode == "all":
    print("  BLOCK_TOKEN_OFFSETS_FROM_END: not used because BLOCK_SCORE_MODE='All'")
else:
    print("  BLOCK_TOKEN_OFFSETS_FROM_END:", _printed_offsets)
for i, sel in enumerate(_test_spans, start=1):
    s, e = token_selection_bounds(sel)
    idx = token_selection_to_indices(sel)
    toks = _test_enc["input_ids"][0, idx]
    print(
        f"  {globals().get('ITEM_WORD', 'Item')} {i}: envelope=[{s}, {e}) selected_n={len(idx)} "
        f"contiguous={sel.is_contiguous} decoded={tokenizer.decode(toks)!r}"
    )


In [ ]:
print(prompts_all_agnews[0][0])

In [ ]:

# ============================================================
# 5. First pass: select features by source-block activation frequency
# ============================================================

@torch.no_grad()
def first_pass_collect_previous_feature_frequency(
    model,
    tokenizer,
    sae,
    prompts_all_agnews,
    layer_idx: int,
    expected_num_blocks: int,
    source_block_index: int | None = None,
):
    """
    For every prompt:
      - capture layer output,
      - encode with SAE,
      - compute active-token frequency on the varied source block.

    The function name is kept for backward compatibility with earlier notebook
    versions, where the source block was always the immediately previous block.

    Returns:
      mean_prev_freq: np.ndarray [num_features]
      used_prompt_count: int
    """
    if source_block_index is None:
        source_block_index = int(LFH_SOURCE_BLOCK_INDEX)

    num_features = sae.n_features
    freq_sum = torch.zeros(num_features, dtype=torch.float64)
    used_prompt_count = 0

    for query_idx in range(len(prompts_all_agnews)):
        for variant_idx in range(len(prompts_all_agnews[query_idx])):
            prompt = prompts_all_agnews[query_idx][variant_idx]

            enc = tokenize_prompt_for_offsets(prompt)
            offset_mapping = enc["offset_mapping"][0].tolist()
            input_ids = enc["input_ids"].to(MODEL_INPUT_DEVICE)
            attention_mask = enc["attention_mask"].to(MODEL_INPUT_DEVICE)

            block_spans = get_example_block_token_spans(
                prompt=prompt,
                offset_mapping=offset_mapping,
                expected_num_blocks=expected_num_blocks,
            )

            source_span = block_spans[source_block_index]

            x = get_layer_output(
                model=model,
                input_ids=input_ids,
                attention_mask=attention_mask,
                layer_idx=layer_idx,
            )  # [1, seq, hidden]

            sae_encoded = sae_activations(sae, x)[0].float().cpu()  # [seq, features]
            source_idx = token_selection_to_torch_indices(source_span, device=sae_encoded.device)
            source_freq = (sae_encoded.index_select(0, source_idx) > 0).float().mean(dim=0).double()

            freq_sum += source_freq
            used_prompt_count += 1

        print(f"first pass done query {query_idx + 1}/{len(prompts_all_agnews)}")

    mean_prev_freq = (freq_sum / max(used_prompt_count, 1)).numpy()
    return mean_prev_freq, used_prompt_count


mean_prev_freq, used_prompt_count = first_pass_collect_previous_feature_frequency(
    model=model,
    tokenizer=tokenizer,
    sae=sae,
    prompts_all_agnews=prompts_all_agnews,
    layer_idx=TARGET_LAYER,
    expected_num_blocks=NUM_BLOCKS_TOTAL,
    source_block_index=LFH_SOURCE_BLOCK_INDEX,
)

freq_over_threshold = {
    feat: float(freq)
    for feat, freq in enumerate(mean_prev_freq)
    if freq >= FREQ_THRESHOLD
}

print(f"Used {used_prompt_count} prompts.")
print(
    f"Selected {len(freq_over_threshold)} features with mean source-block "
    f"frequency >= {FREQ_THRESHOLD:.2f}"
)
print(f"Source block: Example {LFH_SOURCE_BLOCK_NUM}; target block: Example {LFH_TARGET_BLOCK_NUM}")


# ============================================================
# 6. Second pass: collect source/target mean SAE pre-activations
# ============================================================

@torch.no_grad()
def second_pass_collect_prev_tgt_logits(
    model,
    tokenizer,
    sae,
    prompts_all_agnews,
    layer_idx: int,
    expected_num_blocks: int,
    selected_feature_ids: list[int],
    source_block_index: int | None = None,
    target_block_index: int | None = None,
):
    """
    Collect for every prompt:
      - mean SAE pre-activation on the varied source block;
      - mean SAE pre-activation on the fixed target block.

    The returned variable names retain prev/tgt for compatibility with the
    original notebook; here prev means the configured source block.

    Returns:
      prev_logits_by_query: torch.Tensor [num_queries, num_variants, num_selected_features]
      tgt_logits_by_query : torch.Tensor [num_queries, num_variants, num_selected_features]
    """
    if source_block_index is None:
        source_block_index = int(LFH_SOURCE_BLOCK_INDEX)
    if target_block_index is None:
        target_block_index = int(LFH_TARGET_BLOCK_INDEX)

    num_queries = len(prompts_all_agnews)
    num_variants = len(prompts_all_agnews[0])
    selected_feature_ids_tensor = torch.tensor(selected_feature_ids, dtype=torch.long, device=sae.device)
    num_selected = len(selected_feature_ids)

    prev_logits_by_query = torch.zeros(num_queries, num_variants, num_selected, dtype=torch.float32)
    tgt_logits_by_query = torch.zeros(num_queries, num_variants, num_selected, dtype=torch.float32)

    for query_idx in range(num_queries):
        for variant_idx in range(num_variants):
            prompt = prompts_all_agnews[query_idx][variant_idx]

            enc = tokenize_prompt_for_offsets(prompt)
            offset_mapping = enc["offset_mapping"][0].tolist()
            input_ids = enc["input_ids"].to(MODEL_INPUT_DEVICE)
            attention_mask = enc["attention_mask"].to(MODEL_INPUT_DEVICE)

            block_spans = get_example_block_token_spans(
                prompt=prompt,
                offset_mapping=offset_mapping,
                expected_num_blocks=expected_num_blocks,
            )

            source_span = block_spans[source_block_index]
            tgt_span = block_spans[target_block_index]

            x = get_layer_output(
                model=model,
                input_ids=input_ids,
                attention_mask=attention_mask,
                layer_idx=layer_idx,
            )  # [1, seq, hidden]

            logits = sae_logits_before_jumprelu(sae, x)[0]  # [seq, features]

            source_idx = token_selection_to_torch_indices(source_span, device=logits.device)
            tgt_idx = token_selection_to_torch_indices(tgt_span, device=logits.device)
            source_mean = logits.index_select(0, source_idx)[:, selected_feature_ids_tensor].mean(dim=0).detach().float().cpu()
            tgt_mean = logits.index_select(0, tgt_idx)[:, selected_feature_ids_tensor].mean(dim=0).detach().float().cpu()

            prev_logits_by_query[query_idx, variant_idx] = source_mean
            tgt_logits_by_query[query_idx, variant_idx] = tgt_mean

        print(f"second pass done query {query_idx + 1}/{num_queries}")

    return prev_logits_by_query, tgt_logits_by_query


selected_feature_ids = sorted(freq_over_threshold.keys())

prev_logits_by_query, tgt_logits_by_query = second_pass_collect_prev_tgt_logits(
    model=model,
    tokenizer=tokenizer,
    sae=sae,
    prompts_all_agnews=prompts_all_agnews,
    layer_idx=TARGET_LAYER,
    expected_num_blocks=NUM_BLOCKS_TOTAL,
    selected_feature_ids=selected_feature_ids,
    source_block_index=LFH_SOURCE_BLOCK_INDEX,
    target_block_index=LFH_TARGET_BLOCK_INDEX,
)

print("prev/source_logits_by_query:", tuple(prev_logits_by_query.shape))
print("tgt_logits_by_query        :", tuple(tgt_logits_by_query.shape))


In [ ]:

# ============================================================
# 7. Compute per-query Pearson correlations, then average
# ============================================================

def pearson_corr_per_feature_across_variants(prev_mat: np.ndarray, tgt_mat: np.ndarray):
    """
    prev_mat, tgt_mat: [num_variants, num_features]
    Returns:
      r: [num_features]
    """
    if prev_mat.shape != tgt_mat.shape:
        raise ValueError(f"Shape mismatch: prev={prev_mat.shape}, tgt={tgt_mat.shape}")

    x = prev_mat.astype(np.float64)
    y = tgt_mat.astype(np.float64)

    x = x - x.mean(axis=0, keepdims=True)
    y = y - y.mean(axis=0, keepdims=True)

    num = (x * y).sum(axis=0)
    den = np.sqrt((x * x).sum(axis=0) * (y * y).sum(axis=0))

    r = np.full(x.shape[1], np.nan, dtype=np.float64)
    valid = den > 0
    r[valid] = num[valid] / den[valid]
    return r


def compute_final_rho_results(
    prev_logits_by_query: torch.Tensor,
    tgt_logits_by_query: torch.Tensor,
    selected_feature_ids: list[int],
    sort_ascending: bool = True,
):
    """
    Compute:
      - per-query correlation for every selected feature,
      - average correlation across queries.

    Original notebook sorted ascending, i.e. most negative correlations first.
    This is useful if you interpret negative correlation as a prediction-error-like
    or source-target local homeostasis effect.
    """
    prev_np = prev_logits_by_query.numpy()   # [Q, V, F]
    tgt_np = tgt_logits_by_query.numpy()     # [Q, V, F]

    num_queries = prev_np.shape[0]

    rho_by_query = np.stack(
        [
            pearson_corr_per_feature_across_variants(prev_np[q], tgt_np[q])
            for q in range(num_queries)
        ],
        axis=0,
    )  # [Q, F]

    querywise_rho = {}
    final_rho_results = {}

    for feat_local_idx, feat_id in enumerate(selected_feature_ids):
        vals = rho_by_query[:, feat_local_idx]
        querywise_rho[feat_id] = vals.tolist()

        finite_vals = vals[np.isfinite(vals)]
        if finite_vals.size == 0:
            continue
        final_rho_results[feat_id] = float(finite_vals.mean())

    final_rho_results = dict(
        sorted(final_rho_results.items(), key=lambda kv: kv[1], reverse=not sort_ascending)
    )

    return querywise_rho, final_rho_results, rho_by_query


querywise_rho, Final_rho_results, rho_by_query = compute_final_rho_results(
    prev_logits_by_query=prev_logits_by_query,
    tgt_logits_by_query=tgt_logits_by_query,
    selected_feature_ids=selected_feature_ids,
    sort_ascending=True,
)

print(f"Computed final correlations for {len(Final_rho_results)} features.")

rows = []
for rank, (feat, rho) in enumerate(Final_rho_results.items(), start=1):
    rows.append({
        "rank": rank,
        "feature_idx": int(feat),
        "source_block_num": int(LFH_SOURCE_BLOCK_NUM),
        "target_block_num": int(LFH_TARGET_BLOCK_NUM),
        "source_to_target_distance": int(LFH_TARGET_BLOCK_NUM - LFH_SOURCE_BLOCK_NUM),
        "mean_source_block_frequency": float(freq_over_threshold[feat]),
        # Backward-compatible column name used in earlier scratch analysis.
        "mean_prev_block_frequency": float(freq_over_threshold[feat]),
        "avg_querywise_pearson_r": float(rho),
    })

Final_rho_df = pd.DataFrame(rows)

print("=" * 100)
print("Top candidate LFH features, sorted by avg Pearson r ascending")
print(
    f"Source block Example {LFH_SOURCE_BLOCK_NUM} -> target block Example {LFH_TARGET_BLOCK_NUM}; "
    "most negative correlations first."
)
print("=" * 100)
display(Final_rho_df.head(50))

# Convenience defaults for plotting cells.
target_query_idx = 0
prev_demo_idx = 0
target_feature = int(Final_rho_df.iloc[0]["feature_idx"])
print("Default target_feature:", target_feature)


In [ ]:

# # ============================================================
# # 7b. Source-block locality control: vary block 7, 6, or 5
# # ============================================================

# import plotly.express as px

# # User-editable locality sweep config.
# # This keeps Example 8 fixed and rebuilds matched prompt sets where only the
# # source block below is resampled. The expected LFH pattern is strongest for
# # source block 7 and weaker for source blocks 6 and 5.
# RUN_LFH_LOCALITY_SWEEP = True
# LFH_LOCALITY_SWEEP_SOURCE_BLOCK_NUMS = list(LFH_LOCALITY_SWEEP_SOURCE_BLOCK_NUMS)
# LFH_LOCALITY_SWEEP_TOP_K_FEATURES = min(20, len(Final_rho_df))
# LFH_LOCALITY_SWEEP_FEATURE_IDS = None  # None => top-K LFH features from the current source-block run.
# LFH_LOCALITY_SWEEP_MAX_QUERIES = None
# LFH_LOCALITY_SWEEP_MAX_VARIANTS = None


# def _safe_pearson_for_sweep(x, y):
#     x = np.asarray(x, dtype=np.float64)
#     y = np.asarray(y, dtype=np.float64)
#     finite = np.isfinite(x) & np.isfinite(y)
#     x = x[finite]
#     y = y[finite]
#     if len(x) < 3:
#         return np.nan
#     if np.std(x) <= 1e-9 or np.std(y) <= 1e-9:
#         return np.nan
#     return float(pearsonr(x, y)[0])


# @torch.no_grad()
# def sae_selected_feature_pre_activations(
#     sae,
#     x: torch.Tensor,
#     feature_ids: list[int],
#     filter_high_norm_tokens: bool = FILTER_HIGH_NORM_TOKENS_FOR_FEATURE_EXTRACTION,
# ):
#     """
#     Memory-light SAE preactivation for a selected feature subset.

#     Args:
#       x: [batch, seq, d_model]
#       feature_ids: SAE feature IDs to score.

#     Returns:
#       pre: [batch, seq, len(feature_ids)]
#     """
#     feature_ids = [int(f) for f in feature_ids]
#     if len(feature_ids) == 0:
#         raise ValueError("feature_ids must be non-empty.")
#     if min(feature_ids) < 0 or max(feature_ids) >= sae.n_features:
#         raise ValueError(f"feature_ids must be within [0, {sae.n_features}).")

#     feature_ids_tensor = torch.tensor(feature_ids, dtype=torch.long, device=sae.device)
#     x_sae = x.to(device=sae.device, dtype=sae.dtype)
#     x_centered = sae.process_sae_in(x_sae)

#     W = sae.W_enc[:, feature_ids_tensor]
#     b = sae.b_enc[feature_ids_tensor]
#     pre = x_centered @ W + b

#     if filter_high_norm_tokens and x_sae.ndim >= 3:
#         norms = x_sae.detach().float().norm(dim=-1)  # [B, S]
#         med = norms.median(dim=1, keepdim=True).values.clamp_min(1e-6)
#         high_norm_mask = norms > (HIGH_NORM_FILTER_MULTIPLIER * med)
#         pre = pre.masked_fill(high_norm_mask.to(pre.device).unsqueeze(-1), 0.0)

#     return pre


# @torch.no_grad()
# def collect_source_target_logits_for_feature_ids(
#     prompts_all_agnews_for_source,
#     feature_ids: list[int],
#     source_block_num: int,
#     target_block_num: int = LFH_TARGET_BLOCK_NUM,
#     layer_idx: int = TARGET_LAYER,
#     max_queries=None,
#     max_variants=None,
#     verbose_every: int = 10,
# ):
#     """
#     Collect selected-feature preactivations on a chosen source block and the
#     fixed target block. This is used for matched locality controls.
#     """
#     source_block_index = block_num_to_index(source_block_num, total_blocks=NUM_BLOCKS_TOTAL)
#     target_block_index = block_num_to_index(target_block_num, total_blocks=NUM_BLOCKS_TOTAL)

#     num_queries_total = len(prompts_all_agnews_for_source)
#     query_indices = list(range(num_queries_total if max_queries is None else min(max_queries, num_queries_total)))
#     if len(query_indices) == 0:
#         raise ValueError("No queries selected for locality sweep.")

#     num_variants_total = len(prompts_all_agnews_for_source[query_indices[0]])
#     num_variants = num_variants_total if max_variants is None else min(max_variants, num_variants_total)
#     if num_variants == 0:
#         raise ValueError("No variants selected for locality sweep.")

#     feature_ids = [int(f) for f in feature_ids]
#     source_logits = torch.zeros(len(query_indices), num_variants, len(feature_ids), dtype=torch.float32)
#     target_logits = torch.zeros(len(query_indices), num_variants, len(feature_ids), dtype=torch.float32)

#     for q_out, query_idx in enumerate(query_indices):
#         variants = prompts_all_agnews_for_source[query_idx][:num_variants]
#         for variant_idx, prompt in enumerate(variants):
#             enc = tokenize_prompt_for_offsets(prompt)
#             offset_mapping = enc["offset_mapping"][0].tolist()
#             input_ids = enc["input_ids"].to(MODEL_INPUT_DEVICE)
#             attention_mask = enc["attention_mask"].to(MODEL_INPUT_DEVICE)

#             block_spans = get_example_block_token_spans(
#                 prompt=prompt,
#                 offset_mapping=offset_mapping,
#                 expected_num_blocks=NUM_BLOCKS_TOTAL,
#             )
#             source_selection = block_spans[source_block_index]
#             target_selection = block_spans[target_block_index]

#             x = get_layer_output(
#                 model=model,
#                 input_ids=input_ids,
#                 attention_mask=attention_mask,
#                 layer_idx=layer_idx,
#             )
#             pre = sae_selected_feature_pre_activations(sae, x, feature_ids)[0].detach().float().cpu()

#             source_idx = token_selection_to_torch_indices(source_selection, device=pre.device)
#             target_idx = token_selection_to_torch_indices(target_selection, device=pre.device)
#             source_logits[q_out, variant_idx] = pre.index_select(0, source_idx).mean(dim=0)
#             target_logits[q_out, variant_idx] = pre.index_select(0, target_idx).mean(dim=0)

#         if verbose_every is not None and ((q_out + 1) % max(verbose_every, 1) == 0 or q_out == len(query_indices) - 1):
#             print(
#                 f"locality sweep source block {source_block_num}: "
#                 f"processed query {q_out + 1}/{len(query_indices)}"
#             )

#     return source_logits, target_logits


# def summarize_source_target_correlations(source_logits, target_logits, feature_ids, source_block_num: int):
#     _, final_rho_results, rho_by_query_local = compute_final_rho_results(
#         prev_logits_by_query=source_logits,
#         tgt_logits_by_query=target_logits,
#         selected_feature_ids=feature_ids,
#         sort_ascending=True,
#     )

#     source_np = source_logits.numpy()
#     target_np = target_logits.numpy()

#     rows = []
#     for feat_local_idx, feat_id in enumerate(feature_ids):
#         avg_query_r = float(final_rho_results.get(int(feat_id), np.nan))
#         pooled_r = _safe_pearson_for_sweep(
#             source_np[:, :, feat_local_idx].reshape(-1),
#             target_np[:, :, feat_local_idx].reshape(-1),
#         )
#         rows.append({
#             "source_block_num": int(source_block_num),
#             "target_block_num": int(LFH_TARGET_BLOCK_NUM),
#             "source_to_target_distance": int(LFH_TARGET_BLOCK_NUM - int(source_block_num)),
#             "feature_idx": int(feat_id),
#             "avg_querywise_pearson_r": avg_query_r,
#             "pooled_pearson_r": pooled_r,
#         })

#     return pd.DataFrame(rows), rho_by_query_local


# def run_lfh_source_block_locality_sweep(
#     source_block_nums: list[int],
#     feature_ids: list[int],
#     max_queries=None,
#     max_variants=None,
# ):
#     detail_dfs = []

#     for source_block_num in source_block_nums:
#         source_block_num = int(source_block_num)
#         if source_block_num == int(LFH_SOURCE_BLOCK_NUM):
#             prompts_for_source = prompts_all_agnews
#         else:
#             prompts_for_source, _, _, _ = build_prompts_all_agnews(
#                 train_data_agnews=train_data_agnews,
#                 prompt_template_agnews=prompt_template_agnews,
#                 num_test_query=NUM_TEST_QUERY,
#                 num_previous=NUM_PREVIOUS,
#                 varied_block_num=source_block_num,
#                 target_block_num=LFH_TARGET_BLOCK_NUM,
#                 test_query_indices=test_query_indices,
#             )

#         source_logits, target_logits = collect_source_target_logits_for_feature_ids(
#             prompts_all_agnews_for_source=prompts_for_source,
#             feature_ids=feature_ids,
#             source_block_num=source_block_num,
#             target_block_num=LFH_TARGET_BLOCK_NUM,
#             layer_idx=TARGET_LAYER,
#             max_queries=max_queries,
#             max_variants=max_variants,
#         )
#         detail_df, _ = summarize_source_target_correlations(
#             source_logits=source_logits,
#             target_logits=target_logits,
#             feature_ids=feature_ids,
#             source_block_num=source_block_num,
#         )
#         detail_dfs.append(detail_df)

#     detail_df = pd.concat(detail_dfs, ignore_index=True)

#     reference_feature = int(feature_ids[0])
#     summary_rows = []
#     for source_block_num, sub in detail_df.groupby("source_block_num", sort=False):
#         ref = sub[sub["feature_idx"] == reference_feature]
#         summary_rows.append({
#             "source_block_num": int(source_block_num),
#             "target_block_num": int(LFH_TARGET_BLOCK_NUM),
#             "source_to_target_distance": int(LFH_TARGET_BLOCK_NUM - int(source_block_num)),
#             "num_features": int(len(sub)),
#             "mean_avg_querywise_pearson_r": float(np.nanmean(sub["avg_querywise_pearson_r"])),
#             "median_avg_querywise_pearson_r": float(np.nanmedian(sub["avg_querywise_pearson_r"])),
#             "mean_pooled_pearson_r": float(np.nanmean(sub["pooled_pearson_r"])),
#             "median_pooled_pearson_r": float(np.nanmedian(sub["pooled_pearson_r"])),
#             "fraction_negative_features": float(np.nanmean(sub["avg_querywise_pearson_r"] < 0)),
#             "reference_feature_idx": reference_feature,
#             "reference_feature_avg_querywise_r": float(ref["avg_querywise_pearson_r"].iloc[0]) if len(ref) else np.nan,
#             "reference_feature_pooled_r": float(ref["pooled_pearson_r"].iloc[0]) if len(ref) else np.nan,
#         })

#     summary_df = pd.DataFrame(summary_rows).sort_values("source_to_target_distance").reset_index(drop=True)
#     return summary_df, detail_df


# if RUN_LFH_LOCALITY_SWEEP:
#     if LFH_LOCALITY_SWEEP_FEATURE_IDS is None:
#         locality_feature_ids = Final_rho_df["feature_idx"].head(LFH_LOCALITY_SWEEP_TOP_K_FEATURES).astype(int).tolist()
#     else:
#         locality_feature_ids = [int(f) for f in LFH_LOCALITY_SWEEP_FEATURE_IDS]

#     print("LFH locality sweep source blocks:", LFH_LOCALITY_SWEEP_SOURCE_BLOCK_NUMS)
#     print("LFH locality sweep feature IDs:", locality_feature_ids)

#     lfh_locality_summary_df, lfh_locality_detail_df = run_lfh_source_block_locality_sweep(
#         source_block_nums=LFH_LOCALITY_SWEEP_SOURCE_BLOCK_NUMS,
#         feature_ids=locality_feature_ids,
#         max_queries=LFH_LOCALITY_SWEEP_MAX_QUERIES,
#         max_variants=LFH_LOCALITY_SWEEP_MAX_VARIANTS,
#     )

#     print("=" * 100)
#     print("LFH source-block locality summary")
#     print("More negative values at distance 1 than distance 2/3 support local, consecutive-exemplar LFH.")
#     print("=" * 100)
#     display(lfh_locality_summary_df)

#     print("Feature-level locality detail:")
#     display(lfh_locality_detail_df.sort_values(["feature_idx", "source_to_target_distance"]).reset_index(drop=True))

#     lfh_locality_fig = px.line(
#         lfh_locality_summary_df,
#         x="source_to_target_distance",
#         y="mean_avg_querywise_pearson_r",
#         markers=True,
#         hover_data=["source_block_num", "num_features", "reference_feature_avg_querywise_r"],
#         title="LFH locality control: source-to-target correlation by exemplar distance",
#         labels={
#             "source_to_target_distance": "Distance from varied source block to fixed target block",
#             "mean_avg_querywise_pearson_r": "Mean avg query-wise Pearson r over selected features",
#         },
#     )
#     lfh_locality_fig.show()
# else:
#     print("RUN_LFH_LOCALITY_SWEEP is False; skipping source-block locality controls.")


In [ ]:

# ============================================================
# 8. Scatter plot for one feature and one fixed target query
# ============================================================

import plotly.graph_objects as go
from scipy.stats import pearsonr

def plot_lfh_scatter(
    target_query_idx: int,
    target_feature: int,
    prev_logits_by_query,
    tgt_logits_by_query,
    selected_feature_ids,
):
    feature_local_idx = list(selected_feature_ids).index(target_feature)

    X = np.array(prev_logits_by_query[target_query_idx][:, feature_local_idx])
    Y = np.array(tgt_logits_by_query[target_query_idx][:, feature_local_idx])

    finite = np.isfinite(X) & np.isfinite(Y)
    X = X[finite]
    Y = Y[finite]

    if len(X) >= 2 and np.std(X) > 0 and np.std(Y) > 0:
        r, p = pearsonr(X, Y)
        slope, intercept = np.polyfit(X, Y, 1)
        x_line = np.linspace(X.min(), X.max(), 200)
        y_line = slope * x_line + intercept
    else:
        r, p = np.nan, np.nan
        x_line, y_line = np.array([]), np.array([])

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=X,
        y=Y,
        mode="markers",
        marker=dict(
            size=8,
            opacity=0.8,
            line=dict(width=1, color="black"),
        ),
        name="Prompt variants",
    ))

    if len(x_line) > 0:
        fig.add_trace(go.Scatter(
            x=x_line,
            y=y_line,
            mode="lines",
            line=dict(width=3),
            name="Linear fit",
        ))

    fig.update_layout(
        template="seaborn",
        title=dict(
            text=(
                f"Local Feature Homeostasis "
                f"({MODEL_SHORT_NAME}, Layer {TARGET_LAYER}, Feature {target_feature})<br>"
                f"corr={r:.3f}, p={p:.2e}"
            ),
            x=0.5,
            font=dict(family="Times New Roman", size=26),
        ),
        xaxis=dict(
            title=f"Activation on source demonstration (Example {LFH_SOURCE_BLOCK_NUM})",
            title_font=dict(family="Times New Roman", size=20),
            tickfont=dict(family="Times New Roman", size=14),
        ),
        yaxis=dict(
            title=f"Activation on target demonstration (Example {LFH_TARGET_BLOCK_NUM})",
            title_font=dict(family="Times New Roman", size=20),
            tickfont=dict(family="Times New Roman", size=14),
        ),
        height=600,
        width=900,
        showlegend=False,
    )

    fig.show()
    return fig, {"r": r, "p": p, "X": X, "Y": Y}


scatter_fig, scatter_stats = plot_lfh_scatter(
    target_query_idx=target_query_idx,
    target_feature=target_feature,
    prev_logits_by_query=prev_logits_by_query,
    tgt_logits_by_query=tgt_logits_by_query,
    selected_feature_ids=selected_feature_ids,
)


# Scatter 추가 Analysis

In [ ]:
# ============================================================
# Diagnostic: are LFH scatter clusters explained by selected source token?
# ============================================================

import pandas as pd
import numpy as np
import plotly.express as px
from scipy.stats import pearsonr

def _safe_r(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0:
        return np.nan, np.nan
    return pearsonr(x, y)

def get_selected_block_token_info(prompt, block_index):
    enc = tokenize_prompt_for_offsets(prompt)
    input_ids = enc["input_ids"][0]
    offset_mapping = enc["offset_mapping"][0].tolist()

    block_spans = get_example_block_token_spans(
        prompt=prompt,
        offset_mapping=offset_mapping,
        expected_num_blocks=NUM_BLOCKS_TOTAL,
    )

    s, e = block_spans[block_index]
    ids = input_ids[s:e].detach().cpu().tolist()

    return {
        "span_start": int(s),
        "span_end": int(e),
        "token_ids": ids,
        "tokens": tokenizer.convert_ids_to_tokens(ids),
        "decoded": tokenizer.decode(ids),
    }

def diagnose_lfh_scatter_groups(
    target_query_idx,
    target_feature,
    prev_logits_by_query,
    tgt_logits_by_query,
    selected_feature_ids,
):
    feature_local_idx = list(selected_feature_ids).index(target_feature)

    X = np.array(prev_logits_by_query[target_query_idx][:, feature_local_idx], dtype=float)
    Y = np.array(tgt_logits_by_query[target_query_idx][:, feature_local_idx], dtype=float)

    rows = []
    prompts = prompts_all_agnews[target_query_idx]

    for variant_idx, prompt in enumerate(prompts):
        src_info = get_selected_block_token_info(prompt, LFH_SOURCE_BLOCK_NUM - 1)
        tgt_info = get_selected_block_token_info(prompt, LFH_TARGET_BLOCK_NUM - 1)

        rows.append({
            "variant_idx": int(variant_idx),
            "X_source_activation": float(X[variant_idx]),
            "Y_target_activation": float(Y[variant_idx]),
            "source_selected_decoded": repr(src_info["decoded"]),
            "source_selected_tokens": str(src_info["tokens"]),
            "source_selected_token_ids": str(src_info["token_ids"]),
            "target_selected_decoded": repr(tgt_info["decoded"]),
            "target_selected_tokens": str(tgt_info["tokens"]),
            "target_selected_token_ids": str(tgt_info["token_ids"]),
            "source_span_start": src_info["span_start"],
            "target_span_start": tgt_info["span_start"],
        })

    df = pd.DataFrame(rows)
    df = df[np.isfinite(df["X_source_activation"]) & np.isfinite(df["Y_target_activation"])].copy()

    print("Selected source-token counts:")
    display(df["source_selected_decoded"].value_counts().to_frame("count"))

    print("Selected target-token counts:")
    display(df["target_selected_decoded"].value_counts().to_frame("count"))

    print("Group means by selected source token:")
    display(
        df.groupby("source_selected_decoded")
          .agg(
              n=("variant_idx", "count"),
              x_mean=("X_source_activation", "mean"),
              x_std=("X_source_activation", "std"),
              y_mean=("Y_target_activation", "mean"),
              y_std=("Y_target_activation", "std"),
              source_pos_mean=("source_span_start", "mean"),
              target_pos_mean=("target_span_start", "mean"),
          )
          .sort_values("x_mean")
    )

    r_pool, p_pool = _safe_r(df["X_source_activation"], df["Y_target_activation"])
    print(f"Pooled r: {r_pool:.4f}, p={p_pool:.3e}")

    # Residualize by selected source token.
    df["X_resid_by_source_token"] = (
        df["X_source_activation"]
        - df.groupby("source_selected_decoded")["X_source_activation"].transform("mean")
    )
    df["Y_resid_by_source_token"] = (
        df["Y_target_activation"]
        - df.groupby("source_selected_decoded")["Y_target_activation"].transform("mean")
    )

    r_resid, p_resid = _safe_r(df["X_resid_by_source_token"], df["Y_resid_by_source_token"])
    print(f"Within-source-token residual r: {r_resid:.4f}, p={p_resid:.3e}")

    print("Within-source-token correlations:")
    within_rows = []
    for tok, g in df.groupby("source_selected_decoded"):
        r, p = _safe_r(g["X_source_activation"], g["Y_target_activation"])
        within_rows.append({
            "source_selected_decoded": tok,
            "n": len(g),
            "r_within": r,
            "p_within": p,
        })
    display(pd.DataFrame(within_rows).sort_values("source_selected_decoded"))

    fig = px.scatter(
        df,
        x="X_source_activation",
        y="Y_target_activation",
        color="source_selected_decoded",
        hover_data=[
            "variant_idx",
            "source_selected_tokens",
            "target_selected_decoded",
            "source_span_start",
            "target_span_start",
        ],
        title=(
            f"LFH scatter colored by selected source token<br>"
            f"feature={target_feature}, pooled r={r_pool:.3f}, "
            f"residual-by-token r={r_resid:.3f}"
        ),
    )
    fig.update_layout(width=900, height=600)
    fig.show()

    fig_resid = px.scatter(
        df,
        x="X_resid_by_source_token",
        y="Y_resid_by_source_token",
        color="source_selected_decoded",
        hover_data=["variant_idx"],
        title=(
            f"Within-source-token residual LFH scatter<br>"
            f"feature={target_feature}, residual r={r_resid:.3f}"
        ),
    )
    fig_resid.update_layout(width=900, height=600)
    fig_resid.show()

    return df

lfh_group_diag_df = diagnose_lfh_scatter_groups(
    target_query_idx=target_query_idx,
    target_feature=target_feature,
    prev_logits_by_query=prev_logits_by_query,
    tgt_logits_by_query=tgt_logits_by_query,
    selected_feature_ids=selected_feature_ids,
)

# Histograms

# BLOCK_TOKEN_OFFSETS_FROM_END = [-1]

In [ ]:
# ============================================================
# 9. Histogram of average query-wise Pearson correlations
# ============================================================

import plotly.graph_objects as go
import numpy as np
from scipy.stats import gaussian_kde

def plot_rho_histogram(Final_rho_results, num_bins=25):
    values = np.array(list(Final_rho_results.values()), dtype=np.float64)
    values = values[np.isfinite(values)]
    N = len(values)

    if N == 0:
        raise ValueError("No finite rho values to plot.")

    min_val, max_val = float(np.min(values)), float(np.max(values))
    if min_val == max_val:
        min_val -= 1e-3
        max_val += 1e-3

    bin_width = (max_val - min_val) / num_bins

    fig = go.Figure()

    # Histogram with green bars
    fig.add_trace(go.Histogram(
        x=values,
        xbins=dict(start=min_val, end=max_val, size=bin_width),
        opacity=0.7,
        marker_color="green",  # <-- Added green color for bars
        name="Feature count",
    ))

    # KDE curve with dark green line
    if N >= 3 and np.std(values) > 0:
        kde = gaussian_kde(values)
        x_grid = np.linspace(-1, 1, 400)
        y_kde = kde(x_grid) * N * bin_width

        fig.add_trace(go.Scatter(
            x=x_grid,
            y=y_kde,
            mode="lines",
            line=dict(width=4, color="darkgreen"),  # <-- Added dark green color for KDE
            name="KDE Curve",
        ))

    fig.update_layout(
        template="seaborn",  # <-- Changed background to seaborn
        barmode="overlay",
        title=dict(
            text=f"Local Feature Homeostasis Mechanisms in LLMs (Task: NQ)<br>({MODEL_SHORT_NAME}, Layer {TARGET_LAYER})",
            x=0.5,
            font=dict(family="Times New Roman", size=26),
        ),
        xaxis=dict(
            title="Pearson correlation",
            range=[-1, 1],
            title_font=dict(family="Times New Roman", size=20),
            tickfont=dict(family="Times New Roman", size=16),
        ),
        yaxis=dict(
            title="Feature count",
            title_font=dict(family="Times New Roman", size=20),
            tickfont=dict(family="Times New Roman", size=16),
        ),
        height=600,
        width=900,
        showlegend=False
    )

    # --- ADDED: 4x Resolution Config for Interactive 'Download Plot' Button ---
    config = {
        'toImageButtonOptions': {
            'format': 'png',         # one of png, svg, jpeg, webp
            'filename': 'rho_histogram',
            'height': 600,
            'width': 900,
            'scale': 4               # Multiply resolution by 4
        }
    }
    
    fig.show(config=config)
    
    # --- ADDED: Optional direct save to disk at 4x resolution ---
    # Uncomment the line below to save programmatically (requires pip install kaleido)
    # fig.write_image("rho_histogram_highres.png", scale=4)

    return fig

# Uncomment to run if Final_rho_results, MODEL_SHORT_NAME, and TARGET_LAYER are defined
rho_hist_fig = plot_rho_histogram(Final_rho_results)

# BLOCK_TOKEN_OFFSETS_FROM_END = [-2]

In [ ]:
# ============================================================
# 9. Histogram of average query-wise Pearson correlations
# ============================================================

import plotly.graph_objects as go
import numpy as np
from scipy.stats import gaussian_kde

def plot_rho_histogram(Final_rho_results, num_bins=25):
    values = np.array(list(Final_rho_results.values()), dtype=np.float64)
    values = values[np.isfinite(values)]
    N = len(values)

    if N == 0:
        raise ValueError("No finite rho values to plot.")

    min_val, max_val = float(np.min(values)), float(np.max(values))
    if min_val == max_val:
        min_val -= 1e-3
        max_val += 1e-3

    bin_width = (max_val - min_val) / num_bins

    fig = go.Figure()

    # Histogram with green bars
    fig.add_trace(go.Histogram(
        x=values,
        xbins=dict(start=min_val, end=max_val, size=bin_width),
        opacity=0.7,
        marker_color="green",  # <-- Added green color for bars
        name="Feature count",
    ))

    # KDE curve with dark green line
    if N >= 3 and np.std(values) > 0:
        kde = gaussian_kde(values)
        x_grid = np.linspace(-1, 1, 400)
        y_kde = kde(x_grid) * N * bin_width

        fig.add_trace(go.Scatter(
            x=x_grid,
            y=y_kde,
            mode="lines",
            line=dict(width=4, color="darkgreen"),  # <-- Added dark green color for KDE
            name="KDE Curve",
        ))

    fig.update_layout(
        template="seaborn",  # <-- Changed background to seaborn
        barmode="overlay",
        title=dict(
            text=f"Local Feature Homeostasis Mechanisms in LLMs<br>({MODEL_SHORT_NAME}, Layer {TARGET_LAYER})",
            x=0.5,
            font=dict(family="Times New Roman", size=26),
        ),
        xaxis=dict(
            title="Pearson correlation",
            range=[-1, 1],
            title_font=dict(family="Times New Roman", size=20),
            tickfont=dict(family="Times New Roman", size=16),
        ),
        yaxis=dict(
            title="Feature count",
            title_font=dict(family="Times New Roman", size=20),
            tickfont=dict(family="Times New Roman", size=16),
        ),
        height=600,
        width=900,
        showlegend=False
    )

    # --- ADDED: 4x Resolution Config for Interactive 'Download Plot' Button ---
    config = {
        'toImageButtonOptions': {
            'format': 'png',         # one of png, svg, jpeg, webp
            'filename': 'rho_histogram',
            'height': 600,
            'width': 900,
            'scale': 4               # Multiply resolution by 4
        }
    }
    
    fig.show(config=config)
    
    # --- ADDED: Optional direct save to disk at 4x resolution ---
    # Uncomment the line below to save programmatically (requires pip install kaleido)
    # fig.write_image("rho_histogram_highres.png", scale=4)

    return fig

# Uncomment to run if Final_rho_results, MODEL_SHORT_NAME, and TARGET_LAYER are defined
rho_hist_fig = plot_rho_histogram(Final_rho_results)

# BLOCK_TOKEN_OFFSETS_FROM_END = [-1,-2,-3,-4,-5]

In [ ]:
# ============================================================
# 9. Histogram of average query-wise Pearson correlations
# ============================================================

import plotly.graph_objects as go
import numpy as np
from scipy.stats import gaussian_kde

def plot_rho_histogram(Final_rho_results, num_bins=25):
    values = np.array(list(Final_rho_results.values()), dtype=np.float64)
    values = values[np.isfinite(values)]
    N = len(values)

    if N == 0:
        raise ValueError("No finite rho values to plot.")

    min_val, max_val = float(np.min(values)), float(np.max(values))
    if min_val == max_val:
        min_val -= 1e-3
        max_val += 1e-3

    bin_width = (max_val - min_val) / num_bins

    fig = go.Figure()

    # Histogram with green bars
    fig.add_trace(go.Histogram(
        x=values,
        xbins=dict(start=min_val, end=max_val, size=bin_width),
        opacity=0.7,
        marker_color="green",  # <-- Added green color for bars
        name="Feature count",
    ))

    # KDE curve with dark green line
    if N >= 3 and np.std(values) > 0:
        kde = gaussian_kde(values)
        x_grid = np.linspace(-1, 1, 400)
        y_kde = kde(x_grid) * N * bin_width

        fig.add_trace(go.Scatter(
            x=x_grid,
            y=y_kde,
            mode="lines",
            line=dict(width=4, color="darkgreen"),  # <-- Added dark green color for KDE
            name="KDE Curve",
        ))

    fig.update_layout(
        template="seaborn",  # <-- Changed background to seaborn
        barmode="overlay",
        title=dict(
            text=f"Local Feature Homeostasis Mechanisms in LLMs<br>({MODEL_SHORT_NAME}, Layer {TARGET_LAYER})",
            x=0.5,
            font=dict(family="Times New Roman", size=26),
        ),
        xaxis=dict(
            title="Pearson correlation",
            range=[-1, 1],
            title_font=dict(family="Times New Roman", size=20),
            tickfont=dict(family="Times New Roman", size=16),
        ),
        yaxis=dict(
            title="Feature count",
            title_font=dict(family="Times New Roman", size=20),
            tickfont=dict(family="Times New Roman", size=16),
        ),
        height=600,
        width=900,
        showlegend=False
    )

    # --- ADDED: 4x Resolution Config for Interactive 'Download Plot' Button ---
    config = {
        'toImageButtonOptions': {
            'format': 'png',         # one of png, svg, jpeg, webp
            'filename': 'rho_histogram',
            'height': 600,
            'width': 900,
            'scale': 4               # Multiply resolution by 4
        }
    }
    
    fig.show(config=config)
    
    # --- ADDED: Optional direct save to disk at 4x resolution ---
    # Uncomment the line below to save programmatically (requires pip install kaleido)
    # fig.write_image("rho_histogram_highres.png", scale=4)

    return fig

# Uncomment to run if Final_rho_results, MODEL_SHORT_NAME, and TARGET_LAYER are defined
rho_hist_fig = plot_rho_histogram(Final_rho_results)

# BLOCK_TOKEN_OFFSETS_FROM_END = [-1,-2,-3,-4,-5,-6,-7]

In [ ]:
# ============================================================
# 9. Histogram of average query-wise Pearson correlations
# ============================================================

import plotly.graph_objects as go
import numpy as np
from scipy.stats import gaussian_kde

def plot_rho_histogram(Final_rho_results, num_bins=25):
    values = np.array(list(Final_rho_results.values()), dtype=np.float64)
    values = values[np.isfinite(values)]
    N = len(values)

    if N == 0:
        raise ValueError("No finite rho values to plot.")

    min_val, max_val = float(np.min(values)), float(np.max(values))
    if min_val == max_val:
        min_val -= 1e-3
        max_val += 1e-3

    bin_width = (max_val - min_val) / num_bins

    fig = go.Figure()

    # Histogram with green bars
    fig.add_trace(go.Histogram(
        x=values,
        xbins=dict(start=min_val, end=max_val, size=bin_width),
        opacity=0.7,
        marker_color="green",  # <-- Added green color for bars
        name="Feature count",
    ))

    # KDE curve with dark green line
    if N >= 3 and np.std(values) > 0:
        kde = gaussian_kde(values)
        x_grid = np.linspace(-1, 1, 400)
        y_kde = kde(x_grid) * N * bin_width

        fig.add_trace(go.Scatter(
            x=x_grid,
            y=y_kde,
            mode="lines",
            line=dict(width=4, color="darkgreen"),  # <-- Added dark green color for KDE
            name="KDE Curve",
        ))

    fig.update_layout(
        template="seaborn",  # <-- Changed background to seaborn
        barmode="overlay",
        title=dict(
            text=f"Local Feature Homeostasis Mechanisms in LLMs<br>({MODEL_SHORT_NAME}, Layer {TARGET_LAYER})",
            x=0.5,
            font=dict(family="Times New Roman", size=26),
        ),
        xaxis=dict(
            title="Pearson correlation",
            range=[-1, 1],
            title_font=dict(family="Times New Roman", size=20),
            tickfont=dict(family="Times New Roman", size=16),
        ),
        yaxis=dict(
            title="Feature count",
            title_font=dict(family="Times New Roman", size=20),
            tickfont=dict(family="Times New Roman", size=16),
        ),
        height=600,
        width=900,
        showlegend=False
    )

    # --- ADDED: 4x Resolution Config for Interactive 'Download Plot' Button ---
    config = {
        'toImageButtonOptions': {
            'format': 'png',         # one of png, svg, jpeg, webp
            'filename': 'rho_histogram',
            'height': 600,
            'width': 900,
            'scale': 4               # Multiply resolution by 4
        }
    }
    
    fig.show(config=config)
    
    # --- ADDED: Optional direct save to disk at 4x resolution ---
    # Uncomment the line below to save programmatically (requires pip install kaleido)
    # fig.write_image("rho_histogram_highres.png", scale=4)

    return fig

# Uncomment to run if Final_rho_results, MODEL_SHORT_NAME, and TARGET_LAYER are defined
rho_hist_fig = plot_rho_histogram(Final_rho_results)

---

In [ ]:

# ============================================================
# 10. Token-level activation plotting for two prompt variants
# ============================================================

from IPython.display import HTML, display
import html
import torch.nn.functional as F


def print_tokens_with_highlights(decoded_tokens, colored_tokens, color: str = "blue", font_size: str = "18px"):
    colored_set = set(int(i) for i in colored_tokens)
    style = (
        "font-family: 'Times New Roman', Times, serif; "
        f"font-size: {font_size}; "
        "white-space: pre-wrap;"
    )

    parts = []
    for i, tok in enumerate(decoded_tokens):
        safe_tok = html.escape(str(tok))
        if i in colored_set:
            parts.append(f'<span style="color: {color}; font-weight: 700;">{safe_tok}</span>')
        else:
            parts.append(safe_tok)

    display(HTML(f'<div style="{style}">{"".join(parts)}</div>'))


def decode_tokens_from_input_ids(tokenizer, input_ids_1d):
    return [tokenizer.decode([int(tok)]) for tok in input_ids_1d.detach().cpu().tolist()]


@torch.no_grad()
def capture_feature_curve_for_prompt(prompt: str, feature_idx: int, target_layer: int):
    enc = tokenize_prompt_for_offsets(prompt)
    offset_mapping = enc["offset_mapping"][0].tolist()
    input_ids = enc["input_ids"].to(MODEL_INPUT_DEVICE)
    attention_mask = enc["attention_mask"].to(MODEL_INPUT_DEVICE)

    block_spans = get_example_block_token_spans(
        prompt=prompt,
        offset_mapping=offset_mapping,
        expected_num_blocks=NUM_BLOCKS_TOTAL,
    )

    x = get_layer_output(
        model=model,
        input_ids=input_ids,
        attention_mask=attention_mask,
        layer_idx=target_layer,
    )

    pre_all = sae_logits_before_jumprelu(sae, x)[0].detach().float().cpu()
    act_all = sae.activation_fn(pre_all.to(device=sae.device, dtype=sae.dtype)).detach().float().cpu()

    pre_curve = pre_all[:, feature_idx]
    act_curve = act_all[:, feature_idx]

    input_ids_1d = enc["input_ids"][0]
    decoded_tokens = decode_tokens_from_input_ids(tokenizer, input_ids_1d)

    return {
        "prompt": prompt,
        "input_ids": input_ids_1d,
        "offset_mapping": offset_mapping,
        "tokens": decoded_tokens,
        "block_spans": block_spans,
        "pre_curve": pre_curve,
        "act_curve": act_curve,
    }


def plot_single_prompt_feature_curve(curve_info, feature_idx: int, title_prefix: str, value_kind: str = "pre"):
    tokens = curve_info["tokens"]
    y = curve_info["pre_curve"] if value_kind == "pre" else curve_info["act_curve"]
    y_np = y.detach().cpu().numpy()

    hover_texts = []
    for i, tok in enumerate(tokens):
        next_tok = tokens[i + 1] if i + 1 < len(tokens) else ""
        hover_texts.append(f"pos={i}<br>Token: {html.escape(tok)}<br>Next: {html.escape(next_tok)}<br>value={y_np[i]:.4f}")

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=list(range(len(tokens))),
        y=y_np,
        text=hover_texts,
        hovertemplate="%{text}<extra></extra>",
        width=2,
        name=f"feature {feature_idx}",
    ))

    for i, (s, e) in enumerate(curve_info["block_spans"], start=1):
        fig.add_vline(x=s, line_dash="dash", line_width=2)
        fig.add_annotation(
            x=s,
            y=1.02,
            xref="x",
            yref="paper",
            text=f"Ex-{i}",
            showarrow=False,
            font=dict(family="Times New Roman", size=11),
        )

    fig.update_layout(
        title=dict(
            text=f"{title_prefix}<br>{MODEL_SHORT_NAME}, Layer {TARGET_LAYER}, Feature {feature_idx}, {value_kind}-activation",
            x=0.5,
            font=dict(family="Times New Roman", size=22),
        ),
        xaxis_title="Token position",
        yaxis_title="SAE pre-activation" if value_kind == "pre" else "SAE post-threshold activation",
        font=dict(family="Times New Roman", size=16),
        height=550,
        width=1000,
    )

    fig.show()
    return fig


def plot_right_aligned_source_to_target_blocks(
    curve_1,
    curve_2,
    feature_idx: int,
    value_kind: str = "pre",
    source_block_index: int | None = None,
    target_block_index: int | None = None,
):
    y1_all = curve_1["pre_curve"] if value_kind == "pre" else curve_1["act_curve"]
    y2_all = curve_2["pre_curve"] if value_kind == "pre" else curve_2["act_curve"]

    b1 = curve_1["block_spans"]
    b2 = curve_2["block_spans"]

    if source_block_index is None:
        source_block_index = int(LFH_SOURCE_BLOCK_INDEX)
    if target_block_index is None:
        target_block_index = int(LFH_TARGET_BLOCK_INDEX)

    source_block_num = source_block_index + 1
    target_block_num = target_block_index + 1

    # Show the entire region from the varied source block through the fixed target block.
    s1, e1 = b1[source_block_index][0], b1[target_block_index][1]
    s2, e2 = b2[source_block_index][0], b2[target_block_index][1]

    y1 = y1_all[s1:e1].detach().cpu().numpy()
    y2 = y2_all[s2:e2].detach().cpu().numpy()

    tok1 = curve_1["tokens"][s1:e1]
    tok2 = curve_2["tokens"][s2:e2]

    max_len = max(len(y1), len(y2))

    def left_pad(arr, target_len, pad_value=0.0):
        return [pad_value] * (target_len - len(arr)) + list(arr)

    def left_pad_text(arr, target_len):
        return [""] * (target_len - len(arr)) + list(arr)

    y1_pad = left_pad(y1, max_len, 0.0)
    y2_pad = left_pad(y2, max_len, 0.0)
    tok1_pad = left_pad_text(tok1, max_len)
    tok2_pad = left_pad_text(tok2, max_len)

    x = list(range(max_len))

    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=x,
        y=y1_pad,
        text=[f"Prompt 1 token: {html.escape(t)}" for t in tok1_pad],
        hovertemplate="%{text}<br>value=%{y:.4f}<extra></extra>",
        opacity=0.55,
        width=2,
        name="Prompt variant 1",
    ))

    fig.add_trace(go.Bar(
        x=x,
        y=y2_pad,
        text=[f"Prompt 2 token: {html.escape(t)}" for t in tok2_pad],
        hovertemplate="%{text}<br>value=%{y:.4f}<extra></extra>",
        opacity=0.55,
        width=2,
        name="Prompt variant 2",
    ))

    # Mark shared target block boundary.
    target_len_1 = b1[target_block_index][1] - b1[target_block_index][0]
    target_boundary = max_len - target_len_1

    fig.add_vline(x=target_boundary, line_dash="dash", line_width=3)
    fig.add_annotation(
        x=target_boundary,
        y=1.05,
        xref="x",
        yref="paper",
        text=f"Block {target_block_num} target/query<br>(shared target)",
        showarrow=False,
        font=dict(family="Times New Roman", size=16),
    )

    fig.update_layout(
        barmode="overlay",
        title=dict(
            text=(
                f"Right-aligned source block {source_block_num} through target block {target_block_num}<br>"
                f"{MODEL_SHORT_NAME}, Layer {TARGET_LAYER}, Feature {feature_idx}, {value_kind}-activation"
            ),
            x=0.5,
            font=dict(family="Times New Roman", size=22),
        ),
        xaxis_title=f"Right-aligned token position over blocks {source_block_num} through {target_block_num}",
        yaxis_title="SAE pre-activation" if value_kind == "pre" else "SAE post-threshold activation",
        font=dict(family="Times New Roman", size=16),
        height=600,
        width=1000,
    )

    fig.show()
    return fig

def plot_activations_agnews(
    input_prompt_1,
    input_prompt_2,
    target_layer,
    example_idx,
    feature_idx,
    value_kind: str = "pre",
    detailed: bool = True,
    plot_all: bool = True,
):
    """
    Robust model-family-compatible version of the original token-level activation plotting helper.

    It does not rely on decoded tokens equaling exactly 'Example'; instead it uses
    the same regex + offset-mapping block spans used in the main analysis.
    """
    assert value_kind in {"pre", "post"}

    curve_1 = capture_feature_curve_for_prompt(input_prompt_1, feature_idx, target_layer)
    curve_2 = capture_feature_curve_for_prompt(input_prompt_2, feature_idx, target_layer)

    y1 = curve_1["pre_curve"] if value_kind == "pre" else curve_1["act_curve"]
    y2 = curve_2["pre_curve"] if value_kind == "pre" else curve_2["act_curve"]

    if detailed:
        print("=" * 100)
        print(f"feature_idx={feature_idx}, value_kind={value_kind}")
        print("=" * 100)

        active_1 = torch.nonzero(y1 > 0, as_tuple=False).flatten().tolist()
        active_2 = torch.nonzero(y2 > 0, as_tuple=False).flatten().tolist()

        print("Prompt 1 active-token count:", len(active_1))
        print("Prompt 2 active-token count:", len(active_2))

        print("\nPrompt 1 highlighted active tokens:")
        print_tokens_with_highlights(curve_1["tokens"], active_1)

        print("\nPrompt 2 highlighted active tokens:")
        print_tokens_with_highlights(curve_2["tokens"], active_2)

        print("\nBlock spans prompt 1:", curve_1["block_spans"])
        print("Block spans prompt 2:", curve_2["block_spans"])

    figs = {}
    if plot_all:
        figs["prompt_1"] = plot_single_prompt_feature_curve(
            curve_1,
            feature_idx=feature_idx,
            title_prefix=f"Prompt variant 1, query {example_idx}",
            value_kind=value_kind,
        )
        figs["prompt_2"] = plot_single_prompt_feature_curve(
            curve_2,
            feature_idx=feature_idx,
            title_prefix=f"Prompt variant 2, query {example_idx}",
            value_kind=value_kind,
        )
        figs["right_aligned_source_to_target_blocks"] = plot_right_aligned_source_to_target_blocks(
            curve_1,
            curve_2,
            feature_idx=feature_idx,
            value_kind=value_kind,
        )

    return curve_1, curve_2, figs


# Example usage:
example_idx = 0
shuffle_1 = 0
shuffle_2 = min(5, NUM_PREVIOUS - 1)

curve_1, curve_2, activation_figs = plot_activations_agnews(
    input_prompt_1=prompts_all_agnews[example_idx][shuffle_1],
    input_prompt_2=prompts_all_agnews[example_idx][shuffle_2],
    target_layer=TARGET_LAYER,
    example_idx=example_idx,
    feature_idx=target_feature,
    value_kind="pre",
    detailed=True,
    plot_all=True,
)


---


# 12. Empirical test: do target-layer attention heads write the negative feature direction on the next demonstration?

The previous cells identify features whose mean pre-activation on the configured source demonstration block is negatively correlated with their mean pre-activation on the target/next block. This section tests a more mechanistic hypothesis at the same layer as the SAE:

> If feature `f` is high on demonstration `n`, then some target-layer attention heads write a negative component along feature `f`'s SAE direction into the residual stream on the target demonstration.

For each prompt variant, this block captures:

1. target-layer residual output, used to measure SAE pre-activation of feature `f` on the source and target blocks;
2. the input to target-layer `self_attn.o_proj`, split by attention head;
3. each head's signed contribution to the target block along the feature's SAE read direction and write direction.

A candidate implementation head should have:

- negative correlation between source-block feature activation and target-block head write along the feature read/write direction;
- negative high-minus-low difference: prompts where feature `f` is high on the source block should have more negative head writes on the target block;
- ideally, positive correlation between the head's signed write and the target-block feature pre-activation, meaning the head's write direction is actually aligned with the feature's measured target activation.


In [ ]:
# ============================================================
# 12. Attention-head negative-write test for one feature
# ============================================================

import numpy as np
import pandas as pd
import torch
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import pearsonr

# ------------------------------------------------------------
# User-editable config
# ------------------------------------------------------------

ATTN_WRITE_TEST_LAYER = TARGET_LAYER

# By default, test the strongest anti-correlation feature found above.
# Replace this with a specific feature, e.g. 567.
ATTN_WRITE_TEST_FEATURE_IDX = int(target_feature)

# The prompt manipulation varies the configured source block and holds the
# target block fixed. Default: source Example 7 -> target Example 8.
ATTN_WRITE_TEST_PREVIOUS_BLOCK_INDEX = int(LFH_SOURCE_BLOCK_INDEX)
ATTN_WRITE_TEST_TARGET_BLOCK_INDEX = int(LFH_TARGET_BLOCK_INDEX)

# The selected model can be large. Start with a subset; set both to None for the full run.
ATTN_WRITE_TEST_MAX_QUERIES = min(10, len(prompts_all_agnews))
ATTN_WRITE_TEST_MAX_VARIANTS = None

# For high-vs-low comparisons within each fixed target query.
# 0.25 means bottom quartile vs top quartile across source-demonstration variants.
ATTN_WRITE_TEST_QUANTILE = 0.25

EPS = 1e-9


# ------------------------------------------------------------
# Helpers: SAE single-feature preactivation and attention head geometry
# ------------------------------------------------------------

def _safe_pearson(x, y):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    finite = np.isfinite(x) & np.isfinite(y)
    x = x[finite]
    y = y[finite]
    if len(x) < 3:
        return np.nan
    if np.std(x) <= EPS or np.std(y) <= EPS:
        return np.nan
    return float(pearsonr(x, y)[0])


def _unit(v: torch.Tensor) -> torch.Tensor:
    v = v.float()
    return v / v.norm().clamp_min(1e-8)


@torch.no_grad()
def sae_single_feature_pre_activation(
    sae,
    x: torch.Tensor,
    feature_idx: int,
    filter_high_norm_tokens: bool = FILTER_HIGH_NORM_TOKENS_FOR_FEATURE_EXTRACTION,
):
    """
    Memory-light single-feature version of sae_logits_before_jumprelu.

    Args:
      x: [batch, seq, d_model]

    Returns:
      pre: [batch, seq]
    """
    if feature_idx < 0 or feature_idx >= sae.n_features:
        raise ValueError(f"feature_idx={feature_idx} outside [0, {sae.n_features}).")

    x_sae = x.to(device=sae.device, dtype=sae.dtype)
    x_centered = sae.process_sae_in(x_sae)

    w = sae.W_enc[:, feature_idx]
    b = sae.b_enc[feature_idx]
    pre = x_centered @ w + b

    if filter_high_norm_tokens and x_sae.ndim >= 3:
        norms = x_sae.detach().float().norm(dim=-1)  # [B, S]
        med = norms.median(dim=1, keepdim=True).values.clamp_min(1e-6)
        high_norm_mask = norms > (HIGH_NORM_FILTER_MULTIPLIER * med)
        pre = pre.masked_fill(high_norm_mask.to(pre.device), 0.0)

    return pre


@torch.no_grad()
def capture_layer_output_and_attn_o_proj_input(model, input_ids, attention_mask, layer_idx: int):
    """
    One forward pass that captures:
      - output of model.layers[layer_idx], shape [B, S, d_model]
      - input to self_attn.o_proj at that layer, shape [B, S, d_model]

    The o_proj input is the concatenated per-head attention result before W_O.
    Splitting this tensor by heads lets us compute each head's signed output
    contribution along an SAE feature direction.
    """
    cache = {}

    block = model.get_submodule(f"model.layers.{layer_idx}")
    o_proj = model.get_submodule(f"model.layers.{layer_idx}.self_attn.o_proj")

    def block_hook(module, inputs, output):
        x = output[0] if isinstance(output, tuple) else output
        cache["layer_output"] = x.detach()
        return output

    def o_proj_pre_hook(module, inputs):
        cache["o_proj_input"] = inputs[0].detach()
        return None

    h1 = block.register_forward_hook(block_hook)
    h2 = o_proj.register_forward_pre_hook(o_proj_pre_hook)

    try:
        _ = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
        )
    finally:
        h1.remove()
        h2.remove()

    if "layer_output" not in cache:
        raise RuntimeError(f"Failed to capture layer output for layer {layer_idx}.")
    if "o_proj_input" not in cache:
        raise RuntimeError(f"Failed to capture o_proj input for layer {layer_idx}.")

    return cache["layer_output"], cache["o_proj_input"]


@torch.no_grad()
def get_head_projection_vectors_for_feature(model, sae, layer_idx: int, feature_idx: int):
    """
    For each attention head h at layer_idx, compute the vector v_h such that:

        head_output_contribution_h dot feature_direction
        = o_proj_input_head_h dot v_h

    Returns head-specific vectors for both SAE read direction and SAE write direction.
    """
    attn = model.get_submodule(f"model.layers.{layer_idx}.self_attn")
    o_proj = attn.o_proj

    num_heads = int(getattr(attn, "num_heads", model.config.num_attention_heads))
    head_dim = int(getattr(attn, "head_dim", model.config.hidden_size // num_heads))

    W_o = o_proj.weight.detach()  # [d_model_out, d_model_in]
    out_dim, in_dim = W_o.shape

    if in_dim != num_heads * head_dim:
        raise ValueError(
            f"o_proj input dim {in_dim} != num_heads*head_dim {num_heads}*{head_dim}."
        )

    device = W_o.device

    read_dir = _unit(sae.W_enc[:, feature_idx].detach().to(device=device))
    write_dir = _unit(sae.W_dec[feature_idx].detach().to(device=device))

    if read_dir.numel() != out_dim or write_dir.numel() != out_dim:
        raise ValueError(
            f"Feature directions have dim read={read_dir.numel()}, write={write_dir.numel()}, "
            f"but o_proj output dim is {out_dim}."
        )

    # W_o maps concat-head input -> residual output:
    #   y = x @ W_o.T
    # For head h, contribution along direction u is:
    #   (x_h @ W_o_h.T) dot u = x_h dot (W_o_h.T @ u)
    W_heads = W_o.float().reshape(out_dim, num_heads, head_dim).permute(1, 2, 0).contiguous()
    # W_heads: [num_heads, head_dim, out_dim]

    head_vec_read = torch.matmul(W_heads, read_dir.float())    # [num_heads, head_dim]
    head_vec_write = torch.matmul(W_heads, write_dir.float())  # [num_heads, head_dim]

    return {
        "num_heads": num_heads,
        "head_dim": head_dim,
        "read_dir": read_dir.float(),
        "write_dir": write_dir.float(),
        "head_vec_read": head_vec_read.float(),
        "head_vec_write": head_vec_write.float(),
    }


@torch.no_grad()
def measure_prompt_feature_and_head_writes(
    prompt: str,
    feature_idx: int,
    layer_idx: int,
    previous_block_index: int = -2,
    target_block_index: int = -1,
    expected_num_blocks: int = NUM_BLOCKS_TOTAL,
):
    """
    For one prompt variant, measure:
      - mean preactivation of feature_idx on source block;
      - mean preactivation of feature_idx on target block;
      - for every attention head at layer_idx, mean signed write on target block
        along SAE read direction and SAE write direction.
    """
    enc = tokenize_prompt_for_offsets(prompt)
    offset_mapping = enc["offset_mapping"][0].tolist()
    input_ids = enc["input_ids"].to(MODEL_INPUT_DEVICE)
    attention_mask = enc["attention_mask"].to(MODEL_INPUT_DEVICE)

    block_spans = get_example_block_token_spans(
        prompt=prompt,
        offset_mapping=offset_mapping,
        expected_num_blocks=expected_num_blocks,
    )

    prev_span = block_spans[previous_block_index]
    tgt_span = block_spans[target_block_index]

    layer_output, o_proj_input = capture_layer_output_and_attn_o_proj_input(
        model=model,
        input_ids=input_ids,
        attention_mask=attention_mask,
        layer_idx=layer_idx,
    )

    pre_f = sae_single_feature_pre_activation(
        sae=sae,
        x=layer_output,
        feature_idx=feature_idx,
    )[0].detach().float()

    prev_idx = token_selection_to_torch_indices(prev_span, device=pre_f.device)
    tgt_idx = token_selection_to_torch_indices(tgt_span, device=pre_f.device)
    prev_feature_mean = float(pre_f.index_select(0, prev_idx).mean().detach().cpu())
    target_feature_mean = float(pre_f.index_select(0, tgt_idx).mean().detach().cpu())

    head_geom = get_head_projection_vectors_for_feature(
        model=model,
        sae=sae,
        layer_idx=layer_idx,
        feature_idx=feature_idx,
    )

    num_heads = head_geom["num_heads"]
    head_dim = head_geom["head_dim"]
    head_vec_read = head_geom["head_vec_read"].to(device=o_proj_input.device)
    head_vec_write = head_geom["head_vec_write"].to(device=o_proj_input.device)

    # Split o_proj input into heads on the selected target-block tokens.
    tgt_idx_o = token_selection_to_torch_indices(tgt_span, device=o_proj_input.device)
    z = o_proj_input[0].index_select(0, tgt_idx_o)
    z_heads = z.reshape(tgt_idx_o.numel(), num_heads, head_dim).float()

    # Signed contribution of each head's W_O output to the feature read/write directions.
    # Negative values mean the head writes in the negative feature direction.
    target_head_write_to_read = torch.einsum("thd,hd->th", z_heads, head_vec_read.float()).mean(dim=0)
    target_head_write_to_write = torch.einsum("thd,hd->th", z_heads, head_vec_write.float()).mean(dim=0)

    return {
        "prev_feature_mean": prev_feature_mean,
        "target_feature_mean": target_feature_mean,
        "target_head_write_to_read": target_head_write_to_read.detach().cpu().numpy(),
        "target_head_write_to_write": target_head_write_to_write.detach().cpu().numpy(),
        "prev_span": prev_span,
        "target_span": tgt_span,
        "num_heads": num_heads,
    }


def _summarize_head_write_test(prompt_records_df, head_read_mat, head_write_mat, layer_idx: int, feature_idx: int):
    """
    Summarize head-level evidence for:
      high prev activation -> negative target-block attention-head write.
    """
    num_prompts, num_heads = head_read_mat.shape
    rows = []

    x_all = prompt_records_df["prev_feature_mean"].to_numpy(dtype=np.float64)
    y_tgt_all = prompt_records_df["target_feature_mean"].to_numpy(dtype=np.float64)
    q_all = prompt_records_df["query_idx"].to_numpy()

    unique_queries = sorted(prompt_records_df["query_idx"].unique().tolist())

    # Overall feature anti-correlation in the same subset.
    feature_corr_pooled = _safe_pearson(x_all, y_tgt_all)

    feature_corr_by_query = []
    for q in unique_queries:
        mask = q_all == q
        feature_corr_by_query.append(_safe_pearson(x_all[mask], y_tgt_all[mask]))
    feature_corr_mean_query = float(np.nanmean(feature_corr_by_query))

    for h in range(num_heads):
        read_h = head_read_mat[:, h]
        write_h = head_write_mat[:, h]

        corr_prev_to_read_by_q = []
        corr_prev_to_write_by_q = []
        corr_read_to_target_by_q = []
        corr_write_to_target_by_q = []
        delta_read_by_q = []
        delta_write_by_q = []
        delta_read_z_by_q = []
        delta_write_z_by_q = []

        for q in unique_queries:
            mask = q_all == q
            x = x_all[mask]
            y_read = read_h[mask]
            y_write = write_h[mask]
            y_tgt = y_tgt_all[mask]

            corr_prev_to_read_by_q.append(_safe_pearson(x, y_read))
            corr_prev_to_write_by_q.append(_safe_pearson(x, y_write))
            corr_read_to_target_by_q.append(_safe_pearson(y_read, y_tgt))
            corr_write_to_target_by_q.append(_safe_pearson(y_write, y_tgt))

            finite = np.isfinite(x) & np.isfinite(y_read) & np.isfinite(y_write)
            if finite.sum() >= 4:
                xf = x[finite]
                yr = y_read[finite]
                yw = y_write[finite]

                q_low = np.quantile(xf, ATTN_WRITE_TEST_QUANTILE)
                q_high = np.quantile(xf, 1.0 - ATTN_WRITE_TEST_QUANTILE)
                low_mask = xf <= q_low
                high_mask = xf >= q_high

                if low_mask.sum() > 0 and high_mask.sum() > 0:
                    dr = float(np.mean(yr[high_mask]) - np.mean(yr[low_mask]))
                    dw = float(np.mean(yw[high_mask]) - np.mean(yw[low_mask]))
                    delta_read_by_q.append(dr)
                    delta_write_by_q.append(dw)
                    delta_read_z_by_q.append(dr / (float(np.std(yr)) + EPS))
                    delta_write_z_by_q.append(dw / (float(np.std(yw)) + EPS))
                else:
                    delta_read_by_q.append(np.nan)
                    delta_write_by_q.append(np.nan)
                    delta_read_z_by_q.append(np.nan)
                    delta_write_z_by_q.append(np.nan)
            else:
                delta_read_by_q.append(np.nan)
                delta_write_by_q.append(np.nan)
                delta_read_z_by_q.append(np.nan)
                delta_write_z_by_q.append(np.nan)

        corr_prev_to_read_mean = float(np.nanmean(corr_prev_to_read_by_q))
        corr_prev_to_write_mean = float(np.nanmean(corr_prev_to_write_by_q))
        corr_read_to_target_mean = float(np.nanmean(corr_read_to_target_by_q))
        corr_write_to_target_mean = float(np.nanmean(corr_write_to_target_by_q))
        delta_read_mean = float(np.nanmean(delta_read_by_q))
        delta_write_mean = float(np.nanmean(delta_write_by_q))
        delta_read_z_mean = float(np.nanmean(delta_read_z_by_q))
        delta_write_z_mean = float(np.nanmean(delta_write_z_by_q))

        # Mechanism score:
        #   high source feature activation should correlate with NEGATIVE head write.
        #   Therefore, corr(prev, write_to_read) and high-minus-low delta should be negative.
        anti_write_score_read = (
            max(-corr_prev_to_read_mean, 0.0)
            * max(-delta_read_z_mean, 0.0)
            * (0.5 + 0.5 * max(corr_read_to_target_mean, 0.0))
        )
        anti_write_score_write = (
            max(-corr_prev_to_write_mean, 0.0)
            * max(-delta_write_z_mean, 0.0)
            * (0.5 + 0.5 * max(corr_write_to_target_mean, 0.0))
        )

        rows.append({
            "layer": int(layer_idx),
            "head": int(h),
            "feature_idx": int(feature_idx),

            "feature_prev_to_target_corr_pooled": feature_corr_pooled,
            "feature_prev_to_target_corr_mean_query": feature_corr_mean_query,

            "corr_prev_to_target_head_write_to_read_mean_query": corr_prev_to_read_mean,
            "corr_prev_to_target_head_write_to_write_mean_query": corr_prev_to_write_mean,

            "corr_prev_to_target_head_write_to_negative_read_mean_query": -corr_prev_to_read_mean,
            "corr_prev_to_target_head_write_to_negative_write_mean_query": -corr_prev_to_write_mean,

            "high_minus_low_head_write_to_read_mean": delta_read_mean,
            "high_minus_low_head_write_to_write_mean": delta_write_mean,
            "high_minus_low_head_write_to_read_z_mean": delta_read_z_mean,
            "high_minus_low_head_write_to_write_z_mean": delta_write_z_mean,

            "high_minus_low_head_write_to_negative_read_z_mean": -delta_read_z_mean,
            "high_minus_low_head_write_to_negative_write_z_mean": -delta_write_z_mean,

            "corr_head_write_to_read_with_target_feature_mean_query": corr_read_to_target_mean,
            "corr_head_write_to_write_with_target_feature_mean_query": corr_write_to_target_mean,

            "anti_write_score_read": anti_write_score_read,
            "anti_write_score_write": anti_write_score_write,
            "anti_write_score_max": max(anti_write_score_read, anti_write_score_write),
        })

    df = pd.DataFrame(rows)
    df = df.sort_values(
        ["anti_write_score_read", "corr_prev_to_target_head_write_to_negative_read_mean_query"],
        ascending=False,
    ).reset_index(drop=True)

    df["rank_by_read_score"] = np.arange(1, len(df) + 1)

    df_by_write = df.sort_values(
        ["anti_write_score_write", "corr_prev_to_target_head_write_to_negative_write_mean_query"],
        ascending=False,
    ).reset_index(drop=True)
    write_ranks = {
        (int(row.layer), int(row.head)): rank
        for rank, row in enumerate(df_by_write.itertuples(index=False), start=1)
    }
    df["rank_by_write_score"] = [write_ranks[(int(r.layer), int(r.head))] for r in df.itertuples(index=False)]

    return df


@torch.no_grad()
def run_attention_negative_write_test_for_feature(
    feature_idx: int,
    prompts_all_agnews,
    layer_idx: int = TARGET_LAYER,
    previous_block_index: int = -2,
    target_block_index: int = -1,
    max_queries=None,
    max_variants=None,
    verbose_every: int = 10,
):
    """
    Main test.

    For every selected prompt variant:
      x = mean feature preactivation on source block
      y_h = mean layer-attention-head signed write to feature direction on target block

    Heads supporting the proposed mechanism should satisfy:
      corr(x, y_h) < 0
      mean(y_h | x high) - mean(y_h | x low) < 0
    """
    num_queries_total = len(prompts_all_agnews)
    query_indices = list(range(num_queries_total if max_queries is None else min(max_queries, num_queries_total)))

    prompt_rows = []
    read_rows = []
    write_rows = []

    total = 0
    for query_idx in query_indices:
        variants = prompts_all_agnews[query_idx]
        if max_variants is not None:
            variants = variants[:max_variants]

        for variant_idx, prompt in enumerate(variants):
            m = measure_prompt_feature_and_head_writes(
                prompt=prompt,
                feature_idx=feature_idx,
                layer_idx=layer_idx,
                previous_block_index=previous_block_index,
                target_block_index=target_block_index,
                expected_num_blocks=NUM_BLOCKS_TOTAL,
            )

            prompt_rows.append({
                "query_idx": int(query_idx),
                "variant_idx": int(variant_idx),
                "feature_idx": int(feature_idx),
                "layer": int(layer_idx),
                "prev_feature_mean": m["prev_feature_mean"],
                "target_feature_mean": m["target_feature_mean"],
                "prev_span_start": int(m["prev_span"][0]),
                "prev_span_end": int(m["prev_span"][1]),
                "target_span_start": int(m["target_span"][0]),
                "target_span_end": int(m["target_span"][1]),
            })
            read_rows.append(m["target_head_write_to_read"])
            write_rows.append(m["target_head_write_to_write"])

            total += 1

        if verbose_every is not None and (len(prompt_rows) % max(verbose_every, 1) == 0 or query_idx == query_indices[-1]):
            print(f"processed query {query_idx + 1}/{query_indices[-1] + 1}; prompt rows={len(prompt_rows)}")

    prompt_records_df = pd.DataFrame(prompt_rows)
    head_read_mat = np.stack(read_rows, axis=0)
    head_write_mat = np.stack(write_rows, axis=0)

    head_summary_df = _summarize_head_write_test(
        prompt_records_df=prompt_records_df,
        head_read_mat=head_read_mat,
        head_write_mat=head_write_mat,
        layer_idx=layer_idx,
        feature_idx=feature_idx,
    )

    print("=" * 120)
    print(f"Layer-{layer_idx} attention negative-write test for feature {feature_idx}")
    print("Expected mechanism: high source-block feature activation -> negative target-block head write")
    print("Rows/prompts measured:", len(prompt_records_df))
    print("Feature prev->target corr, pooled:", head_summary_df["feature_prev_to_target_corr_pooled"].iloc[0])
    print("Feature prev->target corr, mean query-wise:", head_summary_df["feature_prev_to_target_corr_mean_query"].iloc[0])
    print("=" * 120)

    display(head_summary_df.head(25))

    return head_summary_df, prompt_records_df, head_read_mat, head_write_mat


def plot_attention_write_scatter_for_head(
    prompt_records_df,
    head_mat,
    head: int,
    feature_idx: int,
    layer_idx: int,
    direction_name: str = "read",
):
    """
    Scatter: source feature mean vs target-block attention-head signed write.
    A negative slope supports the proposed negative-write mechanism.
    """
    x = prompt_records_df["prev_feature_mean"].to_numpy(dtype=np.float64)
    y = head_mat[:, int(head)].astype(np.float64)
    q = prompt_records_df["query_idx"].to_numpy()

    r = _safe_pearson(x, y)
    if np.isfinite(r) and np.std(x) > 0 and np.std(y) > 0:
        slope, intercept = np.polyfit(x, y, 1)
        xx = np.linspace(np.min(x), np.max(x), 200)
        yy = slope * xx + intercept
    else:
        slope, intercept = np.nan, np.nan
        xx, yy = np.array([]), np.array([])

    df_plot = pd.DataFrame({
        "prev_feature_mean": x,
        "target_head_write": y,
        "query_idx": q.astype(str),
        "variant_idx": prompt_records_df["variant_idx"].to_numpy(),
        "target_feature_mean": prompt_records_df["target_feature_mean"].to_numpy(),
    })

    fig = px.scatter(
        df_plot,
        x="prev_feature_mean",
        y="target_head_write",
        color="query_idx",
        hover_data=["variant_idx", "target_feature_mean"],
        title=(
            f"Layer {layer_idx} head {head}: source feature activation vs target-block head write<br>"
            f"feature={feature_idx}, direction={direction_name}, pooled r={r:.3f}, slope={slope:.3f}"
        ),
    )

    if len(xx) > 0:
        fig.add_trace(go.Scatter(
            x=xx,
            y=yy,
            mode="lines",
            name="pooled linear fit",
            line=dict(width=3),
        ))

    fig.add_hline(y=0, line_dash="dash", opacity=0.4)
    fig.update_layout(
        width=900,
        height=600,
        xaxis_title="Mean SAE pre-activation on source demonstration block",
        yaxis_title=f"Mean attention-head write to SAE {direction_name} direction on target block",
    )
    fig.show()
    return fig


# ------------------------------------------------------------
# Run the one-feature test
# ------------------------------------------------------------

head_summary_df, head_prompt_records_df, head_read_mat, head_write_mat = run_attention_negative_write_test_for_feature(
    feature_idx=ATTN_WRITE_TEST_FEATURE_IDX,
    prompts_all_agnews=prompts_all_agnews,
    layer_idx=ATTN_WRITE_TEST_LAYER,
    previous_block_index=ATTN_WRITE_TEST_PREVIOUS_BLOCK_INDEX,
    target_block_index=ATTN_WRITE_TEST_TARGET_BLOCK_INDEX,
    max_queries=ATTN_WRITE_TEST_MAX_QUERIES,
    max_variants=ATTN_WRITE_TEST_MAX_VARIANTS,
    verbose_every=10,
)

TOP_NEGATIVE_WRITE_HEAD = int(head_summary_df.iloc[0]["head"])
print("Top negative-write candidate head:", (ATTN_WRITE_TEST_LAYER, TOP_NEGATIVE_WRITE_HEAD))

_ = plot_attention_write_scatter_for_head(
    prompt_records_df=head_prompt_records_df,
    head_mat=head_read_mat,
    head=TOP_NEGATIVE_WRITE_HEAD,
    feature_idx=ATTN_WRITE_TEST_FEATURE_IDX,
    layer_idx=ATTN_WRITE_TEST_LAYER,
    direction_name="SAE read",
)



# 13. Shared-head test for multiple anti-relation features

This section asks whether multiple anti-relation features point to the same small set of target-layer attention heads. If the same `(layer, head)` ranks highly for many features under the negative-write score, that supports the hypothesis that a shared attention-head pathway mediates the previous-to-next demonstration suppression effect.


In [ ]:

# # ============================================================
# # 13. Shared attention-head search for a group of anti-relation features
# # ============================================================

# # Edit this list. Example requested by the user:
# ANTI_RELATION_FEATURES_FOR_HEAD_TEST = [567, 123]

# # If those feature IDs are out of range for the SAE, fall back to the top two
# # anti-correlation features found in Final_rho_df.
# ANTI_RELATION_FEATURES_FOR_HEAD_TEST = [
#     int(f) for f in ANTI_RELATION_FEATURES_FOR_HEAD_TEST
#     if 0 <= int(f) < sae.n_features
# ]
# if len(ANTI_RELATION_FEATURES_FOR_HEAD_TEST) == 0:
#     ANTI_RELATION_FEATURES_FOR_HEAD_TEST = Final_rho_df["feature_idx"].head(2).astype(int).tolist()

# print("Features for shared-head test:", ANTI_RELATION_FEATURES_FOR_HEAD_TEST)


# def combine_shared_heads_across_features(feature_to_head_summary):
#     """
#     Given {feature_idx: head_summary_df}, find heads that are high-ranked across features.
#     """
#     all_heads = sorted({
#         int(h)
#         for df in feature_to_head_summary.values()
#         for h in df["head"].astype(int).tolist()
#     })

#     rows = []

#     for head in all_heads:
#         row = {
#             "layer": int(ATTN_WRITE_TEST_LAYER),
#             "head": int(head),
#         }

#         norm_scores = []
#         read_ranks = []
#         write_ranks = []
#         neg_corrs = []
#         neg_deltas = []

#         for feat, df in feature_to_head_summary.items():
#             sub = df[df["head"].astype(int) == int(head)]
#             if len(sub) != 1:
#                 continue
#             sub = sub.iloc[0]

#             max_score = float(df["anti_write_score_read"].max())
#             score = float(sub["anti_write_score_read"])
#             norm_score = score / (max_score + EPS)

#             rank_read = int(sub["rank_by_read_score"])
#             rank_write = int(sub["rank_by_write_score"])
#             neg_corr = float(sub["corr_prev_to_target_head_write_to_negative_read_mean_query"])
#             neg_delta = float(sub["high_minus_low_head_write_to_negative_read_z_mean"])

#             row[f"feature_{feat}_read_score"] = score
#             row[f"feature_{feat}_read_norm_score"] = norm_score
#             row[f"feature_{feat}_rank_read"] = rank_read
#             row[f"feature_{feat}_rank_write"] = rank_write
#             row[f"feature_{feat}_neg_corr"] = neg_corr
#             row[f"feature_{feat}_neg_delta_z"] = neg_delta

#             norm_scores.append(max(norm_score, 0.0))
#             read_ranks.append(rank_read)
#             write_ranks.append(rank_write)
#             neg_corrs.append(neg_corr)
#             neg_deltas.append(neg_delta)

#         if len(norm_scores) == 0:
#             continue

#         norm_scores = np.asarray(norm_scores, dtype=np.float64)
#         read_ranks = np.asarray(read_ranks, dtype=np.float64)
#         write_ranks = np.asarray(write_ranks, dtype=np.float64)
#         neg_corrs = np.asarray(neg_corrs, dtype=np.float64)
#         neg_deltas = np.asarray(neg_deltas, dtype=np.float64)

#         row["num_features_present"] = int(len(norm_scores))
#         row["shared_geomean_norm_score"] = float(np.exp(np.mean(np.log(norm_scores + EPS))))
#         row["shared_min_norm_score"] = float(np.min(norm_scores))
#         row["shared_mean_rank_read"] = float(np.mean(read_ranks))
#         row["shared_max_rank_read"] = float(np.max(read_ranks))
#         row["shared_mean_negative_corr"] = float(np.nanmean(neg_corrs))
#         row["shared_min_negative_corr"] = float(np.nanmin(neg_corrs))
#         row["shared_mean_negative_delta_z"] = float(np.nanmean(neg_deltas))
#         row["shared_min_negative_delta_z"] = float(np.nanmin(neg_deltas))

#         # A strict shared score rewards heads that are strong for every feature,
#         # not just one feature.
#         row["shared_head_score"] = (
#             row["shared_geomean_norm_score"]
#             * max(row["shared_min_negative_corr"], 0.0)
#             * max(row["shared_min_negative_delta_z"], 0.0)
#         )

#         rows.append(row)

#     out = pd.DataFrame(rows)
#     out = out.sort_values(
#         ["shared_head_score", "shared_geomean_norm_score", "shared_mean_negative_corr"],
#         ascending=False,
#     ).reset_index(drop=True)
#     return out


# @torch.no_grad()
# def run_shared_negative_write_head_test_for_features(
#     feature_indices,
#     prompts_all_agnews,
#     layer_idx: int = TARGET_LAYER,
#     max_queries=None,
#     max_variants=None,
# ):
#     feature_to_head_summary = {}
#     feature_to_prompt_records = {}
#     feature_to_head_mats = {}

#     for feat in feature_indices:
#         print("\n" + "=" * 120)
#         print(f"Running head negative-write test for feature {feat}")
#         print("=" * 120)

#         hdf, pdf, read_mat, write_mat = run_attention_negative_write_test_for_feature(
#             feature_idx=int(feat),
#             prompts_all_agnews=prompts_all_agnews,
#             layer_idx=layer_idx,
#             previous_block_index=ATTN_WRITE_TEST_PREVIOUS_BLOCK_INDEX,
#             target_block_index=ATTN_WRITE_TEST_TARGET_BLOCK_INDEX,
#             max_queries=max_queries,
#             max_variants=max_variants,
#             verbose_every=10,
#         )

#         feature_to_head_summary[int(feat)] = hdf
#         feature_to_prompt_records[int(feat)] = pdf
#         feature_to_head_mats[int(feat)] = {
#             "read": read_mat,
#             "write": write_mat,
#         }

#     shared_heads_df = combine_shared_heads_across_features(feature_to_head_summary)

#     print("\n" + "=" * 120)
#     print("Shared candidate heads across anti-relation features")
#     print("=" * 120)
#     display(shared_heads_df.head(50))

#     return shared_heads_df, feature_to_head_summary, feature_to_prompt_records, feature_to_head_mats


# shared_heads_df, feature_to_head_summary, feature_to_prompt_records, feature_to_head_mats = (
#     run_shared_negative_write_head_test_for_features(
#         feature_indices=ANTI_RELATION_FEATURES_FOR_HEAD_TEST,
#         prompts_all_agnews=prompts_all_agnews,
#         layer_idx=ATTN_WRITE_TEST_LAYER,
#         max_queries=ATTN_WRITE_TEST_MAX_QUERIES,
#         max_variants=ATTN_WRITE_TEST_MAX_VARIANTS,
#     )
# )

# # Plot the strongest shared head for the first feature in the list.
# if len(shared_heads_df) > 0:
#     top_shared_head = int(shared_heads_df.iloc[0]["head"])
#     first_feat = int(ANTI_RELATION_FEATURES_FOR_HEAD_TEST[0])
#     print("Top shared candidate head:", (ATTN_WRITE_TEST_LAYER, top_shared_head))
#     print("Plotting for feature:", first_feat)

#     _ = plot_attention_write_scatter_for_head(
#         prompt_records_df=feature_to_prompt_records[first_feat],
#         head_mat=feature_to_head_mats[first_feat]["read"],
#         head=top_shared_head,
#         feature_idx=first_feat,
#         layer_idx=ATTN_WRITE_TEST_LAYER,
#         direction_name="SAE read",
#     )



# 14. Causal attention-head intervention test

The source/target attention-write section is **observational**: it asks whether a head's signed write on the target exemplar is more negative when the source exemplar's feature activation is high. This section adds a direct causal test.

For a chosen `(feature_idx, layer_idx, head_idx)`, we intervene on that head's `o_proj` input on the **target exemplar tokens only** and then re-measure the source-to-target feature correlation.

The intervention is parameterized by an additive `alpha` on the selected head contribution:

- `alpha = 0`: baseline / no intervention.
- `alpha = -1`: ablation, because the effective multiplier is `1 + alpha = 0`.
- `alpha > 0`: amplification by adding extra copies of the head contribution.

For an anti-relation feature, the expected pattern is:

- If the head is writing a negative feature-direction update on the target exemplar, **ablating** it should weaken the negative correlation.
- **Amplifying** it should strengthen the negative correlation.

This does not prove that the head reads specifically from the source exemplar. It tests the write-side causal claim: the chosen head's target-token output helps implement the source-to-target anti-relation.


# BLOCK_TOKEN_OFFSETS_FROM_END = [-1]

In [ ]:
# ============================================================
# 14. Causal validation: ablate/amplify one attention head and remeasure
#      source-exemplar -> target-exemplar feature correlation
# ============================================================

import numpy as np
import pandas as pd
import torch
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import pearsonr

# ------------------------------------------------------------
# User-editable config
# ------------------------------------------------------------

# Feature to test. By default, use the same feature as the observational
# negative-write test above.
HEAD_CAUSAL_FEATURE_IDX = int(ATTN_WRITE_TEST_FEATURE_IDX) if "ATTN_WRITE_TEST_FEATURE_IDX" in globals() else int(target_feature)

# Attention head to test. By default, use the top head from the observational
# negative-write ranking, if available.
HEAD_CAUSAL_LAYER = int(ATTN_WRITE_TEST_LAYER) if "ATTN_WRITE_TEST_LAYER" in globals() else int(TARGET_LAYER)

# aaaaa
HEAD_CAUSAL_HEAD = int(TOP_NEGATIVE_WRITE_HEAD) if "TOP_NEGATIVE_WRITE_HEAD" in globals() else 0
# HEAD_CAUSAL_HEAD = 13

# The SAE feature is measured at TARGET_LAYER, because the SAE was trained on
# resid_post_layer_{TARGET_LAYER}. A causal head layer must be <= TARGET_LAYER.
HEAD_CAUSAL_FEATURE_SCORE_LAYER = int(TARGET_LAYER)

# Use the configured source/target blocks. Default: source Example 7 -> target Example 8.
HEAD_CAUSAL_PREVIOUS_BLOCK_INDEX = int(LFH_SOURCE_BLOCK_INDEX)
HEAD_CAUSAL_TARGET_BLOCK_INDEX = int(LFH_TARGET_BLOCK_INDEX)

# Intervention scope:
#   "target_block" edits only target/example-8 tokens. This is the cleanest
#   test of the proposed write-side mechanism.
#   "previous_block" edits only source/example tokens.
#   "previous_and_target" edits both.
#   "all_example_tokens" edits all example-block tokens.
HEAD_CAUSAL_TOKEN_SCOPE = "target_block"

# Additive alphas for the selected head's o_proj input on edited tokens.
# The effective multiplier is (1 + alpha):
#   alpha =  0.0: no intervention / baseline
#   alpha = -1.0: ablate this head on edited tokens
#   alpha =  1.0: add one extra copy of this head's contribution
HEAD_CAUSAL_HEAD_ALPHAS = [-5.0, -2.0, -1.0, 0.0, 1.0, 2.0,5.0]

# The selected model can be large. Start small, then scale up.
HEAD_CAUSAL_MAX_QUERIES = ATTN_WRITE_TEST_MAX_QUERIES if "ATTN_WRITE_TEST_MAX_QUERIES" in globals() else min(10, len(prompts_all_agnews))
HEAD_CAUSAL_MAX_VARIANTS = ATTN_WRITE_TEST_MAX_VARIANTS if "ATTN_WRITE_TEST_MAX_VARIANTS" in globals() else None

# For high-vs-low source activation summaries.
HEAD_CAUSAL_QUANTILE = 0.25

EPS = 1e-9

print("=" * 120)
print("Attention-head causal intervention config")
print("=" * 120)
print("feature_idx:", HEAD_CAUSAL_FEATURE_IDX)
print("attention layer:", HEAD_CAUSAL_LAYER)
print("attention head:", HEAD_CAUSAL_HEAD)
print("feature score layer:", HEAD_CAUSAL_FEATURE_SCORE_LAYER)
print("token scope:", HEAD_CAUSAL_TOKEN_SCOPE)
print("head alphas:", HEAD_CAUSAL_HEAD_ALPHAS)
print("max queries:", HEAD_CAUSAL_MAX_QUERIES)
print("max variants:", HEAD_CAUSAL_MAX_VARIANTS)
print("=" * 120)

if HEAD_CAUSAL_LAYER > HEAD_CAUSAL_FEATURE_SCORE_LAYER:
    raise ValueError(
        f"HEAD_CAUSAL_LAYER={HEAD_CAUSAL_LAYER} is later than "
        f"HEAD_CAUSAL_FEATURE_SCORE_LAYER={HEAD_CAUSAL_FEATURE_SCORE_LAYER}. "
        "A later attention head cannot causally affect an earlier SAE-layer activation "
        "in the same forward pass. Choose HEAD_CAUSAL_LAYER <= TARGET_LAYER."
    )


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def _safe_pearson_local(x, y):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    finite = np.isfinite(x) & np.isfinite(y)
    x = x[finite]
    y = y[finite]
    if len(x) < 3:
        return np.nan
    if np.std(x) <= EPS or np.std(y) <= EPS:
        return np.nan
    return float(pearsonr(x, y)[0])


def _attn_num_heads_and_head_dim(model, layer_idx: int):
    attn = model.get_submodule(f"model.layers.{layer_idx}.self_attn")
    num_heads = int(getattr(attn, "num_heads", model.config.num_attention_heads))
    head_dim = int(getattr(attn, "head_dim", model.config.hidden_size // num_heads))
    return num_heads, head_dim


def build_head_intervention_token_mask(
    seq_len: int,
    block_spans,
    previous_block_index: int,
    target_block_index: int,
    token_scope: str = "target_block",
):
    """
    Build a [1, seq_len] bool mask specifying which token positions receive
    the head ablation/amplification.
    """
    mask = torch.zeros((1, seq_len), dtype=torch.bool)

    prev_span = block_spans[previous_block_index]
    tgt_span = block_spans[target_block_index]

    def _mark(selection):
        idx = token_selection_to_torch_indices(selection, device=mask.device)
        mask[0, idx] = True

    if token_scope == "target_block":
        _mark(tgt_span)
    elif token_scope == "previous_block":
        _mark(prev_span)
    elif token_scope == "previous_and_target":
        _mark(prev_span)
        _mark(tgt_span)
    elif token_scope == "all_example_tokens":
        for selection in block_spans:
            _mark(selection)
    else:
        raise ValueError(f"Unknown token_scope: {token_scope}")

    return mask


@torch.no_grad()
def run_forward_capture_feature_layer_with_head_alpha(
    model,
    input_ids,
    attention_mask,
    *,
    head_layer_idx: int,
    head_idx: int,
    feature_layer_idx: int,
    token_mask: torch.Tensor,
    head_alpha: float = 0.0,
):
    """
    Run one forward pass while optionally scaling one attention head's o_proj
    input on selected tokens, then capture the feature_layer_idx residual output.

    The edited tensor is the input to self_attn.o_proj, shaped [B, S, hidden].
    It is a concatenation of per-head attention outputs before W_O.

    head_alpha semantics:
        0.0: no intervention / baseline
       -1.0: ablate selected head on selected tokens
        1.0: add one extra copy of the selected head's contribution
        alpha: effective multiplier is (1 + alpha)
    """
    if head_layer_idx > feature_layer_idx:
        raise ValueError(
            f"head_layer_idx={head_layer_idx} > feature_layer_idx={feature_layer_idx}. "
            "This cannot affect the measured feature activation in the same forward pass."
        )

    cache = {}

    feature_block = model.get_submodule(f"model.layers.{feature_layer_idx}")
    o_proj = model.get_submodule(f"model.layers.{head_layer_idx}.self_attn.o_proj")
    num_heads, head_dim = _attn_num_heads_and_head_dim(model, head_layer_idx)

    if not (0 <= int(head_idx) < num_heads):
        raise ValueError(f"head_idx={head_idx} outside [0, {num_heads}).")

    head_start = int(head_idx) * head_dim
    head_end = head_start + head_dim

    def feature_block_hook(module, inputs, output):
        x = output[0] if isinstance(output, tuple) else output
        cache["feature_layer_output"] = x.detach().clone()
        return output

    def o_proj_pre_hook(module, inputs):
        z = inputs[0]

        # alpha 0.0 is exactly no intervention.
        if float(head_alpha) == 0.0:
            return None

        if z.shape[-1] != num_heads * head_dim:
            raise RuntimeError(
                f"o_proj input last dim {z.shape[-1]} != num_heads*head_dim "
                f"{num_heads}*{head_dim}."
            )

        edit_mask = token_mask.to(device=z.device, dtype=torch.bool)
        if edit_mask.ndim != 2 or edit_mask.shape[0] != z.shape[0] or edit_mask.shape[1] != z.shape[1]:
            raise RuntimeError(
                f"token_mask shape {tuple(edit_mask.shape)} incompatible with o_proj input shape {tuple(z.shape)}."
            )

        z_edit = z.clone()
        head_slice = z_edit[..., head_start:head_end]
        effective_multiplier = 1.0 + float(head_alpha)
        scaled_head_slice = head_slice * effective_multiplier
        z_edit[..., head_start:head_end] = torch.where(edit_mask[..., None], scaled_head_slice, head_slice)

        return (z_edit,) + tuple(inputs[1:])

    h_feature = feature_block.register_forward_hook(feature_block_hook)
    h_head = o_proj.register_forward_pre_hook(o_proj_pre_hook)

    try:
        _ = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
        )
    finally:
        h_head.remove()
        h_feature.remove()

    if "feature_layer_output" not in cache:
        raise RuntimeError(f"Failed to capture feature layer {feature_layer_idx} output.")

    return cache["feature_layer_output"]


@torch.no_grad()
def measure_prompt_feature_with_head_alpha(
    prompt: str,
    *,
    feature_idx: int,
    head_layer_idx: int,
    head_idx: int,
    feature_layer_idx: int,
    head_alpha: float,
    previous_block_index: int = -2,
    target_block_index: int = -1,
    token_scope: str = "target_block",
    expected_num_blocks: int = NUM_BLOCKS_TOTAL,
):
    """
    For one prompt, run a selected attention-head intervention and measure
    feature pre-activation on previous and target exemplar blocks.
    """
    enc = tokenize_prompt_for_offsets(prompt)
    offset_mapping = enc["offset_mapping"][0].tolist()
    input_ids = enc["input_ids"].to(MODEL_INPUT_DEVICE)
    attention_mask = enc["attention_mask"].to(MODEL_INPUT_DEVICE)

    block_spans = get_example_block_token_spans(
        prompt=prompt,
        offset_mapping=offset_mapping,
        expected_num_blocks=expected_num_blocks,
    )

    token_mask = build_head_intervention_token_mask(
        seq_len=input_ids.shape[1],
        block_spans=block_spans,
        previous_block_index=previous_block_index,
        target_block_index=target_block_index,
        token_scope=token_scope,
    ).to(MODEL_INPUT_DEVICE)

    layer_output = run_forward_capture_feature_layer_with_head_alpha(
        model=model,
        input_ids=input_ids,
        attention_mask=attention_mask,
        head_layer_idx=head_layer_idx,
        head_idx=head_idx,
        feature_layer_idx=feature_layer_idx,
        token_mask=token_mask,
        head_alpha=head_alpha,
    )

    pre_f = sae_single_feature_pre_activation(
        sae=sae,
        x=layer_output,
        feature_idx=feature_idx,
    )[0].detach().float()

    prev_selection = block_spans[previous_block_index]
    target_selection = block_spans[target_block_index]
    prev_idx = token_selection_to_torch_indices(prev_selection, device=pre_f.device)
    target_idx = token_selection_to_torch_indices(target_selection, device=pre_f.device)

    return {
        "previous_feature_mean": float(pre_f.index_select(0, prev_idx).mean().detach().cpu()),
        "target_feature_mean": float(pre_f.index_select(0, target_idx).mean().detach().cpu()),
        "prev_span": block_spans[previous_block_index],
        "target_span": block_spans[target_block_index],
    }


def _querywise_corr_summary(records_df, target_col: str, prev_col: str = "baseline_previous_feature_mean"):
    """
    Compute pooled and query-wise correlations for prev_col -> target_col.
    """
    pooled = _safe_pearson_local(records_df[prev_col].values, records_df[target_col].values)

    q_corrs = []
    for q, g in records_df.groupby("query_idx"):
        r = _safe_pearson_local(g[prev_col].values, g[target_col].values)
        if np.isfinite(r):
            q_corrs.append(r)

    if len(q_corrs) == 0:
        return pooled, np.nan, np.nan, 0

    return pooled, float(np.mean(q_corrs)), float(np.std(q_corrs)), int(len(q_corrs))


def _high_low_delta_by_query(records_df, value_col: str, prev_col: str = "baseline_previous_feature_mean", quantile: float = 0.25):
    """
    For each query, compute mean(value | prev high) - mean(value | prev low).
    """
    deltas = []
    for q, g in records_df.groupby("query_idx"):
        x = g[prev_col].to_numpy(dtype=np.float64)
        y = g[value_col].to_numpy(dtype=np.float64)
        finite = np.isfinite(x) & np.isfinite(y)
        x = x[finite]
        y = y[finite]
        if len(x) < 4:
            continue
        lo = np.quantile(x, quantile)
        hi = np.quantile(x, 1.0 - quantile)
        low_mask = x <= lo
        high_mask = x >= hi
        if low_mask.sum() == 0 or high_mask.sum() == 0:
            continue
        deltas.append(float(y[high_mask].mean() - y[low_mask].mean()))
    if len(deltas) == 0:
        return np.nan, np.nan, 0
    return float(np.mean(deltas)), float(np.std(deltas)), int(len(deltas))


@torch.no_grad()
def run_attention_head_alpha_causal_test(
    *,
    feature_idx: int,
    layer_idx: int,
    head_idx: int,
    prompts_all_agnews,
    feature_score_layer: int = TARGET_LAYER,
    head_alphas=(-10.0, -5.0, -1.0, 0.0, 1.0, 5.0, 10.0),
    previous_block_index: int = -2,
    target_block_index: int = -1,
    token_scope: str = "target_block",
    max_queries=None,
    max_variants=None,
    quantile: float = 0.25,
    verbose_every: int = 10,
):
    """
    Main causal test.

    For each prompt variant:
      1. Run baseline with head_alpha=0.0.
      2. Run each requested intervention alpha.
      3. Recompute corr(prev_feature_baseline, target_feature_intervened).

    A candidate negative-write head should show:
      - ablation at alpha=-1: negative correlation weakens;
      - amplification at alpha>0: negative correlation strengthens.
    """
    feature_idx = int(feature_idx)
    layer_idx = int(layer_idx)
    head_idx = int(head_idx)
    feature_score_layer = int(feature_score_layer)

    if layer_idx > feature_score_layer:
        raise ValueError(
            f"layer_idx={layer_idx} > feature_score_layer={feature_score_layer}; "
            "the intervention cannot affect the measured feature in the same forward pass."
        )

    # De-duplicate alphas but keep order; ensure baseline alpha=0.0 is available internally.
    requested_alphas = []
    for alpha in head_alphas:
        af = float(alpha)
        if af not in requested_alphas:
            requested_alphas.append(af)
    if 0.0 not in requested_alphas:
        alphas_to_run = [0.0] + requested_alphas
    else:
        alphas_to_run = requested_alphas

    num_queries_total = len(prompts_all_agnews)
    query_indices = list(range(num_queries_total if max_queries is None else min(max_queries, num_queries_total)))

    rows = []

    for query_counter, query_idx in enumerate(query_indices, start=1):
        variants = prompts_all_agnews[query_idx]
        if max_variants is not None:
            variants = variants[:max_variants]

        for variant_idx, prompt in enumerate(variants):
            # Baseline first.
            baseline = measure_prompt_feature_with_head_alpha(
                prompt=prompt,
                feature_idx=feature_idx,
                head_layer_idx=layer_idx,
                head_idx=head_idx,
                feature_layer_idx=feature_score_layer,
                head_alpha=0.0,
                previous_block_index=previous_block_index,
                target_block_index=target_block_index,
                token_scope=token_scope,
                expected_num_blocks=NUM_BLOCKS_TOTAL,
            )

            base_prev = baseline["previous_feature_mean"]
            base_tgt = baseline["target_feature_mean"]

            for alpha in alphas_to_run:
                if float(alpha) == 0.0:
                    previous_mean = base_prev
                    target_mean = base_tgt
                else:
                    intervened = measure_prompt_feature_with_head_alpha(
                        prompt=prompt,
                        feature_idx=feature_idx,
                        head_layer_idx=layer_idx,
                        head_idx=head_idx,
                        feature_layer_idx=feature_score_layer,
                        head_alpha=float(alpha),
                        previous_block_index=previous_block_index,
                        target_block_index=target_block_index,
                        token_scope=token_scope,
                        expected_num_blocks=NUM_BLOCKS_TOTAL,
                    )
                    previous_mean = intervened["previous_feature_mean"]
                    target_mean = intervened["target_feature_mean"]

                rows.append({
                    "query_idx": int(query_idx),
                    "variant_idx": int(variant_idx),
                    "feature_idx": int(feature_idx),
                    "layer": int(layer_idx),
                    "head": int(head_idx),
                    "feature_score_layer": int(feature_score_layer),
                    "token_scope": token_scope,
                    "head_alpha": float(alpha),
                    "effective_head_multiplier": float(1.0 + float(alpha)),
                    "baseline_previous_feature_mean": float(base_prev),
                    "baseline_target_feature_mean": float(base_tgt),
                    "intervened_previous_feature_mean": float(previous_mean),
                    "intervened_target_feature_mean": float(target_mean),
                    "target_delta_intervened_minus_baseline": float(target_mean - base_tgt),
                    "previous_delta_intervened_minus_baseline": float(previous_mean - base_prev),
                })

        if verbose_every is not None and (query_counter % max(verbose_every, 1) == 0 or query_counter == len(query_indices)):
            print(f"processed query {query_counter}/{len(query_indices)}; rows={len(rows)}")

    records_df = pd.DataFrame(rows)

    # Baseline correlation summary from alpha 0.0 rows.
    baseline_rows = records_df[records_df["head_alpha"] == 0.0].copy()
    baseline_pooled_r, baseline_mean_query_r, baseline_std_query_r, baseline_n_query = _querywise_corr_summary(
        baseline_rows,
        target_col="baseline_target_feature_mean",
        prev_col="baseline_previous_feature_mean",
    )

    summary_rows = []
    for alpha in requested_alphas:
        sub = records_df[records_df["head_alpha"] == float(alpha)].copy()
        if len(sub) == 0:
            continue

        pooled_r, mean_query_r, std_query_r, n_query = _querywise_corr_summary(
            sub,
            target_col="intervened_target_feature_mean",
            prev_col="baseline_previous_feature_mean",
        )

        delta_pooled_r, delta_mean_query_r, delta_std_query_r, delta_n_query = _querywise_corr_summary(
            sub,
            target_col="target_delta_intervened_minus_baseline",
            prev_col="baseline_previous_feature_mean",
        )

        high_low_target_delta_mean, high_low_target_delta_std, high_low_n = _high_low_delta_by_query(
            sub,
            value_col="target_delta_intervened_minus_baseline",
            prev_col="baseline_previous_feature_mean",
            quantile=quantile,
        )

        high_low_target_mean, high_low_target_std, high_low_target_n = _high_low_delta_by_query(
            sub,
            value_col="intervened_target_feature_mean",
            prev_col="baseline_previous_feature_mean",
            quantile=quantile,
        )

        summary_rows.append({
            "feature_idx": int(feature_idx),
            "layer": int(layer_idx),
            "head": int(head_idx),
            "feature_score_layer": int(feature_score_layer),
            "token_scope": token_scope,
            "head_alpha": float(alpha),
            "effective_head_multiplier": float(1.0 + float(alpha)),

            "baseline_pooled_corr_prev_to_target": baseline_pooled_r,
            "baseline_mean_query_corr_prev_to_target": baseline_mean_query_r,
            "baseline_std_query_corr_prev_to_target": baseline_std_query_r,

            "intervened_pooled_corr_prev_to_target": pooled_r,
            "intervened_mean_query_corr_prev_to_target": mean_query_r,
            "intervened_std_query_corr_prev_to_target": std_query_r,

            "corr_change_pooled_intervened_minus_baseline": pooled_r - baseline_pooled_r,
            "corr_change_mean_query_intervened_minus_baseline": mean_query_r - baseline_mean_query_r,

            "corr_prev_to_target_delta_pooled": delta_pooled_r,
            "corr_prev_to_target_delta_mean_query": delta_mean_query_r,

            "mean_target_delta_intervened_minus_baseline": float(sub["target_delta_intervened_minus_baseline"].mean()),
            "median_target_delta_intervened_minus_baseline": float(sub["target_delta_intervened_minus_baseline"].median()),
            "fraction_target_delta_positive": float((sub["target_delta_intervened_minus_baseline"] > 0).mean()),

            "high_minus_low_prev_target_delta_mean": high_low_target_delta_mean,
            "high_minus_low_prev_target_delta_std": high_low_target_delta_std,
            "high_minus_low_prev_intervened_target_mean": high_low_target_mean,
            "high_minus_low_prev_intervened_target_std": high_low_target_std,
            "n_queries_for_high_low": int(high_low_n),
        })

    summary_df = pd.DataFrame(summary_rows).sort_values("head_alpha").reset_index(drop=True)

    print("=" * 120)
    print(f"Head causal test: feature={feature_idx}, layer={layer_idx}, head={head_idx}")
    print(f"Baseline pooled corr(prev, target): {baseline_pooled_r:.4f}")
    print(f"Baseline mean query-wise corr(prev, target): {baseline_mean_query_r:.4f}")
    print("Interpretation:")
    print("  alpha=0.0 leaves this head unchanged on target tokens.")
    print("  alpha=-1.0 ablates this head on target tokens.")
    print("  alpha>0 adds extra copies of this head's contribution on target tokens.")
    print("  effective multiplier = 1 + alpha.")
    print("  If the head implements negative write, ablation should make a negative corr less negative,")
    print("  while amplification should make it more negative.")
    print("=" * 120)
    display(summary_df)

    return summary_df, records_df


# ------------------------------------------------------------
# Plotting helpers (Updated with Custom Font Properties)
# ------------------------------------------------------------

# aaaaa
def plot_head_causal_dose_response(summary_df):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=summary_df["head_alpha"],
        y=summary_df["intervened_mean_query_corr_prev_to_target"],
        mode="lines+markers",
        name="intervened corr(prev, target), mean query-wise",
        hovertemplate="alpha=%{x}<br>corr=%{y:.4f}<extra></extra>",
    ))

    baseline = float(summary_df["baseline_mean_query_corr_prev_to_target"].iloc[0])
    fig.add_hline(y=baseline, line_dash="dash", annotation_text="baseline corr")
    fig.add_vline(x=0.0, line_dash="dash", annotation_text="no intervention")

    fig.update_layout(
        font=dict(family="Times New Roman"),
        title=(
            "Attention-Head Intervention Response on Local Feature Homeostasis<br>"
            f"Model=Gemma2-9B-IT, Feature={int(summary_df['feature_idx'].iloc[0])}, "
            f"Layer={int(summary_df['layer'].iloc[0])}, "
            f"Head={int(summary_df['head'].iloc[0])}"
        ),
        title_font_size=26,
        xaxis_title="Intervention Strength (Alpha) on Target-Block Tokens",
        xaxis_title_font_size=22,
        yaxis_title="Mean corr(source feature, target feature)",
        yaxis_title_font_size=22,
        width=900,
        height=600,
        hovermode="x unified",
    )
    
    # --- ADDED: 4x Resolution Config for Interactive 'Download Plot' Button ---
    config = {
        'toImageButtonOptions': {
            'format': 'png',         # one of png, svg, jpeg, webp
            'filename': 'head_causal_dose_response',
            'height': 600,
            'width': 900,
            'scale': 4               # Multiply resolution by 4
        }
    }
    
    fig.show(config=config)
    
    # --- ADDED: Optional direct save to disk at 4x resolution ---
    # Uncomment the line below to save programmatically (requires pip install kaleido)
    # fig.write_image("head_causal_dose_response_highres.png", scale=4)

    return fig


def plot_head_causal_delta_response(summary_df):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=summary_df["head_alpha"],
        y=summary_df["corr_prev_to_target_delta_mean_query"],
        mode="lines+markers",
        name="corr(prev, target_delta)",
        hovertemplate="alpha=%{x}<br>corr=%{y:.4f}<extra></extra>",
    ))

    fig.add_trace(go.Scatter(
        x=summary_df["head_alpha"],
        y=summary_df["high_minus_low_prev_target_delta_mean"],
        mode="lines+markers",
        name="high-prev minus low-prev target delta",
        hovertemplate="alpha=%{x}<br>high-low delta=%{y:.4f}<extra></extra>",
    ))

    fig.add_hline(y=0.0, line_dash="dash")
    fig.add_vline(x=0.0, line_dash="dash", annotation_text="no intervention")

    fig.update_layout(
        font=dict(family="Times New Roman"),
        title="Does the intervention affect high-previous-activation variants differently?",
        title_font_size=26,
        xaxis_title="Head alpha on target-block tokens (effective multiplier = 1 + alpha)",
        xaxis_title_font_size=22,
        yaxis_title="Effect size",
        yaxis_title_font_size=22,
        width=900,
        height=600,
        hovermode="x unified",
    )
    fig.show()
    return fig


def plot_head_intervention_scatter(records_df, alpha: float):
    sub = records_df[records_df["head_alpha"] == float(alpha)].copy()
    if len(sub) == 0:
        raise ValueError(f"No rows for alpha={alpha}")

    x = sub["baseline_previous_feature_mean"].to_numpy(dtype=np.float64)
    y_base = sub["baseline_target_feature_mean"].to_numpy(dtype=np.float64)
    y_int = sub["intervened_target_feature_mean"].to_numpy(dtype=np.float64)

    r_base = _safe_pearson_local(x, y_base)
    r_int = _safe_pearson_local(x, y_int)

    df_plot = pd.DataFrame({
        "previous_feature_mean": np.concatenate([x, x]),
        "target_feature_mean": np.concatenate([y_base, y_int]),
        "condition": ["baseline"] * len(x) + [f"head_alpha={alpha}"] * len(x),
        "query_idx": np.concatenate([sub["query_idx"].values, sub["query_idx"].values]).astype(str),
        "variant_idx": np.concatenate([sub["variant_idx"].values, sub["variant_idx"].values]),
    })

    fig = px.scatter(
        df_plot,
        x="previous_feature_mean",
        y="target_feature_mean",
        color="condition",
        symbol="condition",
        hover_data=["query_idx", "variant_idx"],
        title=(
            f"Source-to-target feature relation before/after head intervention<br>"
            f"alpha={alpha}, baseline r={r_base:.3f}, intervened r={r_int:.3f}"
        ),
    )

    fig.update_layout(
        font=dict(family="Times New Roman"),
        title_font_size=26,
        width=900,
        height=600,
        xaxis_title="Mean SAE pre-activation on source exemplar",
        xaxis_title_font_size=22,
        yaxis_title="Mean SAE pre-activation on target exemplar",
        yaxis_title_font_size=22,
    )
    fig.show()
    return fig


# ------------------------------------------------------------
# Run the head intervention test
# ------------------------------------------------------------

head_causal_summary_df, head_causal_records_df = run_attention_head_alpha_causal_test(
    feature_idx=HEAD_CAUSAL_FEATURE_IDX,
    layer_idx=HEAD_CAUSAL_LAYER,
    head_idx=HEAD_CAUSAL_HEAD,
    prompts_all_agnews=prompts_all_agnews,
    feature_score_layer=HEAD_CAUSAL_FEATURE_SCORE_LAYER,
    head_alphas=HEAD_CAUSAL_HEAD_ALPHAS,
    previous_block_index=HEAD_CAUSAL_PREVIOUS_BLOCK_INDEX,
    target_block_index=HEAD_CAUSAL_TARGET_BLOCK_INDEX,
    token_scope=HEAD_CAUSAL_TOKEN_SCOPE,
    max_queries=HEAD_CAUSAL_MAX_QUERIES,
    max_variants=HEAD_CAUSAL_MAX_VARIANTS,
    quantile=HEAD_CAUSAL_QUANTILE,
    verbose_every=5,
)

_ = plot_head_causal_dose_response(head_causal_summary_df)
_ = plot_head_causal_delta_response(head_causal_summary_df)

# Plot ablation and amplification views if present.
if -1.0 in set(head_causal_records_df["head_alpha"].astype(float)):
    _ = plot_head_intervention_scatter(head_causal_records_df, alpha=-1.0)
if 1.0 in set(head_causal_records_df["head_alpha"].astype(float)):
    _ = plot_head_intervention_scatter(head_causal_records_df, alpha=1.0)

# BLOCK_TOKEN_OFFSETS_FROM_END = [-2]

In [ ]:
# ============================================================
# 14. Causal validation: ablate/amplify one attention head and remeasure
#      source-exemplar -> target-exemplar feature correlation
# ============================================================

import numpy as np
import pandas as pd
import torch
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import pearsonr

# ------------------------------------------------------------
# User-editable config
# ------------------------------------------------------------

# Feature to test. By default, use the same feature as the observational
# negative-write test above.
HEAD_CAUSAL_FEATURE_IDX = int(ATTN_WRITE_TEST_FEATURE_IDX) if "ATTN_WRITE_TEST_FEATURE_IDX" in globals() else int(target_feature)

# Attention head to test. By default, use the top head from the observational
# negative-write ranking, if available.
HEAD_CAUSAL_LAYER = int(ATTN_WRITE_TEST_LAYER) if "ATTN_WRITE_TEST_LAYER" in globals() else int(TARGET_LAYER)

# aaaaa
HEAD_CAUSAL_HEAD = int(TOP_NEGATIVE_WRITE_HEAD) if "TOP_NEGATIVE_WRITE_HEAD" in globals() else 0
HEAD_CAUSAL_HEAD = 8

# The SAE feature is measured at TARGET_LAYER, because the SAE was trained on
# resid_post_layer_{TARGET_LAYER}. A causal head layer must be <= TARGET_LAYER.
HEAD_CAUSAL_FEATURE_SCORE_LAYER = int(TARGET_LAYER)

# Use the configured source/target blocks. Default: source Example 7 -> target Example 8.
HEAD_CAUSAL_PREVIOUS_BLOCK_INDEX = int(LFH_SOURCE_BLOCK_INDEX)
HEAD_CAUSAL_TARGET_BLOCK_INDEX = int(LFH_TARGET_BLOCK_INDEX)

# Intervention scope:
#   "target_block" edits only target/example-8 tokens. This is the cleanest
#   test of the proposed write-side mechanism.
#   "previous_block" edits only source/example tokens.
#   "previous_and_target" edits both.
#   "all_example_tokens" edits all example-block tokens.
HEAD_CAUSAL_TOKEN_SCOPE = "target_block"

# Additive alphas for the selected head's o_proj input on edited tokens.
# The effective multiplier is (1 + alpha):
#   alpha =  0.0: no intervention / baseline
#   alpha = -1.0: ablate this head on edited tokens
#   alpha =  1.0: add one extra copy of this head's contribution
HEAD_CAUSAL_HEAD_ALPHAS = [-5.0, -2.0, -1.0, 0.0, 1.0, 2.0,5.0]
HEAD_CAUSAL_HEAD_ALPHAS = [-5.0, -2.0, -1.0, 0.0, 1.0, 2.0,5.0]

# The selected model can be large. Start small, then scale up.
HEAD_CAUSAL_MAX_QUERIES = ATTN_WRITE_TEST_MAX_QUERIES if "ATTN_WRITE_TEST_MAX_QUERIES" in globals() else min(10, len(prompts_all_agnews))
HEAD_CAUSAL_MAX_VARIANTS = ATTN_WRITE_TEST_MAX_VARIANTS if "ATTN_WRITE_TEST_MAX_VARIANTS" in globals() else None

# For high-vs-low source activation summaries.
HEAD_CAUSAL_QUANTILE = 0.25

EPS = 1e-9

print("=" * 120)
print("Attention-head causal intervention config")
print("=" * 120)
print("feature_idx:", HEAD_CAUSAL_FEATURE_IDX)
print("attention layer:", HEAD_CAUSAL_LAYER)
print("attention head:", HEAD_CAUSAL_HEAD)
print("feature score layer:", HEAD_CAUSAL_FEATURE_SCORE_LAYER)
print("token scope:", HEAD_CAUSAL_TOKEN_SCOPE)
print("head alphas:", HEAD_CAUSAL_HEAD_ALPHAS)
print("max queries:", HEAD_CAUSAL_MAX_QUERIES)
print("max variants:", HEAD_CAUSAL_MAX_VARIANTS)
print("=" * 120)

if HEAD_CAUSAL_LAYER > HEAD_CAUSAL_FEATURE_SCORE_LAYER:
    raise ValueError(
        f"HEAD_CAUSAL_LAYER={HEAD_CAUSAL_LAYER} is later than "
        f"HEAD_CAUSAL_FEATURE_SCORE_LAYER={HEAD_CAUSAL_FEATURE_SCORE_LAYER}. "
        "A later attention head cannot causally affect an earlier SAE-layer activation "
        "in the same forward pass. Choose HEAD_CAUSAL_LAYER <= TARGET_LAYER."
    )


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def _safe_pearson_local(x, y):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    finite = np.isfinite(x) & np.isfinite(y)
    x = x[finite]
    y = y[finite]
    if len(x) < 3:
        return np.nan
    if np.std(x) <= EPS or np.std(y) <= EPS:
        return np.nan
    return float(pearsonr(x, y)[0])


def _attn_num_heads_and_head_dim(model, layer_idx: int):
    attn = model.get_submodule(f"model.layers.{layer_idx}.self_attn")
    num_heads = int(getattr(attn, "num_heads", model.config.num_attention_heads))
    head_dim = int(getattr(attn, "head_dim", model.config.hidden_size // num_heads))
    return num_heads, head_dim


def build_head_intervention_token_mask(
    seq_len: int,
    block_spans,
    previous_block_index: int,
    target_block_index: int,
    token_scope: str = "target_block",
):
    """
    Build a [1, seq_len] bool mask specifying which token positions receive
    the head ablation/amplification.
    """
    mask = torch.zeros((1, seq_len), dtype=torch.bool)

    prev_span = block_spans[previous_block_index]
    tgt_span = block_spans[target_block_index]

    def _mark(selection):
        idx = token_selection_to_torch_indices(selection, device=mask.device)
        mask[0, idx] = True

    if token_scope == "target_block":
        _mark(tgt_span)
    elif token_scope == "previous_block":
        _mark(prev_span)
    elif token_scope == "previous_and_target":
        _mark(prev_span)
        _mark(tgt_span)
    elif token_scope == "all_example_tokens":
        for selection in block_spans:
            _mark(selection)
    else:
        raise ValueError(f"Unknown token_scope: {token_scope}")

    return mask


@torch.no_grad()
def run_forward_capture_feature_layer_with_head_alpha(
    model,
    input_ids,
    attention_mask,
    *,
    head_layer_idx: int,
    head_idx: int,
    feature_layer_idx: int,
    token_mask: torch.Tensor,
    head_alpha: float = 0.0,
):
    """
    Run one forward pass while optionally scaling one attention head's o_proj
    input on selected tokens, then capture the feature_layer_idx residual output.

    The edited tensor is the input to self_attn.o_proj, shaped [B, S, hidden].
    It is a concatenation of per-head attention outputs before W_O.

    head_alpha semantics:
        0.0: no intervention / baseline
       -1.0: ablate selected head on selected tokens
        1.0: add one extra copy of the selected head's contribution
        alpha: effective multiplier is (1 + alpha)
    """
    if head_layer_idx > feature_layer_idx:
        raise ValueError(
            f"head_layer_idx={head_layer_idx} > feature_layer_idx={feature_layer_idx}. "
            "This cannot affect the measured feature activation in the same forward pass."
        )

    cache = {}

    feature_block = model.get_submodule(f"model.layers.{feature_layer_idx}")
    o_proj = model.get_submodule(f"model.layers.{head_layer_idx}.self_attn.o_proj")
    num_heads, head_dim = _attn_num_heads_and_head_dim(model, head_layer_idx)

    if not (0 <= int(head_idx) < num_heads):
        raise ValueError(f"head_idx={head_idx} outside [0, {num_heads}).")

    head_start = int(head_idx) * head_dim
    head_end = head_start + head_dim

    def feature_block_hook(module, inputs, output):
        x = output[0] if isinstance(output, tuple) else output
        cache["feature_layer_output"] = x.detach().clone()
        return output

    def o_proj_pre_hook(module, inputs):
        z = inputs[0]

        # alpha 0.0 is exactly no intervention.
        if float(head_alpha) == 0.0:
            return None

        if z.shape[-1] != num_heads * head_dim:
            raise RuntimeError(
                f"o_proj input last dim {z.shape[-1]} != num_heads*head_dim "
                f"{num_heads}*{head_dim}."
            )

        edit_mask = token_mask.to(device=z.device, dtype=torch.bool)
        if edit_mask.ndim != 2 or edit_mask.shape[0] != z.shape[0] or edit_mask.shape[1] != z.shape[1]:
            raise RuntimeError(
                f"token_mask shape {tuple(edit_mask.shape)} incompatible with o_proj input shape {tuple(z.shape)}."
            )

        z_edit = z.clone()
        head_slice = z_edit[..., head_start:head_end]
        effective_multiplier = 1.0 + float(head_alpha)
        scaled_head_slice = head_slice * effective_multiplier
        z_edit[..., head_start:head_end] = torch.where(edit_mask[..., None], scaled_head_slice, head_slice)

        return (z_edit,) + tuple(inputs[1:])

    h_feature = feature_block.register_forward_hook(feature_block_hook)
    h_head = o_proj.register_forward_pre_hook(o_proj_pre_hook)

    try:
        _ = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
        )
    finally:
        h_head.remove()
        h_feature.remove()

    if "feature_layer_output" not in cache:
        raise RuntimeError(f"Failed to capture feature layer {feature_layer_idx} output.")

    return cache["feature_layer_output"]


@torch.no_grad()
def measure_prompt_feature_with_head_alpha(
    prompt: str,
    *,
    feature_idx: int,
    head_layer_idx: int,
    head_idx: int,
    feature_layer_idx: int,
    head_alpha: float,
    previous_block_index: int = -2,
    target_block_index: int = -1,
    token_scope: str = "target_block",
    expected_num_blocks: int = NUM_BLOCKS_TOTAL,
):
    """
    For one prompt, run a selected attention-head intervention and measure
    feature pre-activation on previous and target exemplar blocks.
    """
    enc = tokenize_prompt_for_offsets(prompt)
    offset_mapping = enc["offset_mapping"][0].tolist()
    input_ids = enc["input_ids"].to(MODEL_INPUT_DEVICE)
    attention_mask = enc["attention_mask"].to(MODEL_INPUT_DEVICE)

    block_spans = get_example_block_token_spans(
        prompt=prompt,
        offset_mapping=offset_mapping,
        expected_num_blocks=expected_num_blocks,
    )

    token_mask = build_head_intervention_token_mask(
        seq_len=input_ids.shape[1],
        block_spans=block_spans,
        previous_block_index=previous_block_index,
        target_block_index=target_block_index,
        token_scope=token_scope,
    ).to(MODEL_INPUT_DEVICE)

    layer_output = run_forward_capture_feature_layer_with_head_alpha(
        model=model,
        input_ids=input_ids,
        attention_mask=attention_mask,
        head_layer_idx=head_layer_idx,
        head_idx=head_idx,
        feature_layer_idx=feature_layer_idx,
        token_mask=token_mask,
        head_alpha=head_alpha,
    )

    pre_f = sae_single_feature_pre_activation(
        sae=sae,
        x=layer_output,
        feature_idx=feature_idx,
    )[0].detach().float()

    prev_selection = block_spans[previous_block_index]
    target_selection = block_spans[target_block_index]
    prev_idx = token_selection_to_torch_indices(prev_selection, device=pre_f.device)
    target_idx = token_selection_to_torch_indices(target_selection, device=pre_f.device)

    return {
        "previous_feature_mean": float(pre_f.index_select(0, prev_idx).mean().detach().cpu()),
        "target_feature_mean": float(pre_f.index_select(0, target_idx).mean().detach().cpu()),
        "prev_span": block_spans[previous_block_index],
        "target_span": block_spans[target_block_index],
    }


def _querywise_corr_summary(records_df, target_col: str, prev_col: str = "baseline_previous_feature_mean"):
    """
    Compute pooled and query-wise correlations for prev_col -> target_col.
    """
    pooled = _safe_pearson_local(records_df[prev_col].values, records_df[target_col].values)

    q_corrs = []
    for q, g in records_df.groupby("query_idx"):
        r = _safe_pearson_local(g[prev_col].values, g[target_col].values)
        if np.isfinite(r):
            q_corrs.append(r)

    if len(q_corrs) == 0:
        return pooled, np.nan, np.nan, 0

    return pooled, float(np.mean(q_corrs)), float(np.std(q_corrs)), int(len(q_corrs))


def _high_low_delta_by_query(records_df, value_col: str, prev_col: str = "baseline_previous_feature_mean", quantile: float = 0.25):
    """
    For each query, compute mean(value | prev high) - mean(value | prev low).
    """
    deltas = []
    for q, g in records_df.groupby("query_idx"):
        x = g[prev_col].to_numpy(dtype=np.float64)
        y = g[value_col].to_numpy(dtype=np.float64)
        finite = np.isfinite(x) & np.isfinite(y)
        x = x[finite]
        y = y[finite]
        if len(x) < 4:
            continue
        lo = np.quantile(x, quantile)
        hi = np.quantile(x, 1.0 - quantile)
        low_mask = x <= lo
        high_mask = x >= hi
        if low_mask.sum() == 0 or high_mask.sum() == 0:
            continue
        deltas.append(float(y[high_mask].mean() - y[low_mask].mean()))
    if len(deltas) == 0:
        return np.nan, np.nan, 0
    return float(np.mean(deltas)), float(np.std(deltas)), int(len(deltas))


@torch.no_grad()
def run_attention_head_alpha_causal_test(
    *,
    feature_idx: int,
    layer_idx: int,
    head_idx: int,
    prompts_all_agnews,
    feature_score_layer: int = TARGET_LAYER,
    head_alphas=(-10.0, -5.0, -1.0, 0.0, 1.0, 5.0, 10.0),
    previous_block_index: int = -2,
    target_block_index: int = -1,
    token_scope: str = "target_block",
    max_queries=None,
    max_variants=None,
    quantile: float = 0.25,
    verbose_every: int = 10,
):
    """
    Main causal test.

    For each prompt variant:
      1. Run baseline with head_alpha=0.0.
      2. Run each requested intervention alpha.
      3. Recompute corr(prev_feature_baseline, target_feature_intervened).

    A candidate negative-write head should show:
      - ablation at alpha=-1: negative correlation weakens;
      - amplification at alpha>0: negative correlation strengthens.
    """
    feature_idx = int(feature_idx)
    layer_idx = int(layer_idx)
    head_idx = int(head_idx)
    feature_score_layer = int(feature_score_layer)

    if layer_idx > feature_score_layer:
        raise ValueError(
            f"layer_idx={layer_idx} > feature_score_layer={feature_score_layer}; "
            "the intervention cannot affect the measured feature in the same forward pass."
        )

    # De-duplicate alphas but keep order; ensure baseline alpha=0.0 is available internally.
    requested_alphas = []
    for alpha in head_alphas:
        af = float(alpha)
        if af not in requested_alphas:
            requested_alphas.append(af)
    if 0.0 not in requested_alphas:
        alphas_to_run = [0.0] + requested_alphas
    else:
        alphas_to_run = requested_alphas

    num_queries_total = len(prompts_all_agnews)
    query_indices = list(range(num_queries_total if max_queries is None else min(max_queries, num_queries_total)))

    rows = []

    for query_counter, query_idx in enumerate(query_indices, start=1):
        variants = prompts_all_agnews[query_idx]
        if max_variants is not None:
            variants = variants[:max_variants]

        for variant_idx, prompt in enumerate(variants):
            # Baseline first.
            baseline = measure_prompt_feature_with_head_alpha(
                prompt=prompt,
                feature_idx=feature_idx,
                head_layer_idx=layer_idx,
                head_idx=head_idx,
                feature_layer_idx=feature_score_layer,
                head_alpha=0.0,
                previous_block_index=previous_block_index,
                target_block_index=target_block_index,
                token_scope=token_scope,
                expected_num_blocks=NUM_BLOCKS_TOTAL,
            )

            base_prev = baseline["previous_feature_mean"]
            base_tgt = baseline["target_feature_mean"]

            for alpha in alphas_to_run:
                if float(alpha) == 0.0:
                    previous_mean = base_prev
                    target_mean = base_tgt
                else:
                    intervened = measure_prompt_feature_with_head_alpha(
                        prompt=prompt,
                        feature_idx=feature_idx,
                        head_layer_idx=layer_idx,
                        head_idx=head_idx,
                        feature_layer_idx=feature_score_layer,
                        head_alpha=float(alpha),
                        previous_block_index=previous_block_index,
                        target_block_index=target_block_index,
                        token_scope=token_scope,
                        expected_num_blocks=NUM_BLOCKS_TOTAL,
                    )
                    previous_mean = intervened["previous_feature_mean"]
                    target_mean = intervened["target_feature_mean"]

                rows.append({
                    "query_idx": int(query_idx),
                    "variant_idx": int(variant_idx),
                    "feature_idx": int(feature_idx),
                    "layer": int(layer_idx),
                    "head": int(head_idx),
                    "feature_score_layer": int(feature_score_layer),
                    "token_scope": token_scope,
                    "head_alpha": float(alpha),
                    "effective_head_multiplier": float(1.0 + float(alpha)),
                    "baseline_previous_feature_mean": float(base_prev),
                    "baseline_target_feature_mean": float(base_tgt),
                    "intervened_previous_feature_mean": float(previous_mean),
                    "intervened_target_feature_mean": float(target_mean),
                    "target_delta_intervened_minus_baseline": float(target_mean - base_tgt),
                    "previous_delta_intervened_minus_baseline": float(previous_mean - base_prev),
                })

        if verbose_every is not None and (query_counter % max(verbose_every, 1) == 0 or query_counter == len(query_indices)):
            print(f"processed query {query_counter}/{len(query_indices)}; rows={len(rows)}")

    records_df = pd.DataFrame(rows)

    # Baseline correlation summary from alpha 0.0 rows.
    baseline_rows = records_df[records_df["head_alpha"] == 0.0].copy()
    baseline_pooled_r, baseline_mean_query_r, baseline_std_query_r, baseline_n_query = _querywise_corr_summary(
        baseline_rows,
        target_col="baseline_target_feature_mean",
        prev_col="baseline_previous_feature_mean",
    )

    summary_rows = []
    for alpha in requested_alphas:
        sub = records_df[records_df["head_alpha"] == float(alpha)].copy()
        if len(sub) == 0:
            continue

        pooled_r, mean_query_r, std_query_r, n_query = _querywise_corr_summary(
            sub,
            target_col="intervened_target_feature_mean",
            prev_col="baseline_previous_feature_mean",
        )

        delta_pooled_r, delta_mean_query_r, delta_std_query_r, delta_n_query = _querywise_corr_summary(
            sub,
            target_col="target_delta_intervened_minus_baseline",
            prev_col="baseline_previous_feature_mean",
        )

        high_low_target_delta_mean, high_low_target_delta_std, high_low_n = _high_low_delta_by_query(
            sub,
            value_col="target_delta_intervened_minus_baseline",
            prev_col="baseline_previous_feature_mean",
            quantile=quantile,
        )

        high_low_target_mean, high_low_target_std, high_low_target_n = _high_low_delta_by_query(
            sub,
            value_col="intervened_target_feature_mean",
            prev_col="baseline_previous_feature_mean",
            quantile=quantile,
        )

        summary_rows.append({
            "feature_idx": int(feature_idx),
            "layer": int(layer_idx),
            "head": int(head_idx),
            "feature_score_layer": int(feature_score_layer),
            "token_scope": token_scope,
            "head_alpha": float(alpha),
            "effective_head_multiplier": float(1.0 + float(alpha)),

            "baseline_pooled_corr_prev_to_target": baseline_pooled_r,
            "baseline_mean_query_corr_prev_to_target": baseline_mean_query_r,
            "baseline_std_query_corr_prev_to_target": baseline_std_query_r,

            "intervened_pooled_corr_prev_to_target": pooled_r,
            "intervened_mean_query_corr_prev_to_target": mean_query_r,
            "intervened_std_query_corr_prev_to_target": std_query_r,

            "corr_change_pooled_intervened_minus_baseline": pooled_r - baseline_pooled_r,
            "corr_change_mean_query_intervened_minus_baseline": mean_query_r - baseline_mean_query_r,

            "corr_prev_to_target_delta_pooled": delta_pooled_r,
            "corr_prev_to_target_delta_mean_query": delta_mean_query_r,

            "mean_target_delta_intervened_minus_baseline": float(sub["target_delta_intervened_minus_baseline"].mean()),
            "median_target_delta_intervened_minus_baseline": float(sub["target_delta_intervened_minus_baseline"].median()),
            "fraction_target_delta_positive": float((sub["target_delta_intervened_minus_baseline"] > 0).mean()),

            "high_minus_low_prev_target_delta_mean": high_low_target_delta_mean,
            "high_minus_low_prev_target_delta_std": high_low_target_delta_std,
            "high_minus_low_prev_intervened_target_mean": high_low_target_mean,
            "high_minus_low_prev_intervened_target_std": high_low_target_std,
            "n_queries_for_high_low": int(high_low_n),
        })

    summary_df = pd.DataFrame(summary_rows).sort_values("head_alpha").reset_index(drop=True)

    print("=" * 120)
    print(f"Head causal test: feature={feature_idx}, layer={layer_idx}, head={head_idx}")
    print(f"Baseline pooled corr(prev, target): {baseline_pooled_r:.4f}")
    print(f"Baseline mean query-wise corr(prev, target): {baseline_mean_query_r:.4f}")
    print("Interpretation:")
    print("  alpha=0.0 leaves this head unchanged on target tokens.")
    print("  alpha=-1.0 ablates this head on target tokens.")
    print("  alpha>0 adds extra copies of this head's contribution on target tokens.")
    print("  effective multiplier = 1 + alpha.")
    print("  If the head implements negative write, ablation should make a negative corr less negative,")
    print("  while amplification should make it more negative.")
    print("=" * 120)
    display(summary_df)

    return summary_df, records_df


# ------------------------------------------------------------
# Plotting helpers (Updated with Custom Font Properties)
# ------------------------------------------------------------

# aaaaa
def plot_head_causal_dose_response(summary_df):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=summary_df["head_alpha"],
        y=summary_df["intervened_mean_query_corr_prev_to_target"],
        mode="lines+markers",
        name="intervened corr(prev, target), mean query-wise",
        hovertemplate="alpha=%{x}<br>corr=%{y:.4f}<extra></extra>",
    ))

    baseline = float(summary_df["baseline_mean_query_corr_prev_to_target"].iloc[0])
    fig.add_hline(y=baseline, line_dash="dash", annotation_text="baseline corr")
    fig.add_vline(x=0.0, line_dash="dash", annotation_text="no intervention")

    fig.update_layout(
        font=dict(family="Times New Roman"),
        title=(
            "Attention-Head Intervention Response on Local Feature Homeostasis<br>"
            f"Model=Gemma2-9B-IT, Feature={int(summary_df['feature_idx'].iloc[0])}, "
            f"Layer={int(summary_df['layer'].iloc[0])}, "
            f"Head={int(summary_df['head'].iloc[0])}"
        ),
        title_font_size=26,
        xaxis_title="Intervention Strength (Alpha) on Target-Block Tokens",
        xaxis_title_font_size=22,
        yaxis_title="Mean corr(source feature, target feature)",
        yaxis_title_font_size=22,
        width=900,
        height=600,
        hovermode="x unified",
    )
    
    # --- ADDED: 4x Resolution Config for Interactive 'Download Plot' Button ---
    config = {
        'toImageButtonOptions': {
            'format': 'png',         # one of png, svg, jpeg, webp
            'filename': 'head_causal_dose_response',
            'height': 600,
            'width': 900,
            'scale': 4               # Multiply resolution by 4
        }
    }
    
    fig.show(config=config)
    
    # --- ADDED: Optional direct save to disk at 4x resolution ---
    # Uncomment the line below to save programmatically (requires pip install kaleido)
    # fig.write_image("head_causal_dose_response_highres.png", scale=4)

    return fig


def plot_head_causal_delta_response(summary_df):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=summary_df["head_alpha"],
        y=summary_df["corr_prev_to_target_delta_mean_query"],
        mode="lines+markers",
        name="corr(prev, target_delta)",
        hovertemplate="alpha=%{x}<br>corr=%{y:.4f}<extra></extra>",
    ))

    fig.add_trace(go.Scatter(
        x=summary_df["head_alpha"],
        y=summary_df["high_minus_low_prev_target_delta_mean"],
        mode="lines+markers",
        name="high-prev minus low-prev target delta",
        hovertemplate="alpha=%{x}<br>high-low delta=%{y:.4f}<extra></extra>",
    ))

    fig.add_hline(y=0.0, line_dash="dash")
    fig.add_vline(x=0.0, line_dash="dash", annotation_text="no intervention")

    fig.update_layout(
        font=dict(family="Times New Roman"),
        title="Does the intervention affect high-previous-activation variants differently?",
        title_font_size=26,
        xaxis_title="Head alpha on target-block tokens (effective multiplier = 1 + alpha)",
        xaxis_title_font_size=22,
        yaxis_title="Effect size",
        yaxis_title_font_size=22,
        width=900,
        height=600,
        hovermode="x unified",
    )
    fig.show()
    return fig


def plot_head_intervention_scatter(records_df, alpha: float):
    sub = records_df[records_df["head_alpha"] == float(alpha)].copy()
    if len(sub) == 0:
        raise ValueError(f"No rows for alpha={alpha}")

    x = sub["baseline_previous_feature_mean"].to_numpy(dtype=np.float64)
    y_base = sub["baseline_target_feature_mean"].to_numpy(dtype=np.float64)
    y_int = sub["intervened_target_feature_mean"].to_numpy(dtype=np.float64)

    r_base = _safe_pearson_local(x, y_base)
    r_int = _safe_pearson_local(x, y_int)

    df_plot = pd.DataFrame({
        "previous_feature_mean": np.concatenate([x, x]),
        "target_feature_mean": np.concatenate([y_base, y_int]),
        "condition": ["baseline"] * len(x) + [f"head_alpha={alpha}"] * len(x),
        "query_idx": np.concatenate([sub["query_idx"].values, sub["query_idx"].values]).astype(str),
        "variant_idx": np.concatenate([sub["variant_idx"].values, sub["variant_idx"].values]),
    })

    fig = px.scatter(
        df_plot,
        x="previous_feature_mean",
        y="target_feature_mean",
        color="condition",
        symbol="condition",
        hover_data=["query_idx", "variant_idx"],
        title=(
            f"Source-to-target feature relation before/after head intervention<br>"
            f"alpha={alpha}, baseline r={r_base:.3f}, intervened r={r_int:.3f}"
        ),
    )

    fig.update_layout(
        font=dict(family="Times New Roman"),
        title_font_size=26,
        width=900,
        height=600,
        xaxis_title="Mean SAE pre-activation on source exemplar",
        xaxis_title_font_size=22,
        yaxis_title="Mean SAE pre-activation on target exemplar",
        yaxis_title_font_size=22,
    )
    fig.show()
    return fig


# ------------------------------------------------------------
# Run the head intervention test
# ------------------------------------------------------------

head_causal_summary_df, head_causal_records_df = run_attention_head_alpha_causal_test(
    feature_idx=HEAD_CAUSAL_FEATURE_IDX,
    layer_idx=HEAD_CAUSAL_LAYER,
    head_idx=HEAD_CAUSAL_HEAD,
    prompts_all_agnews=prompts_all_agnews,
    feature_score_layer=HEAD_CAUSAL_FEATURE_SCORE_LAYER,
    head_alphas=HEAD_CAUSAL_HEAD_ALPHAS,
    previous_block_index=HEAD_CAUSAL_PREVIOUS_BLOCK_INDEX,
    target_block_index=HEAD_CAUSAL_TARGET_BLOCK_INDEX,
    token_scope=HEAD_CAUSAL_TOKEN_SCOPE,
    max_queries=HEAD_CAUSAL_MAX_QUERIES,
    max_variants=HEAD_CAUSAL_MAX_VARIANTS,
    quantile=HEAD_CAUSAL_QUANTILE,
    verbose_every=5,
)

_ = plot_head_causal_dose_response(head_causal_summary_df)
_ = plot_head_causal_delta_response(head_causal_summary_df)

# Plot ablation and amplification views if present.
if -1.0 in set(head_causal_records_df["head_alpha"].astype(float)):
    _ = plot_head_intervention_scatter(head_causal_records_df, alpha=-1.0)
if 1.0 in set(head_causal_records_df["head_alpha"].astype(float)):
    _ = plot_head_intervention_scatter(head_causal_records_df, alpha=1.0)

In [ ]:
# ============================================================
# 14. Causal validation: ablate/amplify one attention head and remeasure
#      source-exemplar -> target-exemplar feature correlation
# ============================================================

import numpy as np
import pandas as pd
import torch
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import pearsonr

# ------------------------------------------------------------
# User-editable config
# ------------------------------------------------------------

# Feature to test. By default, use the same feature as the observational
# negative-write test above.
HEAD_CAUSAL_FEATURE_IDX = int(ATTN_WRITE_TEST_FEATURE_IDX) if "ATTN_WRITE_TEST_FEATURE_IDX" in globals() else int(target_feature)

# Attention head to test. By default, use the top head from the observational
# negative-write ranking, if available.
HEAD_CAUSAL_LAYER = int(ATTN_WRITE_TEST_LAYER) if "ATTN_WRITE_TEST_LAYER" in globals() else int(TARGET_LAYER)

# aaaaa
HEAD_CAUSAL_HEAD = int(TOP_NEGATIVE_WRITE_HEAD) if "TOP_NEGATIVE_WRITE_HEAD" in globals() else 0
HEAD_CAUSAL_HEAD = 8

# The SAE feature is measured at TARGET_LAYER, because the SAE was trained on
# resid_post_layer_{TARGET_LAYER}. A causal head layer must be <= TARGET_LAYER.
HEAD_CAUSAL_FEATURE_SCORE_LAYER = int(TARGET_LAYER)

# Use the configured source/target blocks. Default: source Example 7 -> target Example 8.
HEAD_CAUSAL_PREVIOUS_BLOCK_INDEX = int(LFH_SOURCE_BLOCK_INDEX)
HEAD_CAUSAL_TARGET_BLOCK_INDEX = int(LFH_TARGET_BLOCK_INDEX)

# Intervention scope:
#   "target_block" edits only target/example-8 tokens. This is the cleanest
#   test of the proposed write-side mechanism.
#   "previous_block" edits only source/example tokens.
#   "previous_and_target" edits both.
#   "all_example_tokens" edits all example-block tokens.
HEAD_CAUSAL_TOKEN_SCOPE = "target_block"

# Additive alphas for the selected head's o_proj input on edited tokens.
# The effective multiplier is (1 + alpha):
#   alpha =  0.0: no intervention / baseline
#   alpha = -1.0: ablate this head on edited tokens
#   alpha =  1.0: add one extra copy of this head's contribution
HEAD_CAUSAL_HEAD_ALPHAS = [-5.0, -2.0, -1.0, 0.0, 1.0, 2.0,5.0]
HEAD_CAUSAL_HEAD_ALPHAS = [-5.0, -2.0, -1.0, 0.0, 1.0, 2.0,5.0]

# The selected model can be large. Start small, then scale up.
HEAD_CAUSAL_MAX_QUERIES = ATTN_WRITE_TEST_MAX_QUERIES if "ATTN_WRITE_TEST_MAX_QUERIES" in globals() else min(10, len(prompts_all_agnews))
HEAD_CAUSAL_MAX_VARIANTS = ATTN_WRITE_TEST_MAX_VARIANTS if "ATTN_WRITE_TEST_MAX_VARIANTS" in globals() else None

# For high-vs-low source activation summaries.
HEAD_CAUSAL_QUANTILE = 0.25

EPS = 1e-9

print("=" * 120)
print("Attention-head causal intervention config")
print("=" * 120)
print("feature_idx:", HEAD_CAUSAL_FEATURE_IDX)
print("attention layer:", HEAD_CAUSAL_LAYER)
print("attention head:", HEAD_CAUSAL_HEAD)
print("feature score layer:", HEAD_CAUSAL_FEATURE_SCORE_LAYER)
print("token scope:", HEAD_CAUSAL_TOKEN_SCOPE)
print("head alphas:", HEAD_CAUSAL_HEAD_ALPHAS)
print("max queries:", HEAD_CAUSAL_MAX_QUERIES)
print("max variants:", HEAD_CAUSAL_MAX_VARIANTS)
print("=" * 120)

if HEAD_CAUSAL_LAYER > HEAD_CAUSAL_FEATURE_SCORE_LAYER:
    raise ValueError(
        f"HEAD_CAUSAL_LAYER={HEAD_CAUSAL_LAYER} is later than "
        f"HEAD_CAUSAL_FEATURE_SCORE_LAYER={HEAD_CAUSAL_FEATURE_SCORE_LAYER}. "
        "A later attention head cannot causally affect an earlier SAE-layer activation "
        "in the same forward pass. Choose HEAD_CAUSAL_LAYER <= TARGET_LAYER."
    )


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def _safe_pearson_local(x, y):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    finite = np.isfinite(x) & np.isfinite(y)
    x = x[finite]
    y = y[finite]
    if len(x) < 3:
        return np.nan
    if np.std(x) <= EPS or np.std(y) <= EPS:
        return np.nan
    return float(pearsonr(x, y)[0])


def _attn_num_heads_and_head_dim(model, layer_idx: int):
    attn = model.get_submodule(f"model.layers.{layer_idx}.self_attn")
    num_heads = int(getattr(attn, "num_heads", model.config.num_attention_heads))
    head_dim = int(getattr(attn, "head_dim", model.config.hidden_size // num_heads))
    return num_heads, head_dim


def build_head_intervention_token_mask(
    seq_len: int,
    block_spans,
    previous_block_index: int,
    target_block_index: int,
    token_scope: str = "target_block",
):
    """
    Build a [1, seq_len] bool mask specifying which token positions receive
    the head ablation/amplification.
    """
    mask = torch.zeros((1, seq_len), dtype=torch.bool)

    prev_span = block_spans[previous_block_index]
    tgt_span = block_spans[target_block_index]

    def _mark(selection):
        idx = token_selection_to_torch_indices(selection, device=mask.device)
        mask[0, idx] = True

    if token_scope == "target_block":
        _mark(tgt_span)
    elif token_scope == "previous_block":
        _mark(prev_span)
    elif token_scope == "previous_and_target":
        _mark(prev_span)
        _mark(tgt_span)
    elif token_scope == "all_example_tokens":
        for selection in block_spans:
            _mark(selection)
    else:
        raise ValueError(f"Unknown token_scope: {token_scope}")

    return mask


@torch.no_grad()
def run_forward_capture_feature_layer_with_head_alpha(
    model,
    input_ids,
    attention_mask,
    *,
    head_layer_idx: int,
    head_idx: int,
    feature_layer_idx: int,
    token_mask: torch.Tensor,
    head_alpha: float = 0.0,
):
    """
    Run one forward pass while optionally scaling one attention head's o_proj
    input on selected tokens, then capture the feature_layer_idx residual output.

    The edited tensor is the input to self_attn.o_proj, shaped [B, S, hidden].
    It is a concatenation of per-head attention outputs before W_O.

    head_alpha semantics:
        0.0: no intervention / baseline
       -1.0: ablate selected head on selected tokens
        1.0: add one extra copy of the selected head's contribution
        alpha: effective multiplier is (1 + alpha)
    """
    if head_layer_idx > feature_layer_idx:
        raise ValueError(
            f"head_layer_idx={head_layer_idx} > feature_layer_idx={feature_layer_idx}. "
            "This cannot affect the measured feature activation in the same forward pass."
        )

    cache = {}

    feature_block = model.get_submodule(f"model.layers.{feature_layer_idx}")
    o_proj = model.get_submodule(f"model.layers.{head_layer_idx}.self_attn.o_proj")
    num_heads, head_dim = _attn_num_heads_and_head_dim(model, head_layer_idx)

    if not (0 <= int(head_idx) < num_heads):
        raise ValueError(f"head_idx={head_idx} outside [0, {num_heads}).")

    head_start = int(head_idx) * head_dim
    head_end = head_start + head_dim

    def feature_block_hook(module, inputs, output):
        x = output[0] if isinstance(output, tuple) else output
        cache["feature_layer_output"] = x.detach().clone()
        return output

    def o_proj_pre_hook(module, inputs):
        z = inputs[0]

        # alpha 0.0 is exactly no intervention.
        if float(head_alpha) == 0.0:
            return None

        if z.shape[-1] != num_heads * head_dim:
            raise RuntimeError(
                f"o_proj input last dim {z.shape[-1]} != num_heads*head_dim "
                f"{num_heads}*{head_dim}."
            )

        edit_mask = token_mask.to(device=z.device, dtype=torch.bool)
        if edit_mask.ndim != 2 or edit_mask.shape[0] != z.shape[0] or edit_mask.shape[1] != z.shape[1]:
            raise RuntimeError(
                f"token_mask shape {tuple(edit_mask.shape)} incompatible with o_proj input shape {tuple(z.shape)}."
            )

        z_edit = z.clone()
        head_slice = z_edit[..., head_start:head_end]
        effective_multiplier = 1.0 + float(head_alpha)
        scaled_head_slice = head_slice * effective_multiplier
        z_edit[..., head_start:head_end] = torch.where(edit_mask[..., None], scaled_head_slice, head_slice)

        return (z_edit,) + tuple(inputs[1:])

    h_feature = feature_block.register_forward_hook(feature_block_hook)
    h_head = o_proj.register_forward_pre_hook(o_proj_pre_hook)

    try:
        _ = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
        )
    finally:
        h_head.remove()
        h_feature.remove()

    if "feature_layer_output" not in cache:
        raise RuntimeError(f"Failed to capture feature layer {feature_layer_idx} output.")

    return cache["feature_layer_output"]


@torch.no_grad()
def measure_prompt_feature_with_head_alpha(
    prompt: str,
    *,
    feature_idx: int,
    head_layer_idx: int,
    head_idx: int,
    feature_layer_idx: int,
    head_alpha: float,
    previous_block_index: int = -2,
    target_block_index: int = -1,
    token_scope: str = "target_block",
    expected_num_blocks: int = NUM_BLOCKS_TOTAL,
):
    """
    For one prompt, run a selected attention-head intervention and measure
    feature pre-activation on previous and target exemplar blocks.
    """
    enc = tokenize_prompt_for_offsets(prompt)
    offset_mapping = enc["offset_mapping"][0].tolist()
    input_ids = enc["input_ids"].to(MODEL_INPUT_DEVICE)
    attention_mask = enc["attention_mask"].to(MODEL_INPUT_DEVICE)

    block_spans = get_example_block_token_spans(
        prompt=prompt,
        offset_mapping=offset_mapping,
        expected_num_blocks=expected_num_blocks,
    )

    token_mask = build_head_intervention_token_mask(
        seq_len=input_ids.shape[1],
        block_spans=block_spans,
        previous_block_index=previous_block_index,
        target_block_index=target_block_index,
        token_scope=token_scope,
    ).to(MODEL_INPUT_DEVICE)

    layer_output = run_forward_capture_feature_layer_with_head_alpha(
        model=model,
        input_ids=input_ids,
        attention_mask=attention_mask,
        head_layer_idx=head_layer_idx,
        head_idx=head_idx,
        feature_layer_idx=feature_layer_idx,
        token_mask=token_mask,
        head_alpha=head_alpha,
    )

    pre_f = sae_single_feature_pre_activation(
        sae=sae,
        x=layer_output,
        feature_idx=feature_idx,
    )[0].detach().float()

    prev_selection = block_spans[previous_block_index]
    target_selection = block_spans[target_block_index]
    prev_idx = token_selection_to_torch_indices(prev_selection, device=pre_f.device)
    target_idx = token_selection_to_torch_indices(target_selection, device=pre_f.device)

    return {
        "previous_feature_mean": float(pre_f.index_select(0, prev_idx).mean().detach().cpu()),
        "target_feature_mean": float(pre_f.index_select(0, target_idx).mean().detach().cpu()),
        "prev_span": block_spans[previous_block_index],
        "target_span": block_spans[target_block_index],
    }


def _querywise_corr_summary(records_df, target_col: str, prev_col: str = "baseline_previous_feature_mean"):
    """
    Compute pooled and query-wise correlations for prev_col -> target_col.
    """
    pooled = _safe_pearson_local(records_df[prev_col].values, records_df[target_col].values)

    q_corrs = []
    for q, g in records_df.groupby("query_idx"):
        r = _safe_pearson_local(g[prev_col].values, g[target_col].values)
        if np.isfinite(r):
            q_corrs.append(r)

    if len(q_corrs) == 0:
        return pooled, np.nan, np.nan, 0

    return pooled, float(np.mean(q_corrs)), float(np.std(q_corrs)), int(len(q_corrs))


def _high_low_delta_by_query(records_df, value_col: str, prev_col: str = "baseline_previous_feature_mean", quantile: float = 0.25):
    """
    For each query, compute mean(value | prev high) - mean(value | prev low).
    """
    deltas = []
    for q, g in records_df.groupby("query_idx"):
        x = g[prev_col].to_numpy(dtype=np.float64)
        y = g[value_col].to_numpy(dtype=np.float64)
        finite = np.isfinite(x) & np.isfinite(y)
        x = x[finite]
        y = y[finite]
        if len(x) < 4:
            continue
        lo = np.quantile(x, quantile)
        hi = np.quantile(x, 1.0 - quantile)
        low_mask = x <= lo
        high_mask = x >= hi
        if low_mask.sum() == 0 or high_mask.sum() == 0:
            continue
        deltas.append(float(y[high_mask].mean() - y[low_mask].mean()))
    if len(deltas) == 0:
        return np.nan, np.nan, 0
    return float(np.mean(deltas)), float(np.std(deltas)), int(len(deltas))


@torch.no_grad()
def run_attention_head_alpha_causal_test(
    *,
    feature_idx: int,
    layer_idx: int,
    head_idx: int,
    prompts_all_agnews,
    feature_score_layer: int = TARGET_LAYER,
    head_alphas=(-10.0, -5.0, -1.0, 0.0, 1.0, 5.0, 10.0),
    previous_block_index: int = -2,
    target_block_index: int = -1,
    token_scope: str = "target_block",
    max_queries=None,
    max_variants=None,
    quantile: float = 0.25,
    verbose_every: int = 10,
):
    """
    Main causal test.

    For each prompt variant:
      1. Run baseline with head_alpha=0.0.
      2. Run each requested intervention alpha.
      3. Recompute corr(prev_feature_baseline, target_feature_intervened).

    A candidate negative-write head should show:
      - ablation at alpha=-1: negative correlation weakens;
      - amplification at alpha>0: negative correlation strengthens.
    """
    feature_idx = int(feature_idx)
    layer_idx = int(layer_idx)
    head_idx = int(head_idx)
    feature_score_layer = int(feature_score_layer)

    if layer_idx > feature_score_layer:
        raise ValueError(
            f"layer_idx={layer_idx} > feature_score_layer={feature_score_layer}; "
            "the intervention cannot affect the measured feature in the same forward pass."
        )

    # De-duplicate alphas but keep order; ensure baseline alpha=0.0 is available internally.
    requested_alphas = []
    for alpha in head_alphas:
        af = float(alpha)
        if af not in requested_alphas:
            requested_alphas.append(af)
    if 0.0 not in requested_alphas:
        alphas_to_run = [0.0] + requested_alphas
    else:
        alphas_to_run = requested_alphas

    num_queries_total = len(prompts_all_agnews)
    query_indices = list(range(num_queries_total if max_queries is None else min(max_queries, num_queries_total)))

    rows = []

    for query_counter, query_idx in enumerate(query_indices, start=1):
        variants = prompts_all_agnews[query_idx]
        if max_variants is not None:
            variants = variants[:max_variants]

        for variant_idx, prompt in enumerate(variants):
            # Baseline first.
            baseline = measure_prompt_feature_with_head_alpha(
                prompt=prompt,
                feature_idx=feature_idx,
                head_layer_idx=layer_idx,
                head_idx=head_idx,
                feature_layer_idx=feature_score_layer,
                head_alpha=0.0,
                previous_block_index=previous_block_index,
                target_block_index=target_block_index,
                token_scope=token_scope,
                expected_num_blocks=NUM_BLOCKS_TOTAL,
            )

            base_prev = baseline["previous_feature_mean"]
            base_tgt = baseline["target_feature_mean"]

            for alpha in alphas_to_run:
                if float(alpha) == 0.0:
                    previous_mean = base_prev
                    target_mean = base_tgt
                else:
                    intervened = measure_prompt_feature_with_head_alpha(
                        prompt=prompt,
                        feature_idx=feature_idx,
                        head_layer_idx=layer_idx,
                        head_idx=head_idx,
                        feature_layer_idx=feature_score_layer,
                        head_alpha=float(alpha),
                        previous_block_index=previous_block_index,
                        target_block_index=target_block_index,
                        token_scope=token_scope,
                        expected_num_blocks=NUM_BLOCKS_TOTAL,
                    )
                    previous_mean = intervened["previous_feature_mean"]
                    target_mean = intervened["target_feature_mean"]

                rows.append({
                    "query_idx": int(query_idx),
                    "variant_idx": int(variant_idx),
                    "feature_idx": int(feature_idx),
                    "layer": int(layer_idx),
                    "head": int(head_idx),
                    "feature_score_layer": int(feature_score_layer),
                    "token_scope": token_scope,
                    "head_alpha": float(alpha),
                    "effective_head_multiplier": float(1.0 + float(alpha)),
                    "baseline_previous_feature_mean": float(base_prev),
                    "baseline_target_feature_mean": float(base_tgt),
                    "intervened_previous_feature_mean": float(previous_mean),
                    "intervened_target_feature_mean": float(target_mean),
                    "target_delta_intervened_minus_baseline": float(target_mean - base_tgt),
                    "previous_delta_intervened_minus_baseline": float(previous_mean - base_prev),
                })

        if verbose_every is not None and (query_counter % max(verbose_every, 1) == 0 or query_counter == len(query_indices)):
            print(f"processed query {query_counter}/{len(query_indices)}; rows={len(rows)}")

    records_df = pd.DataFrame(rows)

    # Baseline correlation summary from alpha 0.0 rows.
    baseline_rows = records_df[records_df["head_alpha"] == 0.0].copy()
    baseline_pooled_r, baseline_mean_query_r, baseline_std_query_r, baseline_n_query = _querywise_corr_summary(
        baseline_rows,
        target_col="baseline_target_feature_mean",
        prev_col="baseline_previous_feature_mean",
    )

    summary_rows = []
    for alpha in requested_alphas:
        sub = records_df[records_df["head_alpha"] == float(alpha)].copy()
        if len(sub) == 0:
            continue

        pooled_r, mean_query_r, std_query_r, n_query = _querywise_corr_summary(
            sub,
            target_col="intervened_target_feature_mean",
            prev_col="baseline_previous_feature_mean",
        )

        delta_pooled_r, delta_mean_query_r, delta_std_query_r, delta_n_query = _querywise_corr_summary(
            sub,
            target_col="target_delta_intervened_minus_baseline",
            prev_col="baseline_previous_feature_mean",
        )

        high_low_target_delta_mean, high_low_target_delta_std, high_low_n = _high_low_delta_by_query(
            sub,
            value_col="target_delta_intervened_minus_baseline",
            prev_col="baseline_previous_feature_mean",
            quantile=quantile,
        )

        high_low_target_mean, high_low_target_std, high_low_target_n = _high_low_delta_by_query(
            sub,
            value_col="intervened_target_feature_mean",
            prev_col="baseline_previous_feature_mean",
            quantile=quantile,
        )

        summary_rows.append({
            "feature_idx": int(feature_idx),
            "layer": int(layer_idx),
            "head": int(head_idx),
            "feature_score_layer": int(feature_score_layer),
            "token_scope": token_scope,
            "head_alpha": float(alpha),
            "effective_head_multiplier": float(1.0 + float(alpha)),

            "baseline_pooled_corr_prev_to_target": baseline_pooled_r,
            "baseline_mean_query_corr_prev_to_target": baseline_mean_query_r,
            "baseline_std_query_corr_prev_to_target": baseline_std_query_r,

            "intervened_pooled_corr_prev_to_target": pooled_r,
            "intervened_mean_query_corr_prev_to_target": mean_query_r,
            "intervened_std_query_corr_prev_to_target": std_query_r,

            "corr_change_pooled_intervened_minus_baseline": pooled_r - baseline_pooled_r,
            "corr_change_mean_query_intervened_minus_baseline": mean_query_r - baseline_mean_query_r,

            "corr_prev_to_target_delta_pooled": delta_pooled_r,
            "corr_prev_to_target_delta_mean_query": delta_mean_query_r,

            "mean_target_delta_intervened_minus_baseline": float(sub["target_delta_intervened_minus_baseline"].mean()),
            "median_target_delta_intervened_minus_baseline": float(sub["target_delta_intervened_minus_baseline"].median()),
            "fraction_target_delta_positive": float((sub["target_delta_intervened_minus_baseline"] > 0).mean()),

            "high_minus_low_prev_target_delta_mean": high_low_target_delta_mean,
            "high_minus_low_prev_target_delta_std": high_low_target_delta_std,
            "high_minus_low_prev_intervened_target_mean": high_low_target_mean,
            "high_minus_low_prev_intervened_target_std": high_low_target_std,
            "n_queries_for_high_low": int(high_low_n),
        })

    summary_df = pd.DataFrame(summary_rows).sort_values("head_alpha").reset_index(drop=True)

    print("=" * 120)
    print(f"Head causal test: feature={feature_idx}, layer={layer_idx}, head={head_idx}")
    print(f"Baseline pooled corr(prev, target): {baseline_pooled_r:.4f}")
    print(f"Baseline mean query-wise corr(prev, target): {baseline_mean_query_r:.4f}")
    print("Interpretation:")
    print("  alpha=0.0 leaves this head unchanged on target tokens.")
    print("  alpha=-1.0 ablates this head on target tokens.")
    print("  alpha>0 adds extra copies of this head's contribution on target tokens.")
    print("  effective multiplier = 1 + alpha.")
    print("  If the head implements negative write, ablation should make a negative corr less negative,")
    print("  while amplification should make it more negative.")
    print("=" * 120)
    display(summary_df)

    return summary_df, records_df


# ------------------------------------------------------------
# Plotting helpers (Updated with Custom Font Properties)
# ------------------------------------------------------------

# aaaaa
def plot_head_causal_dose_response(summary_df):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=summary_df["head_alpha"],
        y=summary_df["intervened_mean_query_corr_prev_to_target"],
        mode="lines+markers",
        name="intervened corr(prev, target), mean query-wise",
        hovertemplate="alpha=%{x}<br>corr=%{y:.4f}<extra></extra>",
    ))

    baseline = float(summary_df["baseline_mean_query_corr_prev_to_target"].iloc[0])
    fig.add_hline(y=baseline, line_dash="dash", annotation_text="baseline corr")
    fig.add_vline(x=0.0, line_dash="dash", annotation_text="no intervention")

    fig.update_layout(
        font=dict(family="Times New Roman"),
        title=(
            "Attention-Head Intervention Response on Local Feature Homeostasis<br>"
            f"Model=Gemma2-9B-IT, Feature={int(summary_df['feature_idx'].iloc[0])}, "
            f"Layer={int(summary_df['layer'].iloc[0])}, "
            f"Head={int(summary_df['head'].iloc[0])}"
        ),
        title_font_size=26,
        xaxis_title="Intervention Strength (Alpha) on Target-Block Tokens",
        xaxis_title_font_size=22,
        yaxis_title="Mean corr(source feature, target feature)",
        yaxis_title_font_size=22,
        width=900,
        height=600,
        hovermode="x unified",
    )
    
    # --- ADDED: 4x Resolution Config for Interactive 'Download Plot' Button ---
    config = {
        'toImageButtonOptions': {
            'format': 'png',         # one of png, svg, jpeg, webp
            'filename': 'head_causal_dose_response',
            'height': 600,
            'width': 900,
            'scale': 4               # Multiply resolution by 4
        }
    }
    
    fig.show(config=config)
    
    # --- ADDED: Optional direct save to disk at 4x resolution ---
    # Uncomment the line below to save programmatically (requires pip install kaleido)
    # fig.write_image("head_causal_dose_response_highres.png", scale=4)

    return fig


def plot_head_causal_delta_response(summary_df):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=summary_df["head_alpha"],
        y=summary_df["corr_prev_to_target_delta_mean_query"],
        mode="lines+markers",
        name="corr(prev, target_delta)",
        hovertemplate="alpha=%{x}<br>corr=%{y:.4f}<extra></extra>",
    ))

    fig.add_trace(go.Scatter(
        x=summary_df["head_alpha"],
        y=summary_df["high_minus_low_prev_target_delta_mean"],
        mode="lines+markers",
        name="high-prev minus low-prev target delta",
        hovertemplate="alpha=%{x}<br>high-low delta=%{y:.4f}<extra></extra>",
    ))

    fig.add_hline(y=0.0, line_dash="dash")
    fig.add_vline(x=0.0, line_dash="dash", annotation_text="no intervention")

    fig.update_layout(
        font=dict(family="Times New Roman"),
        title="Does the intervention affect high-previous-activation variants differently?",
        title_font_size=26,
        xaxis_title="Head alpha on target-block tokens (effective multiplier = 1 + alpha)",
        xaxis_title_font_size=22,
        yaxis_title="Effect size",
        yaxis_title_font_size=22,
        width=900,
        height=600,
        hovermode="x unified",
    )
    fig.show()
    return fig


def plot_head_intervention_scatter(records_df, alpha: float):
    sub = records_df[records_df["head_alpha"] == float(alpha)].copy()
    if len(sub) == 0:
        raise ValueError(f"No rows for alpha={alpha}")

    x = sub["baseline_previous_feature_mean"].to_numpy(dtype=np.float64)
    y_base = sub["baseline_target_feature_mean"].to_numpy(dtype=np.float64)
    y_int = sub["intervened_target_feature_mean"].to_numpy(dtype=np.float64)

    r_base = _safe_pearson_local(x, y_base)
    r_int = _safe_pearson_local(x, y_int)

    df_plot = pd.DataFrame({
        "previous_feature_mean": np.concatenate([x, x]),
        "target_feature_mean": np.concatenate([y_base, y_int]),
        "condition": ["baseline"] * len(x) + [f"head_alpha={alpha}"] * len(x),
        "query_idx": np.concatenate([sub["query_idx"].values, sub["query_idx"].values]).astype(str),
        "variant_idx": np.concatenate([sub["variant_idx"].values, sub["variant_idx"].values]),
    })

    fig = px.scatter(
        df_plot,
        x="previous_feature_mean",
        y="target_feature_mean",
        color="condition",
        symbol="condition",
        hover_data=["query_idx", "variant_idx"],
        title=(
            f"Source-to-target feature relation before/after head intervention<br>"
            f"alpha={alpha}, baseline r={r_base:.3f}, intervened r={r_int:.3f}"
        ),
    )

    fig.update_layout(
        font=dict(family="Times New Roman"),
        title_font_size=26,
        width=900,
        height=600,
        xaxis_title="Mean SAE pre-activation on source exemplar",
        xaxis_title_font_size=22,
        yaxis_title="Mean SAE pre-activation on target exemplar",
        yaxis_title_font_size=22,
    )
    fig.show()
    return fig


# ------------------------------------------------------------
# Run the head intervention test
# ------------------------------------------------------------

head_causal_summary_df, head_causal_records_df = run_attention_head_alpha_causal_test(
    feature_idx=HEAD_CAUSAL_FEATURE_IDX,
    layer_idx=HEAD_CAUSAL_LAYER,
    head_idx=HEAD_CAUSAL_HEAD,
    prompts_all_agnews=prompts_all_agnews,
    feature_score_layer=HEAD_CAUSAL_FEATURE_SCORE_LAYER,
    head_alphas=HEAD_CAUSAL_HEAD_ALPHAS,
    previous_block_index=HEAD_CAUSAL_PREVIOUS_BLOCK_INDEX,
    target_block_index=HEAD_CAUSAL_TARGET_BLOCK_INDEX,
    token_scope=HEAD_CAUSAL_TOKEN_SCOPE,
    max_queries=HEAD_CAUSAL_MAX_QUERIES,
    max_variants=HEAD_CAUSAL_MAX_VARIANTS,
    quantile=HEAD_CAUSAL_QUANTILE,
    verbose_every=5,
)

_ = plot_head_causal_dose_response(head_causal_summary_df)
_ = plot_head_causal_delta_response(head_causal_summary_df)

# Plot ablation and amplification views if present.
if -1.0 in set(head_causal_records_df["head_alpha"].astype(float)):
    _ = plot_head_intervention_scatter(head_causal_records_df, alpha=-1.0)
if 1.0 in set(head_causal_records_df["head_alpha"].astype(float)):
    _ = plot_head_intervention_scatter(head_causal_records_df, alpha=1.0)

# BLOCK_TOKEN_OFFSETS_FROM_END = [-1,-2,-3,-4,-5]

In [ ]:
# ============================================================
# 14. Causal validation: ablate/amplify one attention head and remeasure
#      source-exemplar -> target-exemplar feature correlation
# ============================================================

import numpy as np
import pandas as pd
import torch
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import pearsonr

# ------------------------------------------------------------
# User-editable config
# ------------------------------------------------------------

# Feature to test. By default, use the same feature as the observational
# negative-write test above.
HEAD_CAUSAL_FEATURE_IDX = int(ATTN_WRITE_TEST_FEATURE_IDX) if "ATTN_WRITE_TEST_FEATURE_IDX" in globals() else int(target_feature)

# Attention head to test. By default, use the top head from the observational
# negative-write ranking, if available.
HEAD_CAUSAL_LAYER = int(ATTN_WRITE_TEST_LAYER) if "ATTN_WRITE_TEST_LAYER" in globals() else int(TARGET_LAYER)

# aaaaa
HEAD_CAUSAL_HEAD = int(TOP_NEGATIVE_WRITE_HEAD) if "TOP_NEGATIVE_WRITE_HEAD" in globals() else 0
# HEAD_CAUSAL_HEAD = 13

# The SAE feature is measured at TARGET_LAYER, because the SAE was trained on
# resid_post_layer_{TARGET_LAYER}. A causal head layer must be <= TARGET_LAYER.
HEAD_CAUSAL_FEATURE_SCORE_LAYER = int(TARGET_LAYER)

# Use the configured source/target blocks. Default: source Example 7 -> target Example 8.
HEAD_CAUSAL_PREVIOUS_BLOCK_INDEX = int(LFH_SOURCE_BLOCK_INDEX)
HEAD_CAUSAL_TARGET_BLOCK_INDEX = int(LFH_TARGET_BLOCK_INDEX)

# Intervention scope:
#   "target_block" edits only target/example-8 tokens. This is the cleanest
#   test of the proposed write-side mechanism.
#   "previous_block" edits only source/example tokens.
#   "previous_and_target" edits both.
#   "all_example_tokens" edits all example-block tokens.
HEAD_CAUSAL_TOKEN_SCOPE = "target_block"

# Additive alphas for the selected head's o_proj input on edited tokens.
# The effective multiplier is (1 + alpha):
#   alpha =  0.0: no intervention / baseline
#   alpha = -1.0: ablate this head on edited tokens
#   alpha =  1.0: add one extra copy of this head's contribution
HEAD_CAUSAL_HEAD_ALPHAS = [-5.0, -2.0, -1.0, 0.0, 1.0, 2.0, 5.0]

# The selected model can be large. Start small, then scale up.
HEAD_CAUSAL_MAX_QUERIES = ATTN_WRITE_TEST_MAX_QUERIES if "ATTN_WRITE_TEST_MAX_QUERIES" in globals() else min(10, len(prompts_all_agnews))
HEAD_CAUSAL_MAX_VARIANTS = ATTN_WRITE_TEST_MAX_VARIANTS if "ATTN_WRITE_TEST_MAX_VARIANTS" in globals() else None

# For high-vs-low source activation summaries.
HEAD_CAUSAL_QUANTILE = 0.25

EPS = 1e-9

print("=" * 120)
print("Attention-head causal intervention config")
print("=" * 120)
print("feature_idx:", HEAD_CAUSAL_FEATURE_IDX)
print("attention layer:", HEAD_CAUSAL_LAYER)
print("attention head:", HEAD_CAUSAL_HEAD)
print("feature score layer:", HEAD_CAUSAL_FEATURE_SCORE_LAYER)
print("token scope:", HEAD_CAUSAL_TOKEN_SCOPE)
print("head alphas:", HEAD_CAUSAL_HEAD_ALPHAS)
print("max queries:", HEAD_CAUSAL_MAX_QUERIES)
print("max variants:", HEAD_CAUSAL_MAX_VARIANTS)
print("=" * 120)

if HEAD_CAUSAL_LAYER > HEAD_CAUSAL_FEATURE_SCORE_LAYER:
    raise ValueError(
        f"HEAD_CAUSAL_LAYER={HEAD_CAUSAL_LAYER} is later than "
        f"HEAD_CAUSAL_FEATURE_SCORE_LAYER={HEAD_CAUSAL_FEATURE_SCORE_LAYER}. "
        "A later attention head cannot causally affect an earlier SAE-layer activation "
        "in the same forward pass. Choose HEAD_CAUSAL_LAYER <= TARGET_LAYER."
    )


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def _safe_pearson_local(x, y):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    finite = np.isfinite(x) & np.isfinite(y)
    x = x[finite]
    y = y[finite]
    if len(x) < 3:
        return np.nan
    if np.std(x) <= EPS or np.std(y) <= EPS:
        return np.nan
    return float(pearsonr(x, y)[0])


def _attn_num_heads_and_head_dim(model, layer_idx: int):
    attn = model.get_submodule(f"model.layers.{layer_idx}.self_attn")
    num_heads = int(getattr(attn, "num_heads", model.config.num_attention_heads))
    head_dim = int(getattr(attn, "head_dim", model.config.hidden_size // num_heads))
    return num_heads, head_dim


def build_head_intervention_token_mask(
    seq_len: int,
    block_spans,
    previous_block_index: int,
    target_block_index: int,
    token_scope: str = "target_block",
):
    """
    Build a [1, seq_len] bool mask specifying which token positions receive
    the head ablation/amplification.
    """
    mask = torch.zeros((1, seq_len), dtype=torch.bool)

    prev_span = block_spans[previous_block_index]
    tgt_span = block_spans[target_block_index]

    def _mark(selection):
        idx = token_selection_to_torch_indices(selection, device=mask.device)
        mask[0, idx] = True

    if token_scope == "target_block":
        _mark(tgt_span)
    elif token_scope == "previous_block":
        _mark(prev_span)
    elif token_scope == "previous_and_target":
        _mark(prev_span)
        _mark(tgt_span)
    elif token_scope == "all_example_tokens":
        for selection in block_spans:
            _mark(selection)
    else:
        raise ValueError(f"Unknown token_scope: {token_scope}")

    return mask


@torch.no_grad()
def run_forward_capture_feature_layer_with_head_alpha(
    model,
    input_ids,
    attention_mask,
    *,
    head_layer_idx: int,
    head_idx: int,
    feature_layer_idx: int,
    token_mask: torch.Tensor,
    head_alpha: float = 0.0,
):
    """
    Run one forward pass while optionally scaling one attention head's o_proj
    input on selected tokens, then capture the feature_layer_idx residual output.

    The edited tensor is the input to self_attn.o_proj, shaped [B, S, hidden].
    It is a concatenation of per-head attention outputs before W_O.

    head_alpha semantics:
        0.0: no intervention / baseline
       -1.0: ablate selected head on selected tokens
        1.0: add one extra copy of the selected head's contribution
        alpha: effective multiplier is (1 + alpha)
    """
    if head_layer_idx > feature_layer_idx:
        raise ValueError(
            f"head_layer_idx={head_layer_idx} > feature_layer_idx={feature_layer_idx}. "
            "This cannot affect the measured feature activation in the same forward pass."
        )

    cache = {}

    feature_block = model.get_submodule(f"model.layers.{feature_layer_idx}")
    o_proj = model.get_submodule(f"model.layers.{head_layer_idx}.self_attn.o_proj")
    num_heads, head_dim = _attn_num_heads_and_head_dim(model, head_layer_idx)

    if not (0 <= int(head_idx) < num_heads):
        raise ValueError(f"head_idx={head_idx} outside [0, {num_heads}).")

    head_start = int(head_idx) * head_dim
    head_end = head_start + head_dim

    def feature_block_hook(module, inputs, output):
        x = output[0] if isinstance(output, tuple) else output
        cache["feature_layer_output"] = x.detach().clone()
        return output

    def o_proj_pre_hook(module, inputs):
        z = inputs[0]

        # alpha 0.0 is exactly no intervention.
        if float(head_alpha) == 0.0:
            return None

        if z.shape[-1] != num_heads * head_dim:
            raise RuntimeError(
                f"o_proj input last dim {z.shape[-1]} != num_heads*head_dim "
                f"{num_heads}*{head_dim}."
            )

        edit_mask = token_mask.to(device=z.device, dtype=torch.bool)
        if edit_mask.ndim != 2 or edit_mask.shape[0] != z.shape[0] or edit_mask.shape[1] != z.shape[1]:
            raise RuntimeError(
                f"token_mask shape {tuple(edit_mask.shape)} incompatible with o_proj input shape {tuple(z.shape)}."
            )

        z_edit = z.clone()
        head_slice = z_edit[..., head_start:head_end]
        effective_multiplier = 1.0 + float(head_alpha)
        scaled_head_slice = head_slice * effective_multiplier
        z_edit[..., head_start:head_end] = torch.where(edit_mask[..., None], scaled_head_slice, head_slice)

        return (z_edit,) + tuple(inputs[1:])

    h_feature = feature_block.register_forward_hook(feature_block_hook)
    h_head = o_proj.register_forward_pre_hook(o_proj_pre_hook)

    try:
        _ = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
        )
    finally:
        h_head.remove()
        h_feature.remove()

    if "feature_layer_output" not in cache:
        raise RuntimeError(f"Failed to capture feature layer {feature_layer_idx} output.")

    return cache["feature_layer_output"]


@torch.no_grad()
def measure_prompt_feature_with_head_alpha(
    prompt: str,
    *,
    feature_idx: int,
    head_layer_idx: int,
    head_idx: int,
    feature_layer_idx: int,
    head_alpha: float,
    previous_block_index: int = -2,
    target_block_index: int = -1,
    token_scope: str = "target_block",
    expected_num_blocks: int = NUM_BLOCKS_TOTAL,
):
    """
    For one prompt, run a selected attention-head intervention and measure
    feature pre-activation on previous and target exemplar blocks.
    """
    enc = tokenize_prompt_for_offsets(prompt)
    offset_mapping = enc["offset_mapping"][0].tolist()
    input_ids = enc["input_ids"].to(MODEL_INPUT_DEVICE)
    attention_mask = enc["attention_mask"].to(MODEL_INPUT_DEVICE)

    block_spans = get_example_block_token_spans(
        prompt=prompt,
        offset_mapping=offset_mapping,
        expected_num_blocks=expected_num_blocks,
    )

    token_mask = build_head_intervention_token_mask(
        seq_len=input_ids.shape[1],
        block_spans=block_spans,
        previous_block_index=previous_block_index,
        target_block_index=target_block_index,
        token_scope=token_scope,
    ).to(MODEL_INPUT_DEVICE)

    layer_output = run_forward_capture_feature_layer_with_head_alpha(
        model=model,
        input_ids=input_ids,
        attention_mask=attention_mask,
        head_layer_idx=head_layer_idx,
        head_idx=head_idx,
        feature_layer_idx=feature_layer_idx,
        token_mask=token_mask,
        head_alpha=head_alpha,
    )

    pre_f = sae_single_feature_pre_activation(
        sae=sae,
        x=layer_output,
        feature_idx=feature_idx,
    )[0].detach().float()

    prev_selection = block_spans[previous_block_index]
    target_selection = block_spans[target_block_index]
    prev_idx = token_selection_to_torch_indices(prev_selection, device=pre_f.device)
    target_idx = token_selection_to_torch_indices(target_selection, device=pre_f.device)

    return {
        "previous_feature_mean": float(pre_f.index_select(0, prev_idx).mean().detach().cpu()),
        "target_feature_mean": float(pre_f.index_select(0, target_idx).mean().detach().cpu()),
        "prev_span": block_spans[previous_block_index],
        "target_span": block_spans[target_block_index],
    }


def _querywise_corr_summary(records_df, target_col: str, prev_col: str = "baseline_previous_feature_mean"):
    """
    Compute pooled and query-wise correlations for prev_col -> target_col.
    """
    pooled = _safe_pearson_local(records_df[prev_col].values, records_df[target_col].values)

    q_corrs = []
    for q, g in records_df.groupby("query_idx"):
        r = _safe_pearson_local(g[prev_col].values, g[target_col].values)
        if np.isfinite(r):
            q_corrs.append(r)

    if len(q_corrs) == 0:
        return pooled, np.nan, np.nan, 0

    return pooled, float(np.mean(q_corrs)), float(np.std(q_corrs)), int(len(q_corrs))


def _high_low_delta_by_query(records_df, value_col: str, prev_col: str = "baseline_previous_feature_mean", quantile: float = 0.25):
    """
    For each query, compute mean(value | prev high) - mean(value | prev low).
    """
    deltas = []
    for q, g in records_df.groupby("query_idx"):
        x = g[prev_col].to_numpy(dtype=np.float64)
        y = g[value_col].to_numpy(dtype=np.float64)
        finite = np.isfinite(x) & np.isfinite(y)
        x = x[finite]
        y = y[finite]
        if len(x) < 4:
            continue
        lo = np.quantile(x, quantile)
        hi = np.quantile(x, 1.0 - quantile)
        low_mask = x <= lo
        high_mask = x >= hi
        if low_mask.sum() == 0 or high_mask.sum() == 0:
            continue
        deltas.append(float(y[high_mask].mean() - y[low_mask].mean()))
    if len(deltas) == 0:
        return np.nan, np.nan, 0
    return float(np.mean(deltas)), float(np.std(deltas)), int(len(deltas))


@torch.no_grad()
def run_attention_head_alpha_causal_test(
    *,
    feature_idx: int,
    layer_idx: int,
    head_idx: int,
    prompts_all_agnews,
    feature_score_layer: int = TARGET_LAYER,
    head_alphas=(-10.0, -5.0, -1.0, 0.0, 1.0, 5.0, 10.0),
    previous_block_index: int = -2,
    target_block_index: int = -1,
    token_scope: str = "target_block",
    max_queries=None,
    max_variants=None,
    quantile: float = 0.25,
    verbose_every: int = 10,
):
    """
    Main causal test.

    For each prompt variant:
      1. Run baseline with head_alpha=0.0.
      2. Run each requested intervention alpha.
      3. Recompute corr(prev_feature_baseline, target_feature_intervened).

    A candidate negative-write head should show:
      - ablation at alpha=-1: negative correlation weakens;
      - amplification at alpha>0: negative correlation strengthens.
    """
    feature_idx = int(feature_idx)
    layer_idx = int(layer_idx)
    head_idx = int(head_idx)
    feature_score_layer = int(feature_score_layer)

    if layer_idx > feature_score_layer:
        raise ValueError(
            f"layer_idx={layer_idx} > feature_score_layer={feature_score_layer}; "
            "the intervention cannot affect the measured feature in the same forward pass."
        )

    # De-duplicate alphas but keep order; ensure baseline alpha=0.0 is available internally.
    requested_alphas = []
    for alpha in head_alphas:
        af = float(alpha)
        if af not in requested_alphas:
            requested_alphas.append(af)
    if 0.0 not in requested_alphas:
        alphas_to_run = [0.0] + requested_alphas
    else:
        alphas_to_run = requested_alphas

    num_queries_total = len(prompts_all_agnews)
    query_indices = list(range(num_queries_total if max_queries is None else min(max_queries, num_queries_total)))

    rows = []

    for query_counter, query_idx in enumerate(query_indices, start=1):
        variants = prompts_all_agnews[query_idx]
        if max_variants is not None:
            variants = variants[:max_variants]

        for variant_idx, prompt in enumerate(variants):
            # Baseline first.
            baseline = measure_prompt_feature_with_head_alpha(
                prompt=prompt,
                feature_idx=feature_idx,
                head_layer_idx=layer_idx,
                head_idx=head_idx,
                feature_layer_idx=feature_score_layer,
                head_alpha=0.0,
                previous_block_index=previous_block_index,
                target_block_index=target_block_index,
                token_scope=token_scope,
                expected_num_blocks=NUM_BLOCKS_TOTAL,
            )

            base_prev = baseline["previous_feature_mean"]
            base_tgt = baseline["target_feature_mean"]

            for alpha in alphas_to_run:
                if float(alpha) == 0.0:
                    previous_mean = base_prev
                    target_mean = base_tgt
                else:
                    intervened = measure_prompt_feature_with_head_alpha(
                        prompt=prompt,
                        feature_idx=feature_idx,
                        head_layer_idx=layer_idx,
                        head_idx=head_idx,
                        feature_layer_idx=feature_score_layer,
                        head_alpha=float(alpha),
                        previous_block_index=previous_block_index,
                        target_block_index=target_block_index,
                        token_scope=token_scope,
                        expected_num_blocks=NUM_BLOCKS_TOTAL,
                    )
                    previous_mean = intervened["previous_feature_mean"]
                    target_mean = intervened["target_feature_mean"]

                rows.append({
                    "query_idx": int(query_idx),
                    "variant_idx": int(variant_idx),
                    "feature_idx": int(feature_idx),
                    "layer": int(layer_idx),
                    "head": int(head_idx),
                    "feature_score_layer": int(feature_score_layer),
                    "token_scope": token_scope,
                    "head_alpha": float(alpha),
                    "effective_head_multiplier": float(1.0 + float(alpha)),
                    "baseline_previous_feature_mean": float(base_prev),
                    "baseline_target_feature_mean": float(base_tgt),
                    "intervened_previous_feature_mean": float(previous_mean),
                    "intervened_target_feature_mean": float(target_mean),
                    "target_delta_intervened_minus_baseline": float(target_mean - base_tgt),
                    "previous_delta_intervened_minus_baseline": float(previous_mean - base_prev),
                })

        if verbose_every is not None and (query_counter % max(verbose_every, 1) == 0 or query_counter == len(query_indices)):
            print(f"processed query {query_counter}/{len(query_indices)}; rows={len(rows)}")

    records_df = pd.DataFrame(rows)

    # Baseline correlation summary from alpha 0.0 rows.
    baseline_rows = records_df[records_df["head_alpha"] == 0.0].copy()
    baseline_pooled_r, baseline_mean_query_r, baseline_std_query_r, baseline_n_query = _querywise_corr_summary(
        baseline_rows,
        target_col="baseline_target_feature_mean",
        prev_col="baseline_previous_feature_mean",
    )

    summary_rows = []
    for alpha in requested_alphas:
        sub = records_df[records_df["head_alpha"] == float(alpha)].copy()
        if len(sub) == 0:
            continue

        pooled_r, mean_query_r, std_query_r, n_query = _querywise_corr_summary(
            sub,
            target_col="intervened_target_feature_mean",
            prev_col="baseline_previous_feature_mean",
        )

        delta_pooled_r, delta_mean_query_r, delta_std_query_r, delta_n_query = _querywise_corr_summary(
            sub,
            target_col="target_delta_intervened_minus_baseline",
            prev_col="baseline_previous_feature_mean",
        )

        high_low_target_delta_mean, high_low_target_delta_std, high_low_n = _high_low_delta_by_query(
            sub,
            value_col="target_delta_intervened_minus_baseline",
            prev_col="baseline_previous_feature_mean",
            quantile=quantile,
        )

        high_low_target_mean, high_low_target_std, high_low_target_n = _high_low_delta_by_query(
            sub,
            value_col="intervened_target_feature_mean",
            prev_col="baseline_previous_feature_mean",
            quantile=quantile,
        )

        summary_rows.append({
            "feature_idx": int(feature_idx),
            "layer": int(layer_idx),
            "head": int(head_idx),
            "feature_score_layer": int(feature_score_layer),
            "token_scope": token_scope,
            "head_alpha": float(alpha),
            "effective_head_multiplier": float(1.0 + float(alpha)),

            "baseline_pooled_corr_prev_to_target": baseline_pooled_r,
            "baseline_mean_query_corr_prev_to_target": baseline_mean_query_r,
            "baseline_std_query_corr_prev_to_target": baseline_std_query_r,

            "intervened_pooled_corr_prev_to_target": pooled_r,
            "intervened_mean_query_corr_prev_to_target": mean_query_r,
            "intervened_std_query_corr_prev_to_target": std_query_r,

            "corr_change_pooled_intervened_minus_baseline": pooled_r - baseline_pooled_r,
            "corr_change_mean_query_intervened_minus_baseline": mean_query_r - baseline_mean_query_r,

            "corr_prev_to_target_delta_pooled": delta_pooled_r,
            "corr_prev_to_target_delta_mean_query": delta_mean_query_r,

            "mean_target_delta_intervened_minus_baseline": float(sub["target_delta_intervened_minus_baseline"].mean()),
            "median_target_delta_intervened_minus_baseline": float(sub["target_delta_intervened_minus_baseline"].median()),
            "fraction_target_delta_positive": float((sub["target_delta_intervened_minus_baseline"] > 0).mean()),

            "high_minus_low_prev_target_delta_mean": high_low_target_delta_mean,
            "high_minus_low_prev_target_delta_std": high_low_target_delta_std,
            "high_minus_low_prev_intervened_target_mean": high_low_target_mean,
            "high_minus_low_prev_intervened_target_std": high_low_target_std,
            "n_queries_for_high_low": int(high_low_n),
        })

    summary_df = pd.DataFrame(summary_rows).sort_values("head_alpha").reset_index(drop=True)

    print("=" * 120)
    print(f"Head causal test: feature={feature_idx}, layer={layer_idx}, head={head_idx}")
    print(f"Baseline pooled corr(prev, target): {baseline_pooled_r:.4f}")
    print(f"Baseline mean query-wise corr(prev, target): {baseline_mean_query_r:.4f}")
    print("Interpretation:")
    print("  alpha=0.0 leaves this head unchanged on target tokens.")
    print("  alpha=-1.0 ablates this head on target tokens.")
    print("  alpha>0 adds extra copies of this head's contribution on target tokens.")
    print("  effective multiplier = 1 + alpha.")
    print("  If the head implements negative write, ablation should make a negative corr less negative,")
    print("  while amplification should make it more negative.")
    print("=" * 120)
    display(summary_df)

    return summary_df, records_df


# ------------------------------------------------------------
# Plotting helpers (Updated with Custom Font Properties)
# ------------------------------------------------------------

# aaaaa
def plot_head_causal_dose_response(summary_df):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=summary_df["head_alpha"],
        y=summary_df["intervened_mean_query_corr_prev_to_target"],
        mode="lines+markers",
        name="intervened corr(prev, target), mean query-wise",
        hovertemplate="alpha=%{x}<br>corr=%{y:.4f}<extra></extra>",
    ))

    baseline = float(summary_df["baseline_mean_query_corr_prev_to_target"].iloc[0])
    fig.add_hline(y=baseline, line_dash="dash", annotation_text="baseline corr")
    fig.add_vline(x=0.0, line_dash="dash", annotation_text="no intervention")

    fig.update_layout(
        font=dict(family="Times New Roman"),
        title=(
            "Attention-Head Intervention Response on Local Feature Homeostasis<br>"
            f"Model=Gemma2-9B-IT, Feature={int(summary_df['feature_idx'].iloc[0])}, "
            f"Layer={int(summary_df['layer'].iloc[0])}, "
            f"Head={int(summary_df['head'].iloc[0])}"
        ),
        title_font_size=26,
        xaxis_title="Intervention Strength (Alpha) on Target-Block Tokens",
        xaxis_title_font_size=22,
        yaxis_title="Mean corr(source feature, target feature)",
        yaxis_title_font_size=22,
        width=900,
        height=600,
        hovermode="x unified",
    )
    
    # --- ADDED: 4x Resolution Config for Interactive 'Download Plot' Button ---
    config = {
        'toImageButtonOptions': {
            'format': 'png',         # one of png, svg, jpeg, webp
            'filename': 'head_causal_dose_response',
            'height': 600,
            'width': 900,
            'scale': 4               # Multiply resolution by 4
        }
    }
    
    fig.show(config=config)
    
    # --- ADDED: Optional direct save to disk at 4x resolution ---
    # Uncomment the line below to save programmatically (requires pip install kaleido)
    # fig.write_image("head_causal_dose_response_highres.png", scale=4)

    return fig


def plot_head_causal_delta_response(summary_df):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=summary_df["head_alpha"],
        y=summary_df["corr_prev_to_target_delta_mean_query"],
        mode="lines+markers",
        name="corr(prev, target_delta)",
        hovertemplate="alpha=%{x}<br>corr=%{y:.4f}<extra></extra>",
    ))

    fig.add_trace(go.Scatter(
        x=summary_df["head_alpha"],
        y=summary_df["high_minus_low_prev_target_delta_mean"],
        mode="lines+markers",
        name="high-prev minus low-prev target delta",
        hovertemplate="alpha=%{x}<br>high-low delta=%{y:.4f}<extra></extra>",
    ))

    fig.add_hline(y=0.0, line_dash="dash")
    fig.add_vline(x=0.0, line_dash="dash", annotation_text="no intervention")

    fig.update_layout(
        font=dict(family="Times New Roman"),
        title="Does the intervention affect high-previous-activation variants differently?",
        title_font_size=26,
        xaxis_title="Head alpha on target-block tokens (effective multiplier = 1 + alpha)",
        xaxis_title_font_size=22,
        yaxis_title="Effect size",
        yaxis_title_font_size=22,
        width=900,
        height=600,
        hovermode="x unified",
    )
    fig.show()
    return fig


def plot_head_intervention_scatter(records_df, alpha: float):
    sub = records_df[records_df["head_alpha"] == float(alpha)].copy()
    if len(sub) == 0:
        raise ValueError(f"No rows for alpha={alpha}")

    x = sub["baseline_previous_feature_mean"].to_numpy(dtype=np.float64)
    y_base = sub["baseline_target_feature_mean"].to_numpy(dtype=np.float64)
    y_int = sub["intervened_target_feature_mean"].to_numpy(dtype=np.float64)

    r_base = _safe_pearson_local(x, y_base)
    r_int = _safe_pearson_local(x, y_int)

    df_plot = pd.DataFrame({
        "previous_feature_mean": np.concatenate([x, x]),
        "target_feature_mean": np.concatenate([y_base, y_int]),
        "condition": ["baseline"] * len(x) + [f"head_alpha={alpha}"] * len(x),
        "query_idx": np.concatenate([sub["query_idx"].values, sub["query_idx"].values]).astype(str),
        "variant_idx": np.concatenate([sub["variant_idx"].values, sub["variant_idx"].values]),
    })

    fig = px.scatter(
        df_plot,
        x="previous_feature_mean",
        y="target_feature_mean",
        color="condition",
        symbol="condition",
        hover_data=["query_idx", "variant_idx"],
        title=(
            f"Source-to-target feature relation before/after head intervention<br>"
            f"alpha={alpha}, baseline r={r_base:.3f}, intervened r={r_int:.3f}"
        ),
    )

    fig.update_layout(
        font=dict(family="Times New Roman"),
        title_font_size=26,
        width=900,
        height=600,
        xaxis_title="Mean SAE pre-activation on source exemplar",
        xaxis_title_font_size=22,
        yaxis_title="Mean SAE pre-activation on target exemplar",
        yaxis_title_font_size=22,
    )
    fig.show()
    return fig


# ------------------------------------------------------------
# Run the head intervention test
# ------------------------------------------------------------

head_causal_summary_df, head_causal_records_df = run_attention_head_alpha_causal_test(
    feature_idx=HEAD_CAUSAL_FEATURE_IDX,
    layer_idx=HEAD_CAUSAL_LAYER,
    head_idx=HEAD_CAUSAL_HEAD,
    prompts_all_agnews=prompts_all_agnews,
    feature_score_layer=HEAD_CAUSAL_FEATURE_SCORE_LAYER,
    head_alphas=HEAD_CAUSAL_HEAD_ALPHAS,
    previous_block_index=HEAD_CAUSAL_PREVIOUS_BLOCK_INDEX,
    target_block_index=HEAD_CAUSAL_TARGET_BLOCK_INDEX,
    token_scope=HEAD_CAUSAL_TOKEN_SCOPE,
    max_queries=HEAD_CAUSAL_MAX_QUERIES,
    max_variants=HEAD_CAUSAL_MAX_VARIANTS,
    quantile=HEAD_CAUSAL_QUANTILE,
    verbose_every=5,
)

_ = plot_head_causal_dose_response(head_causal_summary_df)
_ = plot_head_causal_delta_response(head_causal_summary_df)

# Plot ablation and amplification views if present.
if -1.0 in set(head_causal_records_df["head_alpha"].astype(float)):
    _ = plot_head_intervention_scatter(head_causal_records_df, alpha=-1.0)
if 1.0 in set(head_causal_records_df["head_alpha"].astype(float)):
    _ = plot_head_intervention_scatter(head_causal_records_df, alpha=1.0)

# BLOCK_TOKEN_OFFSETS_FROM_END = [-1,-2,-3,-4,-5,-6,-7]

In [ ]:
# ============================================================
# 14. Causal validation: ablate/amplify one attention head and remeasure
#      source-exemplar -> target-exemplar feature correlation
# ============================================================

import numpy as np
import pandas as pd
import torch
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import pearsonr

# ------------------------------------------------------------
# User-editable config
# ------------------------------------------------------------

# Feature to test. By default, use the same feature as the observational
# negative-write test above.
HEAD_CAUSAL_FEATURE_IDX = int(ATTN_WRITE_TEST_FEATURE_IDX) if "ATTN_WRITE_TEST_FEATURE_IDX" in globals() else int(target_feature)

# Attention head to test. By default, use the top head from the observational
# negative-write ranking, if available.
HEAD_CAUSAL_LAYER = int(ATTN_WRITE_TEST_LAYER) if "ATTN_WRITE_TEST_LAYER" in globals() else int(TARGET_LAYER)

# aaaaa
HEAD_CAUSAL_HEAD = int(TOP_NEGATIVE_WRITE_HEAD) if "TOP_NEGATIVE_WRITE_HEAD" in globals() else 0
HEAD_CAUSAL_HEAD = 8

# The SAE feature is measured at TARGET_LAYER, because the SAE was trained on
# resid_post_layer_{TARGET_LAYER}. A causal head layer must be <= TARGET_LAYER.
HEAD_CAUSAL_FEATURE_SCORE_LAYER = int(TARGET_LAYER)

# Use the configured source/target blocks. Default: source Example 7 -> target Example 8.
HEAD_CAUSAL_PREVIOUS_BLOCK_INDEX = int(LFH_SOURCE_BLOCK_INDEX)
HEAD_CAUSAL_TARGET_BLOCK_INDEX = int(LFH_TARGET_BLOCK_INDEX)

# Intervention scope:
#   "target_block" edits only target/example-8 tokens. This is the cleanest
#   test of the proposed write-side mechanism.
#   "previous_block" edits only source/example tokens.
#   "previous_and_target" edits both.
#   "all_example_tokens" edits all example-block tokens.
HEAD_CAUSAL_TOKEN_SCOPE = "target_block"

# Additive alphas for the selected head's o_proj input on edited tokens.
# The effective multiplier is (1 + alpha):
#   alpha =  0.0: no intervention / baseline
#   alpha = -1.0: ablate this head on edited tokens
#   alpha =  1.0: add one extra copy of this head's contribution
HEAD_CAUSAL_HEAD_ALPHAS = [-5.0, -2.0, -1.0, 0.0, 1.0, 2.0,5.0]
HEAD_CAUSAL_HEAD_ALPHAS = [-5.0, -2.0, -1.0, 0.0, 1.0, 2.0,5.0]

# The selected model can be large. Start small, then scale up.
HEAD_CAUSAL_MAX_QUERIES = ATTN_WRITE_TEST_MAX_QUERIES if "ATTN_WRITE_TEST_MAX_QUERIES" in globals() else min(10, len(prompts_all_agnews))
HEAD_CAUSAL_MAX_VARIANTS = ATTN_WRITE_TEST_MAX_VARIANTS if "ATTN_WRITE_TEST_MAX_VARIANTS" in globals() else None

# For high-vs-low source activation summaries.
HEAD_CAUSAL_QUANTILE = 0.25

EPS = 1e-9

print("=" * 120)
print("Attention-head causal intervention config")
print("=" * 120)
print("feature_idx:", HEAD_CAUSAL_FEATURE_IDX)
print("attention layer:", HEAD_CAUSAL_LAYER)
print("attention head:", HEAD_CAUSAL_HEAD)
print("feature score layer:", HEAD_CAUSAL_FEATURE_SCORE_LAYER)
print("token scope:", HEAD_CAUSAL_TOKEN_SCOPE)
print("head alphas:", HEAD_CAUSAL_HEAD_ALPHAS)
print("max queries:", HEAD_CAUSAL_MAX_QUERIES)
print("max variants:", HEAD_CAUSAL_MAX_VARIANTS)
print("=" * 120)

if HEAD_CAUSAL_LAYER > HEAD_CAUSAL_FEATURE_SCORE_LAYER:
    raise ValueError(
        f"HEAD_CAUSAL_LAYER={HEAD_CAUSAL_LAYER} is later than "
        f"HEAD_CAUSAL_FEATURE_SCORE_LAYER={HEAD_CAUSAL_FEATURE_SCORE_LAYER}. "
        "A later attention head cannot causally affect an earlier SAE-layer activation "
        "in the same forward pass. Choose HEAD_CAUSAL_LAYER <= TARGET_LAYER."
    )


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def _safe_pearson_local(x, y):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    finite = np.isfinite(x) & np.isfinite(y)
    x = x[finite]
    y = y[finite]
    if len(x) < 3:
        return np.nan
    if np.std(x) <= EPS or np.std(y) <= EPS:
        return np.nan
    return float(pearsonr(x, y)[0])


def _attn_num_heads_and_head_dim(model, layer_idx: int):
    attn = model.get_submodule(f"model.layers.{layer_idx}.self_attn")
    num_heads = int(getattr(attn, "num_heads", model.config.num_attention_heads))
    head_dim = int(getattr(attn, "head_dim", model.config.hidden_size // num_heads))
    return num_heads, head_dim


def build_head_intervention_token_mask(
    seq_len: int,
    block_spans,
    previous_block_index: int,
    target_block_index: int,
    token_scope: str = "target_block",
):
    """
    Build a [1, seq_len] bool mask specifying which token positions receive
    the head ablation/amplification.
    """
    mask = torch.zeros((1, seq_len), dtype=torch.bool)

    prev_span = block_spans[previous_block_index]
    tgt_span = block_spans[target_block_index]

    def _mark(selection):
        idx = token_selection_to_torch_indices(selection, device=mask.device)
        mask[0, idx] = True

    if token_scope == "target_block":
        _mark(tgt_span)
    elif token_scope == "previous_block":
        _mark(prev_span)
    elif token_scope == "previous_and_target":
        _mark(prev_span)
        _mark(tgt_span)
    elif token_scope == "all_example_tokens":
        for selection in block_spans:
            _mark(selection)
    else:
        raise ValueError(f"Unknown token_scope: {token_scope}")

    return mask


@torch.no_grad()
def run_forward_capture_feature_layer_with_head_alpha(
    model,
    input_ids,
    attention_mask,
    *,
    head_layer_idx: int,
    head_idx: int,
    feature_layer_idx: int,
    token_mask: torch.Tensor,
    head_alpha: float = 0.0,
):
    """
    Run one forward pass while optionally scaling one attention head's o_proj
    input on selected tokens, then capture the feature_layer_idx residual output.

    The edited tensor is the input to self_attn.o_proj, shaped [B, S, hidden].
    It is a concatenation of per-head attention outputs before W_O.

    head_alpha semantics:
        0.0: no intervention / baseline
       -1.0: ablate selected head on selected tokens
        1.0: add one extra copy of the selected head's contribution
        alpha: effective multiplier is (1 + alpha)
    """
    if head_layer_idx > feature_layer_idx:
        raise ValueError(
            f"head_layer_idx={head_layer_idx} > feature_layer_idx={feature_layer_idx}. "
            "This cannot affect the measured feature activation in the same forward pass."
        )

    cache = {}

    feature_block = model.get_submodule(f"model.layers.{feature_layer_idx}")
    o_proj = model.get_submodule(f"model.layers.{head_layer_idx}.self_attn.o_proj")
    num_heads, head_dim = _attn_num_heads_and_head_dim(model, head_layer_idx)

    if not (0 <= int(head_idx) < num_heads):
        raise ValueError(f"head_idx={head_idx} outside [0, {num_heads}).")

    head_start = int(head_idx) * head_dim
    head_end = head_start + head_dim

    def feature_block_hook(module, inputs, output):
        x = output[0] if isinstance(output, tuple) else output
        cache["feature_layer_output"] = x.detach().clone()
        return output

    def o_proj_pre_hook(module, inputs):
        z = inputs[0]

        # alpha 0.0 is exactly no intervention.
        if float(head_alpha) == 0.0:
            return None

        if z.shape[-1] != num_heads * head_dim:
            raise RuntimeError(
                f"o_proj input last dim {z.shape[-1]} != num_heads*head_dim "
                f"{num_heads}*{head_dim}."
            )

        edit_mask = token_mask.to(device=z.device, dtype=torch.bool)
        if edit_mask.ndim != 2 or edit_mask.shape[0] != z.shape[0] or edit_mask.shape[1] != z.shape[1]:
            raise RuntimeError(
                f"token_mask shape {tuple(edit_mask.shape)} incompatible with o_proj input shape {tuple(z.shape)}."
            )

        z_edit = z.clone()
        head_slice = z_edit[..., head_start:head_end]
        effective_multiplier = 1.0 + float(head_alpha)
        scaled_head_slice = head_slice * effective_multiplier
        z_edit[..., head_start:head_end] = torch.where(edit_mask[..., None], scaled_head_slice, head_slice)

        return (z_edit,) + tuple(inputs[1:])

    h_feature = feature_block.register_forward_hook(feature_block_hook)
    h_head = o_proj.register_forward_pre_hook(o_proj_pre_hook)

    try:
        _ = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
        )
    finally:
        h_head.remove()
        h_feature.remove()

    if "feature_layer_output" not in cache:
        raise RuntimeError(f"Failed to capture feature layer {feature_layer_idx} output.")

    return cache["feature_layer_output"]


@torch.no_grad()
def measure_prompt_feature_with_head_alpha(
    prompt: str,
    *,
    feature_idx: int,
    head_layer_idx: int,
    head_idx: int,
    feature_layer_idx: int,
    head_alpha: float,
    previous_block_index: int = -2,
    target_block_index: int = -1,
    token_scope: str = "target_block",
    expected_num_blocks: int = NUM_BLOCKS_TOTAL,
):
    """
    For one prompt, run a selected attention-head intervention and measure
    feature pre-activation on previous and target exemplar blocks.
    """
    enc = tokenize_prompt_for_offsets(prompt)
    offset_mapping = enc["offset_mapping"][0].tolist()
    input_ids = enc["input_ids"].to(MODEL_INPUT_DEVICE)
    attention_mask = enc["attention_mask"].to(MODEL_INPUT_DEVICE)

    block_spans = get_example_block_token_spans(
        prompt=prompt,
        offset_mapping=offset_mapping,
        expected_num_blocks=expected_num_blocks,
    )

    token_mask = build_head_intervention_token_mask(
        seq_len=input_ids.shape[1],
        block_spans=block_spans,
        previous_block_index=previous_block_index,
        target_block_index=target_block_index,
        token_scope=token_scope,
    ).to(MODEL_INPUT_DEVICE)

    layer_output = run_forward_capture_feature_layer_with_head_alpha(
        model=model,
        input_ids=input_ids,
        attention_mask=attention_mask,
        head_layer_idx=head_layer_idx,
        head_idx=head_idx,
        feature_layer_idx=feature_layer_idx,
        token_mask=token_mask,
        head_alpha=head_alpha,
    )

    pre_f = sae_single_feature_pre_activation(
        sae=sae,
        x=layer_output,
        feature_idx=feature_idx,
    )[0].detach().float()

    prev_selection = block_spans[previous_block_index]
    target_selection = block_spans[target_block_index]
    prev_idx = token_selection_to_torch_indices(prev_selection, device=pre_f.device)
    target_idx = token_selection_to_torch_indices(target_selection, device=pre_f.device)

    return {
        "previous_feature_mean": float(pre_f.index_select(0, prev_idx).mean().detach().cpu()),
        "target_feature_mean": float(pre_f.index_select(0, target_idx).mean().detach().cpu()),
        "prev_span": block_spans[previous_block_index],
        "target_span": block_spans[target_block_index],
    }


def _querywise_corr_summary(records_df, target_col: str, prev_col: str = "baseline_previous_feature_mean"):
    """
    Compute pooled and query-wise correlations for prev_col -> target_col.
    """
    pooled = _safe_pearson_local(records_df[prev_col].values, records_df[target_col].values)

    q_corrs = []
    for q, g in records_df.groupby("query_idx"):
        r = _safe_pearson_local(g[prev_col].values, g[target_col].values)
        if np.isfinite(r):
            q_corrs.append(r)

    if len(q_corrs) == 0:
        return pooled, np.nan, np.nan, 0

    return pooled, float(np.mean(q_corrs)), float(np.std(q_corrs)), int(len(q_corrs))


def _high_low_delta_by_query(records_df, value_col: str, prev_col: str = "baseline_previous_feature_mean", quantile: float = 0.25):
    """
    For each query, compute mean(value | prev high) - mean(value | prev low).
    """
    deltas = []
    for q, g in records_df.groupby("query_idx"):
        x = g[prev_col].to_numpy(dtype=np.float64)
        y = g[value_col].to_numpy(dtype=np.float64)
        finite = np.isfinite(x) & np.isfinite(y)
        x = x[finite]
        y = y[finite]
        if len(x) < 4:
            continue
        lo = np.quantile(x, quantile)
        hi = np.quantile(x, 1.0 - quantile)
        low_mask = x <= lo
        high_mask = x >= hi
        if low_mask.sum() == 0 or high_mask.sum() == 0:
            continue
        deltas.append(float(y[high_mask].mean() - y[low_mask].mean()))
    if len(deltas) == 0:
        return np.nan, np.nan, 0
    return float(np.mean(deltas)), float(np.std(deltas)), int(len(deltas))


@torch.no_grad()
def run_attention_head_alpha_causal_test(
    *,
    feature_idx: int,
    layer_idx: int,
    head_idx: int,
    prompts_all_agnews,
    feature_score_layer: int = TARGET_LAYER,
    head_alphas=(-10.0, -5.0, -1.0, 0.0, 1.0, 5.0, 10.0),
    previous_block_index: int = -2,
    target_block_index: int = -1,
    token_scope: str = "target_block",
    max_queries=None,
    max_variants=None,
    quantile: float = 0.25,
    verbose_every: int = 10,
):
    """
    Main causal test.

    For each prompt variant:
      1. Run baseline with head_alpha=0.0.
      2. Run each requested intervention alpha.
      3. Recompute corr(prev_feature_baseline, target_feature_intervened).

    A candidate negative-write head should show:
      - ablation at alpha=-1: negative correlation weakens;
      - amplification at alpha>0: negative correlation strengthens.
    """
    feature_idx = int(feature_idx)
    layer_idx = int(layer_idx)
    head_idx = int(head_idx)
    feature_score_layer = int(feature_score_layer)

    if layer_idx > feature_score_layer:
        raise ValueError(
            f"layer_idx={layer_idx} > feature_score_layer={feature_score_layer}; "
            "the intervention cannot affect the measured feature in the same forward pass."
        )

    # De-duplicate alphas but keep order; ensure baseline alpha=0.0 is available internally.
    requested_alphas = []
    for alpha in head_alphas:
        af = float(alpha)
        if af not in requested_alphas:
            requested_alphas.append(af)
    if 0.0 not in requested_alphas:
        alphas_to_run = [0.0] + requested_alphas
    else:
        alphas_to_run = requested_alphas

    num_queries_total = len(prompts_all_agnews)
    query_indices = list(range(num_queries_total if max_queries is None else min(max_queries, num_queries_total)))

    rows = []

    for query_counter, query_idx in enumerate(query_indices, start=1):
        variants = prompts_all_agnews[query_idx]
        if max_variants is not None:
            variants = variants[:max_variants]

        for variant_idx, prompt in enumerate(variants):
            # Baseline first.
            baseline = measure_prompt_feature_with_head_alpha(
                prompt=prompt,
                feature_idx=feature_idx,
                head_layer_idx=layer_idx,
                head_idx=head_idx,
                feature_layer_idx=feature_score_layer,
                head_alpha=0.0,
                previous_block_index=previous_block_index,
                target_block_index=target_block_index,
                token_scope=token_scope,
                expected_num_blocks=NUM_BLOCKS_TOTAL,
            )

            base_prev = baseline["previous_feature_mean"]
            base_tgt = baseline["target_feature_mean"]

            for alpha in alphas_to_run:
                if float(alpha) == 0.0:
                    previous_mean = base_prev
                    target_mean = base_tgt
                else:
                    intervened = measure_prompt_feature_with_head_alpha(
                        prompt=prompt,
                        feature_idx=feature_idx,
                        head_layer_idx=layer_idx,
                        head_idx=head_idx,
                        feature_layer_idx=feature_score_layer,
                        head_alpha=float(alpha),
                        previous_block_index=previous_block_index,
                        target_block_index=target_block_index,
                        token_scope=token_scope,
                        expected_num_blocks=NUM_BLOCKS_TOTAL,
                    )
                    previous_mean = intervened["previous_feature_mean"]
                    target_mean = intervened["target_feature_mean"]

                rows.append({
                    "query_idx": int(query_idx),
                    "variant_idx": int(variant_idx),
                    "feature_idx": int(feature_idx),
                    "layer": int(layer_idx),
                    "head": int(head_idx),
                    "feature_score_layer": int(feature_score_layer),
                    "token_scope": token_scope,
                    "head_alpha": float(alpha),
                    "effective_head_multiplier": float(1.0 + float(alpha)),
                    "baseline_previous_feature_mean": float(base_prev),
                    "baseline_target_feature_mean": float(base_tgt),
                    "intervened_previous_feature_mean": float(previous_mean),
                    "intervened_target_feature_mean": float(target_mean),
                    "target_delta_intervened_minus_baseline": float(target_mean - base_tgt),
                    "previous_delta_intervened_minus_baseline": float(previous_mean - base_prev),
                })

        if verbose_every is not None and (query_counter % max(verbose_every, 1) == 0 or query_counter == len(query_indices)):
            print(f"processed query {query_counter}/{len(query_indices)}; rows={len(rows)}")

    records_df = pd.DataFrame(rows)

    # Baseline correlation summary from alpha 0.0 rows.
    baseline_rows = records_df[records_df["head_alpha"] == 0.0].copy()
    baseline_pooled_r, baseline_mean_query_r, baseline_std_query_r, baseline_n_query = _querywise_corr_summary(
        baseline_rows,
        target_col="baseline_target_feature_mean",
        prev_col="baseline_previous_feature_mean",
    )

    summary_rows = []
    for alpha in requested_alphas:
        sub = records_df[records_df["head_alpha"] == float(alpha)].copy()
        if len(sub) == 0:
            continue

        pooled_r, mean_query_r, std_query_r, n_query = _querywise_corr_summary(
            sub,
            target_col="intervened_target_feature_mean",
            prev_col="baseline_previous_feature_mean",
        )

        delta_pooled_r, delta_mean_query_r, delta_std_query_r, delta_n_query = _querywise_corr_summary(
            sub,
            target_col="target_delta_intervened_minus_baseline",
            prev_col="baseline_previous_feature_mean",
        )

        high_low_target_delta_mean, high_low_target_delta_std, high_low_n = _high_low_delta_by_query(
            sub,
            value_col="target_delta_intervened_minus_baseline",
            prev_col="baseline_previous_feature_mean",
            quantile=quantile,
        )

        high_low_target_mean, high_low_target_std, high_low_target_n = _high_low_delta_by_query(
            sub,
            value_col="intervened_target_feature_mean",
            prev_col="baseline_previous_feature_mean",
            quantile=quantile,
        )

        summary_rows.append({
            "feature_idx": int(feature_idx),
            "layer": int(layer_idx),
            "head": int(head_idx),
            "feature_score_layer": int(feature_score_layer),
            "token_scope": token_scope,
            "head_alpha": float(alpha),
            "effective_head_multiplier": float(1.0 + float(alpha)),

            "baseline_pooled_corr_prev_to_target": baseline_pooled_r,
            "baseline_mean_query_corr_prev_to_target": baseline_mean_query_r,
            "baseline_std_query_corr_prev_to_target": baseline_std_query_r,

            "intervened_pooled_corr_prev_to_target": pooled_r,
            "intervened_mean_query_corr_prev_to_target": mean_query_r,
            "intervened_std_query_corr_prev_to_target": std_query_r,

            "corr_change_pooled_intervened_minus_baseline": pooled_r - baseline_pooled_r,
            "corr_change_mean_query_intervened_minus_baseline": mean_query_r - baseline_mean_query_r,

            "corr_prev_to_target_delta_pooled": delta_pooled_r,
            "corr_prev_to_target_delta_mean_query": delta_mean_query_r,

            "mean_target_delta_intervened_minus_baseline": float(sub["target_delta_intervened_minus_baseline"].mean()),
            "median_target_delta_intervened_minus_baseline": float(sub["target_delta_intervened_minus_baseline"].median()),
            "fraction_target_delta_positive": float((sub["target_delta_intervened_minus_baseline"] > 0).mean()),

            "high_minus_low_prev_target_delta_mean": high_low_target_delta_mean,
            "high_minus_low_prev_target_delta_std": high_low_target_delta_std,
            "high_minus_low_prev_intervened_target_mean": high_low_target_mean,
            "high_minus_low_prev_intervened_target_std": high_low_target_std,
            "n_queries_for_high_low": int(high_low_n),
        })

    summary_df = pd.DataFrame(summary_rows).sort_values("head_alpha").reset_index(drop=True)

    print("=" * 120)
    print(f"Head causal test: feature={feature_idx}, layer={layer_idx}, head={head_idx}")
    print(f"Baseline pooled corr(prev, target): {baseline_pooled_r:.4f}")
    print(f"Baseline mean query-wise corr(prev, target): {baseline_mean_query_r:.4f}")
    print("Interpretation:")
    print("  alpha=0.0 leaves this head unchanged on target tokens.")
    print("  alpha=-1.0 ablates this head on target tokens.")
    print("  alpha>0 adds extra copies of this head's contribution on target tokens.")
    print("  effective multiplier = 1 + alpha.")
    print("  If the head implements negative write, ablation should make a negative corr less negative,")
    print("  while amplification should make it more negative.")
    print("=" * 120)
    display(summary_df)

    return summary_df, records_df


# ------------------------------------------------------------
# Plotting helpers (Updated with Custom Font Properties)
# ------------------------------------------------------------

# aaaaa
def plot_head_causal_dose_response(summary_df):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=summary_df["head_alpha"],
        y=summary_df["intervened_mean_query_corr_prev_to_target"],
        mode="lines+markers",
        name="intervened corr(prev, target), mean query-wise",
        hovertemplate="alpha=%{x}<br>corr=%{y:.4f}<extra></extra>",
    ))

    baseline = float(summary_df["baseline_mean_query_corr_prev_to_target"].iloc[0])
    fig.add_hline(y=baseline, line_dash="dash", annotation_text="baseline corr")
    fig.add_vline(x=0.0, line_dash="dash", annotation_text="no intervention")

    fig.update_layout(
        font=dict(family="Times New Roman"),
        title=(
            "Attention-Head Intervention Response on Local Feature Homeostasis<br>"
            f"Model=Gemma2-9B-IT, Feature={int(summary_df['feature_idx'].iloc[0])}, "
            f"Layer={int(summary_df['layer'].iloc[0])}, "
            f"Head={int(summary_df['head'].iloc[0])}"
        ),
        title_font_size=26,
        xaxis_title="Intervention Strength (Alpha) on Target-Block Tokens",
        xaxis_title_font_size=22,
        yaxis_title="Mean corr(source feature, target feature)",
        yaxis_title_font_size=22,
        width=900,
        height=600,
        hovermode="x unified",
    )
    
    # --- ADDED: 4x Resolution Config for Interactive 'Download Plot' Button ---
    config = {
        'toImageButtonOptions': {
            'format': 'png',         # one of png, svg, jpeg, webp
            'filename': 'head_causal_dose_response',
            'height': 600,
            'width': 900,
            'scale': 4               # Multiply resolution by 4
        }
    }
    
    fig.show(config=config)
    
    # --- ADDED: Optional direct save to disk at 4x resolution ---
    # Uncomment the line below to save programmatically (requires pip install kaleido)
    # fig.write_image("head_causal_dose_response_highres.png", scale=4)

    return fig


def plot_head_causal_delta_response(summary_df):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=summary_df["head_alpha"],
        y=summary_df["corr_prev_to_target_delta_mean_query"],
        mode="lines+markers",
        name="corr(prev, target_delta)",
        hovertemplate="alpha=%{x}<br>corr=%{y:.4f}<extra></extra>",
    ))

    fig.add_trace(go.Scatter(
        x=summary_df["head_alpha"],
        y=summary_df["high_minus_low_prev_target_delta_mean"],
        mode="lines+markers",
        name="high-prev minus low-prev target delta",
        hovertemplate="alpha=%{x}<br>high-low delta=%{y:.4f}<extra></extra>",
    ))

    fig.add_hline(y=0.0, line_dash="dash")
    fig.add_vline(x=0.0, line_dash="dash", annotation_text="no intervention")

    fig.update_layout(
        font=dict(family="Times New Roman"),
        title="Does the intervention affect high-previous-activation variants differently?",
        title_font_size=26,
        xaxis_title="Head alpha on target-block tokens (effective multiplier = 1 + alpha)",
        xaxis_title_font_size=22,
        yaxis_title="Effect size",
        yaxis_title_font_size=22,
        width=900,
        height=600,
        hovermode="x unified",
    )
    fig.show()
    return fig


def plot_head_intervention_scatter(records_df, alpha: float):
    sub = records_df[records_df["head_alpha"] == float(alpha)].copy()
    if len(sub) == 0:
        raise ValueError(f"No rows for alpha={alpha}")

    x = sub["baseline_previous_feature_mean"].to_numpy(dtype=np.float64)
    y_base = sub["baseline_target_feature_mean"].to_numpy(dtype=np.float64)
    y_int = sub["intervened_target_feature_mean"].to_numpy(dtype=np.float64)

    r_base = _safe_pearson_local(x, y_base)
    r_int = _safe_pearson_local(x, y_int)

    df_plot = pd.DataFrame({
        "previous_feature_mean": np.concatenate([x, x]),
        "target_feature_mean": np.concatenate([y_base, y_int]),
        "condition": ["baseline"] * len(x) + [f"head_alpha={alpha}"] * len(x),
        "query_idx": np.concatenate([sub["query_idx"].values, sub["query_idx"].values]).astype(str),
        "variant_idx": np.concatenate([sub["variant_idx"].values, sub["variant_idx"].values]),
    })

    fig = px.scatter(
        df_plot,
        x="previous_feature_mean",
        y="target_feature_mean",
        color="condition",
        symbol="condition",
        hover_data=["query_idx", "variant_idx"],
        title=(
            f"Source-to-target feature relation before/after head intervention<br>"
            f"alpha={alpha}, baseline r={r_base:.3f}, intervened r={r_int:.3f}"
        ),
    )

    fig.update_layout(
        font=dict(family="Times New Roman"),
        title_font_size=26,
        width=900,
        height=600,
        xaxis_title="Mean SAE pre-activation on source exemplar",
        xaxis_title_font_size=22,
        yaxis_title="Mean SAE pre-activation on target exemplar",
        yaxis_title_font_size=22,
    )
    fig.show()
    return fig


# ------------------------------------------------------------
# Run the head intervention test
# ------------------------------------------------------------

head_causal_summary_df, head_causal_records_df = run_attention_head_alpha_causal_test(
    feature_idx=HEAD_CAUSAL_FEATURE_IDX,
    layer_idx=HEAD_CAUSAL_LAYER,
    head_idx=HEAD_CAUSAL_HEAD,
    prompts_all_agnews=prompts_all_agnews,
    feature_score_layer=HEAD_CAUSAL_FEATURE_SCORE_LAYER,
    head_alphas=HEAD_CAUSAL_HEAD_ALPHAS,
    previous_block_index=HEAD_CAUSAL_PREVIOUS_BLOCK_INDEX,
    target_block_index=HEAD_CAUSAL_TARGET_BLOCK_INDEX,
    token_scope=HEAD_CAUSAL_TOKEN_SCOPE,
    max_queries=HEAD_CAUSAL_MAX_QUERIES,
    max_variants=HEAD_CAUSAL_MAX_VARIANTS,
    quantile=HEAD_CAUSAL_QUANTILE,
    verbose_every=5,
)

_ = plot_head_causal_dose_response(head_causal_summary_df)
_ = plot_head_causal_delta_response(head_causal_summary_df)

# Plot ablation and amplification views if present.
if -1.0 in set(head_causal_records_df["head_alpha"].astype(float)):
    _ = plot_head_intervention_scatter(head_causal_records_df, alpha=-1.0)
if 1.0 in set(head_causal_records_df["head_alpha"].astype(float)):
    _ = plot_head_intervention_scatter(head_causal_records_df, alpha=1.0)

---

# Ablating top-K (e.g., k = 5) heads

In [ ]:
# # ============================================================
# # 15. Causal validation: ablate/amplify multiple attention heads
# #     and remeasure source-exemplar -> target-exemplar feature correlation
# # ============================================================

# # ------------------------------------------------------------
# # User-editable config
# # ------------------------------------------------------------

# MULTI_HEAD_CAUSAL_FEATURE_IDX = int(HEAD_CAUSAL_FEATURE_IDX)
# MULTI_HEAD_CAUSAL_LAYER = int(HEAD_CAUSAL_LAYER)
# MULTI_HEAD_CAUSAL_FEATURE_SCORE_LAYER = int(HEAD_CAUSAL_FEATURE_SCORE_LAYER)

# # By default, ablate/amplify the top-K heads from the shared-head search if
# # available; otherwise use the one-feature negative-write ranking.
# MULTI_HEAD_CAUSAL_TOP_K = 3
# MULTI_HEAD_CAUSAL_HEADS = None

# MULTI_HEAD_CAUSAL_HEAD_ALPHAS = list(HEAD_CAUSAL_HEAD_ALPHAS)
# MULTI_HEAD_CAUSAL_PREVIOUS_BLOCK_INDEX = int(HEAD_CAUSAL_PREVIOUS_BLOCK_INDEX)
# MULTI_HEAD_CAUSAL_TARGET_BLOCK_INDEX = int(HEAD_CAUSAL_TARGET_BLOCK_INDEX)
# MULTI_HEAD_CAUSAL_TOKEN_SCOPE = HEAD_CAUSAL_TOKEN_SCOPE
# MULTI_HEAD_CAUSAL_MAX_QUERIES = HEAD_CAUSAL_MAX_QUERIES
# MULTI_HEAD_CAUSAL_MAX_VARIANTS = HEAD_CAUSAL_MAX_VARIANTS
# MULTI_HEAD_CAUSAL_QUANTILE = HEAD_CAUSAL_QUANTILE


# def _ordered_unique_ints(values):
#     out = []
#     seen = set()
#     for value in values:
#         value = int(value)
#         if value not in seen:
#             out.append(value)
#             seen.add(value)
#     return out


# def default_multihead_causal_heads(top_k: int = 5):
#     """
#     Pick the default top-K heads for the multi-head intervention.

#     Preference order:
#       1. shared_heads_df, if the shared-head search has been run;
#       2. head_summary_df, if only the one-feature negative-write test has run;
#       3. the single head used in the previous causal test.
#     """
#     top_k = int(top_k)

#     if "shared_heads_df" in globals() and isinstance(shared_heads_df, pd.DataFrame):
#         if len(shared_heads_df) > 0 and "head" in shared_heads_df.columns:
#             return _ordered_unique_ints(shared_heads_df["head"].head(top_k).tolist())

#     if "head_summary_df" in globals() and isinstance(head_summary_df, pd.DataFrame):
#         if len(head_summary_df) > 0 and "head" in head_summary_df.columns:
#             return _ordered_unique_ints(head_summary_df["head"].head(top_k).tolist())

#     return [int(HEAD_CAUSAL_HEAD)]


# def validate_multihead_indices(head_indices, layer_idx: int):
#     num_heads, _ = _attn_num_heads_and_head_dim(model, int(layer_idx))
#     head_indices = _ordered_unique_ints(head_indices)

#     if len(head_indices) == 0:
#         raise ValueError("At least one attention head is required.")

#     bad = [h for h in head_indices if h < 0 or h >= num_heads]
#     if len(bad) > 0:
#         raise ValueError(f"Head indices {bad} outside valid range [0, {num_heads}).")

#     return head_indices


# if MULTI_HEAD_CAUSAL_HEADS is None:
#     MULTI_HEAD_CAUSAL_HEADS = default_multihead_causal_heads(MULTI_HEAD_CAUSAL_TOP_K)

# MULTI_HEAD_CAUSAL_HEADS = validate_multihead_indices(
#     MULTI_HEAD_CAUSAL_HEADS,
#     layer_idx=MULTI_HEAD_CAUSAL_LAYER,
# )

# print("=" * 120)
# print("Multi-head causal intervention config")
# print("=" * 120)
# print("feature_idx:", MULTI_HEAD_CAUSAL_FEATURE_IDX)
# print("attention layer:", MULTI_HEAD_CAUSAL_LAYER)
# print("attention heads:", MULTI_HEAD_CAUSAL_HEADS)
# print("feature score layer:", MULTI_HEAD_CAUSAL_FEATURE_SCORE_LAYER)
# print("token scope:", MULTI_HEAD_CAUSAL_TOKEN_SCOPE)
# print("head alphas:", MULTI_HEAD_CAUSAL_HEAD_ALPHAS)
# print("max queries:", MULTI_HEAD_CAUSAL_MAX_QUERIES)
# print("max variants:", MULTI_HEAD_CAUSAL_MAX_VARIANTS)
# print("=" * 120)

# if MULTI_HEAD_CAUSAL_LAYER > MULTI_HEAD_CAUSAL_FEATURE_SCORE_LAYER:
#     raise ValueError(
#         f"MULTI_HEAD_CAUSAL_LAYER={MULTI_HEAD_CAUSAL_LAYER} is later than "
#         f"MULTI_HEAD_CAUSAL_FEATURE_SCORE_LAYER={MULTI_HEAD_CAUSAL_FEATURE_SCORE_LAYER}. "
#         "A later attention head cannot causally affect an earlier SAE-layer activation "
#         "in the same forward pass. Choose MULTI_HEAD_CAUSAL_LAYER <= TARGET_LAYER."
#     )


# @torch.no_grad()
# def run_forward_capture_feature_layer_with_multihead_alpha(
#     model,
#     input_ids,
#     attention_mask,
#     *,
#     head_layer_idx: int,
#     head_indices,
#     feature_layer_idx: int,
#     token_mask: torch.Tensor,
#     head_alpha: float = 0.0,
# ):
#     """
#     Run one forward pass while applying the same additive alpha to multiple
#     attention heads' o_proj inputs on selected tokens, then capture the
#     feature_layer_idx residual output.

#     alpha semantics:
#        0.0: no intervention / baseline
#       -1.0: ablate selected heads on selected tokens
#        1.0: add one extra copy of each selected head's contribution
#       alpha: effective multiplier is (1 + alpha)
#     """
#     if head_layer_idx > feature_layer_idx:
#         raise ValueError(
#             f"head_layer_idx={head_layer_idx} > feature_layer_idx={feature_layer_idx}. "
#             "This cannot affect the measured feature activation in the same forward pass."
#         )

#     cache = {}

#     feature_block = model.get_submodule(f"model.layers.{feature_layer_idx}")
#     o_proj = model.get_submodule(f"model.layers.{head_layer_idx}.self_attn.o_proj")
#     num_heads, head_dim = _attn_num_heads_and_head_dim(model, head_layer_idx)
#     head_indices = validate_multihead_indices(head_indices, layer_idx=head_layer_idx)
#     head_slices = [(int(h) * head_dim, int(h) * head_dim + head_dim) for h in head_indices]

#     def feature_block_hook(module, inputs, output):
#         x = output[0] if isinstance(output, tuple) else output
#         cache["feature_layer_output"] = x.detach().clone()
#         return output

#     def o_proj_pre_hook(module, inputs):
#         z = inputs[0]

#         # alpha 0.0 is exactly no intervention.
#         if float(head_alpha) == 0.0:
#             return None

#         if z.shape[-1] != num_heads * head_dim:
#             raise RuntimeError(
#                 f"o_proj input last dim {z.shape[-1]} != num_heads*head_dim "
#                 f"{num_heads}*{head_dim}."
#             )

#         edit_mask = token_mask.to(device=z.device, dtype=torch.bool)
#         if edit_mask.ndim != 2 or edit_mask.shape[0] != z.shape[0] or edit_mask.shape[1] != z.shape[1]:
#             raise RuntimeError(
#                 f"token_mask shape {tuple(edit_mask.shape)} incompatible with o_proj input shape {tuple(z.shape)}."
#             )

#         z_edit = z.clone()
#         effective_multiplier = 1.0 + float(head_alpha)

#         for head_start, head_end in head_slices:
#             head_slice = z[..., head_start:head_end]
#             scaled_head_slice = head_slice * effective_multiplier
#             z_edit[..., head_start:head_end] = torch.where(edit_mask[..., None], scaled_head_slice, head_slice)

#         return (z_edit,) + tuple(inputs[1:])

#     h_feature = feature_block.register_forward_hook(feature_block_hook)
#     h_head = o_proj.register_forward_pre_hook(o_proj_pre_hook)

#     try:
#         _ = model(
#             input_ids=input_ids,
#             attention_mask=attention_mask,
#             use_cache=False,
#         )
#     finally:
#         h_head.remove()
#         h_feature.remove()

#     if "feature_layer_output" not in cache:
#         raise RuntimeError(f"Failed to capture feature layer {feature_layer_idx} output.")

#     return cache["feature_layer_output"]


# @torch.no_grad()
# def measure_prompt_feature_with_multihead_alpha(
#     prompt: str,
#     *,
#     feature_idx: int,
#     head_layer_idx: int,
#     head_indices,
#     feature_layer_idx: int,
#     head_alpha: float,
#     previous_block_index: int = -2,
#     target_block_index: int = -1,
#     token_scope: str = "target_block",
#     expected_num_blocks: int = NUM_BLOCKS_TOTAL,
# ):
#     """
#     For one prompt, run a multi-head intervention and measure feature
#     pre-activation on source and target exemplar blocks.
#     """
#     enc = tokenize_prompt_for_offsets(prompt)
#     offset_mapping = enc["offset_mapping"][0].tolist()
#     input_ids = enc["input_ids"].to(MODEL_INPUT_DEVICE)
#     attention_mask = enc["attention_mask"].to(MODEL_INPUT_DEVICE)

#     block_spans = get_example_block_token_spans(
#         prompt=prompt,
#         offset_mapping=offset_mapping,
#         expected_num_blocks=expected_num_blocks,
#     )

#     token_mask = build_head_intervention_token_mask(
#         seq_len=input_ids.shape[1],
#         block_spans=block_spans,
#         previous_block_index=previous_block_index,
#         target_block_index=target_block_index,
#         token_scope=token_scope,
#     ).to(MODEL_INPUT_DEVICE)

#     layer_output = run_forward_capture_feature_layer_with_multihead_alpha(
#         model=model,
#         input_ids=input_ids,
#         attention_mask=attention_mask,
#         head_layer_idx=head_layer_idx,
#         head_indices=head_indices,
#         feature_layer_idx=feature_layer_idx,
#         token_mask=token_mask,
#         head_alpha=head_alpha,
#     )

#     pre_f = sae_single_feature_pre_activation(
#         sae=sae,
#         x=layer_output,
#         feature_idx=feature_idx,
#     )[0].detach().float()

#     prev_selection = block_spans[previous_block_index]
#     target_selection = block_spans[target_block_index]
#     prev_idx = token_selection_to_torch_indices(prev_selection, device=pre_f.device)
#     target_idx = token_selection_to_torch_indices(target_selection, device=pre_f.device)

#     return {
#         "previous_feature_mean": float(pre_f.index_select(0, prev_idx).mean().detach().cpu()),
#         "target_feature_mean": float(pre_f.index_select(0, target_idx).mean().detach().cpu()),
#         "prev_span": block_spans[previous_block_index],
#         "target_span": block_spans[target_block_index],
#     }


# @torch.no_grad()
# def run_attention_multihead_alpha_causal_test(
#     *,
#     feature_idx: int,
#     layer_idx: int,
#     head_indices,
#     prompts_all_agnews,
#     feature_score_layer: int = TARGET_LAYER,
#     head_alphas=(-10.0, -5.0, -1.0, 0.0, 1.0, 5.0, 10.0),
#     previous_block_index: int = -2,
#     target_block_index: int = -1,
#     token_scope: str = "target_block",
#     max_queries=None,
#     max_variants=None,
#     quantile: float = 0.25,
#     verbose_every: int = 10,
# ):
#     """
#     Main multi-head causal test.

#     For each prompt variant:
#       1. Run baseline with head_alpha=0.0.
#       2. Run each requested intervention alpha on all selected heads at once.
#       3. Recompute corr(prev_feature_baseline, target_feature_intervened).
#     """
#     feature_idx = int(feature_idx)
#     layer_idx = int(layer_idx)
#     feature_score_layer = int(feature_score_layer)
#     head_indices = validate_multihead_indices(head_indices, layer_idx=layer_idx)
#     heads_label = ",".join(str(h) for h in head_indices)

#     if layer_idx > feature_score_layer:
#         raise ValueError(
#             f"layer_idx={layer_idx} > feature_score_layer={feature_score_layer}; "
#             "the intervention cannot affect the measured feature in the same forward pass."
#         )

#     requested_alphas = []
#     for alpha in head_alphas:
#         af = float(alpha)
#         if af not in requested_alphas:
#             requested_alphas.append(af)

#     if 0.0 not in requested_alphas:
#         alphas_to_run = [0.0] + requested_alphas
#     else:
#         alphas_to_run = requested_alphas

#     num_queries_total = len(prompts_all_agnews)
#     query_indices = list(range(num_queries_total if max_queries is None else min(max_queries, num_queries_total)))

#     rows = []

#     for query_counter, query_idx in enumerate(query_indices, start=1):
#         variants = prompts_all_agnews[query_idx]
#         if max_variants is not None:
#             variants = variants[:max_variants]

#         for variant_idx, prompt in enumerate(variants):
#             baseline = measure_prompt_feature_with_multihead_alpha(
#                 prompt=prompt,
#                 feature_idx=feature_idx,
#                 head_layer_idx=layer_idx,
#                 head_indices=head_indices,
#                 feature_layer_idx=feature_score_layer,
#                 head_alpha=0.0,
#                 previous_block_index=previous_block_index,
#                 target_block_index=target_block_index,
#                 token_scope=token_scope,
#                 expected_num_blocks=NUM_BLOCKS_TOTAL,
#             )

#             base_prev = baseline["previous_feature_mean"]
#             base_tgt = baseline["target_feature_mean"]

#             for alpha in alphas_to_run:
#                 if float(alpha) == 0.0:
#                     previous_mean = base_prev
#                     target_mean = base_tgt
#                 else:
#                     intervened = measure_prompt_feature_with_multihead_alpha(
#                         prompt=prompt,
#                         feature_idx=feature_idx,
#                         head_layer_idx=layer_idx,
#                         head_indices=head_indices,
#                         feature_layer_idx=feature_score_layer,
#                         head_alpha=float(alpha),
#                         previous_block_index=previous_block_index,
#                         target_block_index=target_block_index,
#                         token_scope=token_scope,
#                         expected_num_blocks=NUM_BLOCKS_TOTAL,
#                     )
#                     previous_mean = intervened["previous_feature_mean"]
#                     target_mean = intervened["target_feature_mean"]

#                 rows.append({
#                     "query_idx": int(query_idx),
#                     "variant_idx": int(variant_idx),
#                     "feature_idx": int(feature_idx),
#                     "layer": int(layer_idx),
#                     "heads": heads_label,
#                     "head_indices": tuple(int(h) for h in head_indices),
#                     "num_heads": int(len(head_indices)),
#                     "feature_score_layer": int(feature_score_layer),
#                     "token_scope": token_scope,
#                     "head_alpha": float(alpha),
#                     "effective_head_multiplier": float(1.0 + float(alpha)),
#                     "baseline_previous_feature_mean": float(base_prev),
#                     "baseline_target_feature_mean": float(base_tgt),
#                     "intervened_previous_feature_mean": float(previous_mean),
#                     "intervened_target_feature_mean": float(target_mean),
#                     "target_delta_intervened_minus_baseline": float(target_mean - base_tgt),
#                     "previous_delta_intervened_minus_baseline": float(previous_mean - base_prev),
#                 })

#         if verbose_every is not None and (query_counter % max(verbose_every, 1) == 0 or query_counter == len(query_indices)):
#             print(f"processed query {query_counter}/{len(query_indices)}; rows={len(rows)}")

#     records_df = pd.DataFrame(rows)

#     baseline_rows = records_df[records_df["head_alpha"] == 0.0].copy()
#     baseline_pooled_r, baseline_mean_query_r, baseline_std_query_r, baseline_n_query = _querywise_corr_summary(
#         baseline_rows,
#         target_col="baseline_target_feature_mean",
#         prev_col="baseline_previous_feature_mean",
#     )

#     summary_rows = []
#     for alpha in requested_alphas:
#         sub = records_df[records_df["head_alpha"] == float(alpha)].copy()
#         if len(sub) == 0:
#             continue

#         pooled_r, mean_query_r, std_query_r, n_query = _querywise_corr_summary(
#             sub,
#             target_col="intervened_target_feature_mean",
#             prev_col="baseline_previous_feature_mean",
#         )

#         delta_pooled_r, delta_mean_query_r, delta_std_query_r, delta_n_query = _querywise_corr_summary(
#             sub,
#             target_col="target_delta_intervened_minus_baseline",
#             prev_col="baseline_previous_feature_mean",
#         )

#         high_low_target_delta_mean, high_low_target_delta_std, high_low_n = _high_low_delta_by_query(
#             sub,
#             value_col="target_delta_intervened_minus_baseline",
#             prev_col="baseline_previous_feature_mean",
#             quantile=quantile,
#         )

#         high_low_target_mean, high_low_target_std, high_low_target_n = _high_low_delta_by_query(
#             sub,
#             value_col="intervened_target_feature_mean",
#             prev_col="baseline_previous_feature_mean",
#             quantile=quantile,
#         )

#         summary_rows.append({
#             "feature_idx": int(feature_idx),
#             "layer": int(layer_idx),
#             "heads": heads_label,
#             "head_indices": tuple(int(h) for h in head_indices),
#             "num_heads": int(len(head_indices)),
#             "feature_score_layer": int(feature_score_layer),
#             "token_scope": token_scope,
#             "head_alpha": float(alpha),
#             "effective_head_multiplier": float(1.0 + float(alpha)),

#             "baseline_pooled_corr_prev_to_target": baseline_pooled_r,
#             "baseline_mean_query_corr_prev_to_target": baseline_mean_query_r,
#             "baseline_std_query_corr_prev_to_target": baseline_std_query_r,

#             "intervened_pooled_corr_prev_to_target": pooled_r,
#             "intervened_mean_query_corr_prev_to_target": mean_query_r,
#             "intervened_std_query_corr_prev_to_target": std_query_r,

#             "corr_change_pooled_intervened_minus_baseline": pooled_r - baseline_pooled_r,
#             "corr_change_mean_query_intervened_minus_baseline": mean_query_r - baseline_mean_query_r,

#             "corr_prev_to_target_delta_pooled": delta_pooled_r,
#             "corr_prev_to_target_delta_mean_query": delta_mean_query_r,

#             "mean_target_delta_intervened_minus_baseline": float(sub["target_delta_intervened_minus_baseline"].mean()),
#             "median_target_delta_intervened_minus_baseline": float(sub["target_delta_intervened_minus_baseline"].median()),
#             "fraction_target_delta_positive": float((sub["target_delta_intervened_minus_baseline"] > 0).mean()),

#             "high_minus_low_prev_target_delta_mean": high_low_target_delta_mean,
#             "high_minus_low_prev_target_delta_std": high_low_target_delta_std,
#             "high_minus_low_prev_intervened_target_mean": high_low_target_mean,
#             "high_minus_low_prev_intervened_target_std": high_low_target_std,
#             "n_queries_for_high_low": int(high_low_n),
#         })

#     summary_df = pd.DataFrame(summary_rows).sort_values("head_alpha").reset_index(drop=True)

#     print("=" * 120)
#     print(f"Multi-head causal test: feature={feature_idx}, layer={layer_idx}, heads={head_indices}")
#     print(f"Baseline pooled corr(prev, target): {baseline_pooled_r:.4f}")
#     print(f"Baseline mean query-wise corr(prev, target): {baseline_mean_query_r:.4f}")
#     print("Interpretation:")
#     print("  alpha=0.0 leaves selected heads unchanged on target tokens.")
#     print("  alpha=-1.0 ablates selected heads on target tokens.")
#     print("  alpha>0 adds extra copies of each selected head's contribution on target tokens.")
#     print("  effective multiplier = 1 + alpha.")
#     print("=" * 120)
#     display(summary_df)

#     return summary_df, records_df


# def plot_multihead_causal_dose_response(summary_df):
#     fig = go.Figure()

#     fig.add_trace(go.Scatter(
#         x=summary_df["head_alpha"],
#         y=summary_df["intervened_mean_query_corr_prev_to_target"],
#         mode="lines+markers",
#         name="intervened corr(prev, target), mean query-wise",
#         hovertemplate="alpha=%{x}<br>corr=%{y:.4f}<extra></extra>",
#     ))

#     baseline = float(summary_df["baseline_mean_query_corr_prev_to_target"].iloc[0])
#     fig.add_hline(y=baseline, line_dash="dash", annotation_text="baseline corr")
#     fig.add_vline(x=0.0, line_dash="dash", annotation_text="no intervention")

#     fig.update_layout(
#         title=(
#             "Multi-head intervention dose response<br>"
#             f"feature={int(summary_df['feature_idx'].iloc[0])}, "
#             f"layer={int(summary_df['layer'].iloc[0])}, "
#             f"heads={summary_df['heads'].iloc[0]}"
#         ),
#         xaxis_title="Head alpha on target-block tokens (effective multiplier = 1 + alpha)",
#         yaxis_title="Mean query-wise corr(source feature, target feature)",
#         width=900,
#         height=600,
#         hovermode="x unified",
#     )
#     fig.show()
#     return fig


# def plot_multihead_causal_delta_response(summary_df):
#     fig = go.Figure()

#     fig.add_trace(go.Scatter(
#         x=summary_df["head_alpha"],
#         y=summary_df["corr_prev_to_target_delta_mean_query"],
#         mode="lines+markers",
#         name="corr(prev, target_delta)",
#         hovertemplate="alpha=%{x}<br>corr=%{y:.4f}<extra></extra>",
#     ))

#     fig.add_trace(go.Scatter(
#         x=summary_df["head_alpha"],
#         y=summary_df["high_minus_low_prev_target_delta_mean"],
#         mode="lines+markers",
#         name="high-prev minus low-prev target delta",
#         hovertemplate="alpha=%{x}<br>high-low delta=%{y:.4f}<extra></extra>",
#     ))

#     fig.add_hline(y=0.0, line_dash="dash")
#     fig.add_vline(x=0.0, line_dash="dash", annotation_text="no intervention")

#     fig.update_layout(
#         title="Does the multi-head intervention affect high-previous-activation variants differently?",
#         xaxis_title="Head alpha on target-block tokens (effective multiplier = 1 + alpha)",
#         yaxis_title="Effect size",
#         width=900,
#         height=600,
#         hovermode="x unified",
#     )
#     fig.show()
#     return fig


# def plot_multihead_intervention_scatter(records_df, alpha: float):
#     sub = records_df[records_df["head_alpha"] == float(alpha)].copy()
#     if len(sub) == 0:
#         raise ValueError(f"No rows for alpha={alpha}")

#     x = sub["baseline_previous_feature_mean"].to_numpy(dtype=np.float64)
#     y_base = sub["baseline_target_feature_mean"].to_numpy(dtype=np.float64)
#     y_int = sub["intervened_target_feature_mean"].to_numpy(dtype=np.float64)

#     r_base = _safe_pearson_local(x, y_base)
#     r_int = _safe_pearson_local(x, y_int)

#     df_plot = pd.DataFrame({
#         "previous_feature_mean": np.concatenate([x, x]),
#         "target_feature_mean": np.concatenate([y_base, y_int]),
#         "condition": ["baseline"] * len(x) + [f"multihead_alpha={alpha}"] * len(x),
#         "query_idx": np.concatenate([sub["query_idx"].values, sub["query_idx"].values]).astype(str),
#         "variant_idx": np.concatenate([sub["variant_idx"].values, sub["variant_idx"].values]),
#     })

#     fig = px.scatter(
#         df_plot,
#         x="previous_feature_mean",
#         y="target_feature_mean",
#         color="condition",
#         symbol="condition",
#         hover_data=["query_idx", "variant_idx"],
#         title=(
#             f"Previous-to-target feature relation before/after multi-head intervention<br>"
#             f"alpha={alpha}, baseline r={r_base:.3f}, intervened r={r_int:.3f}"
#         ),
#     )

#     fig.update_layout(
#         width=900,
#         height=600,
#         xaxis_title="Mean SAE pre-activation on source exemplar",
#         yaxis_title="Mean SAE pre-activation on target exemplar",
#     )
#     fig.show()
#     return fig


# # ------------------------------------------------------------
# # Run the multi-head intervention test
# # ------------------------------------------------------------

# multihead_causal_summary_df, multihead_causal_records_df = run_attention_multihead_alpha_causal_test(
#     feature_idx=MULTI_HEAD_CAUSAL_FEATURE_IDX,
#     layer_idx=MULTI_HEAD_CAUSAL_LAYER,
#     head_indices=MULTI_HEAD_CAUSAL_HEADS,
#     prompts_all_agnews=prompts_all_agnews,
#     feature_score_layer=MULTI_HEAD_CAUSAL_FEATURE_SCORE_LAYER,
#     head_alphas=MULTI_HEAD_CAUSAL_HEAD_ALPHAS,
#     previous_block_index=MULTI_HEAD_CAUSAL_PREVIOUS_BLOCK_INDEX,
#     target_block_index=MULTI_HEAD_CAUSAL_TARGET_BLOCK_INDEX,
#     token_scope=MULTI_HEAD_CAUSAL_TOKEN_SCOPE,
#     max_queries=MULTI_HEAD_CAUSAL_MAX_QUERIES,
#     max_variants=MULTI_HEAD_CAUSAL_MAX_VARIANTS,
#     quantile=MULTI_HEAD_CAUSAL_QUANTILE,
#     verbose_every=5,
# )

# _ = plot_multihead_causal_dose_response(multihead_causal_summary_df)
# _ = plot_multihead_causal_delta_response(multihead_causal_summary_df)

# # Plot ablation and amplification views if present.
# if -1.0 in set(multihead_causal_records_df["head_alpha"].astype(float)):
#     _ = plot_multihead_intervention_scatter(multihead_causal_records_df, alpha=-1.0)
# if 1.0 in set(multihead_causal_records_df["head_alpha"].astype(float)):
#     _ = plot_multihead_intervention_scatter(multihead_causal_records_df, alpha=1.0)


---

# 12B. MLP negative-write test for one LFH feature

In [ ]:
# ============================================================
# 12B. MLP negative-write test for one feature
# ============================================================

import numpy as np
import pandas as pd
import torch
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import pearsonr

# ------------------------------------------------------------
# User-editable config
# ------------------------------------------------------------

MLP_WRITE_TEST_LAYER = TARGET_LAYER
MLP_WRITE_TEST_FEATURE_IDX = int(target_feature)

MLP_WRITE_TEST_PREVIOUS_BLOCK_INDEX = int(LFH_SOURCE_BLOCK_INDEX)
MLP_WRITE_TEST_TARGET_BLOCK_INDEX = int(LFH_TARGET_BLOCK_INDEX)

MLP_WRITE_TEST_MAX_QUERIES = min(10, len(prompts_all_agnews))
MLP_WRITE_TEST_MAX_VARIANTS = None

# Same high-vs-low definition as the attention-head test.
MLP_WRITE_TEST_QUANTILE = 0.25

EPS = 1e-9


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def _safe_pearson(x, y):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    finite = np.isfinite(x) & np.isfinite(y)
    x = x[finite]
    y = y[finite]
    if len(x) < 3:
        return np.nan
    if np.std(x) <= EPS or np.std(y) <= EPS:
        return np.nan
    return float(pearsonr(x, y)[0])


def _nanmean_safe(xs):
    xs = np.asarray(xs, dtype=np.float64)
    if not np.isfinite(xs).any():
        return np.nan
    return float(np.nanmean(xs))


def _unit(v: torch.Tensor) -> torch.Tensor:
    v = v.float()
    return v / v.norm().clamp_min(1e-8)


def get_mlp_residual_write_module(model, layer_idx: int):
    """
    Return the module whose output is the MLP-path residual write.

    For Llama/Qwen-style blocks, this is usually the MLP output itself.
    For Gemma-2-style blocks, the MLP output is followed by a post-feedforward
    layernorm before being added to the residual stream, so we hook that output
    instead. This makes the measured vector closer to the actual residual write.
    """
    block = model.get_submodule(f"model.layers.{layer_idx}")

    for name in [
        "post_feedforward_layernorm",
        "post_mlp_layernorm",
        "post_ffn_layernorm",
    ]:
        if hasattr(block, name):
            return getattr(block, name), name

    if hasattr(block, "mlp"):
        return block.mlp, "mlp"

    raise AttributeError(f"Could not find an MLP residual-write module for layer {layer_idx}.")


@torch.no_grad()
def sae_single_feature_pre_activation(
    sae,
    x: torch.Tensor,
    feature_idx: int,
    filter_high_norm_tokens: bool = FILTER_HIGH_NORM_TOKENS_FOR_FEATURE_EXTRACTION,
):
    """
    Memory-light single-feature version of sae_logits_before_jumprelu.

    Args:
      x: [batch, seq, d_model]

    Returns:
      pre: [batch, seq]
    """
    if feature_idx < 0 or feature_idx >= sae.n_features:
        raise ValueError(f"feature_idx={feature_idx} outside [0, {sae.n_features}).")

    x_sae = x.to(device=sae.device, dtype=sae.dtype)
    x_centered = sae.process_sae_in(x_sae)

    w = sae.W_enc[:, feature_idx]
    b = sae.b_enc[feature_idx]
    pre = x_centered @ w + b

    if filter_high_norm_tokens and x_sae.ndim >= 3:
        norms = x_sae.detach().float().norm(dim=-1)
        med = norms.median(dim=1, keepdim=True).values.clamp_min(1e-6)
        high_norm_mask = norms > (HIGH_NORM_FILTER_MULTIPLIER * med)
        pre = pre.masked_fill(high_norm_mask.to(pre.device), 0.0)

    return pre


@torch.no_grad()
def capture_layer_output_and_mlp_write_output(model, input_ids, attention_mask, layer_idx: int):
    """
    One forward pass that captures:
      - output of model.layers[layer_idx], shape [B, S, d_model]
      - MLP-path residual write output, shape [B, S, d_model]

    The captured MLP-path vector is projected onto the SAE read/write directions.
    """
    cache = {}

    block = model.get_submodule(f"model.layers.{layer_idx}")
    mlp_write_module, mlp_write_module_name = get_mlp_residual_write_module(model, layer_idx)

    def block_hook(module, inputs, output):
        x = output[0] if isinstance(output, tuple) else output
        cache["layer_output"] = x.detach()
        return output

    def mlp_write_hook(module, inputs, output):
        x = output[0] if isinstance(output, tuple) else output
        cache["mlp_write_output"] = x.detach()
        return output

    h1 = block.register_forward_hook(block_hook)
    h2 = mlp_write_module.register_forward_hook(mlp_write_hook)

    try:
        _ = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
        )
    finally:
        h1.remove()
        h2.remove()

    if "layer_output" not in cache:
        raise RuntimeError(f"Failed to capture layer output for layer {layer_idx}.")
    if "mlp_write_output" not in cache:
        raise RuntimeError(
            f"Failed to capture MLP write output for layer {layer_idx} "
            f"from module {mlp_write_module_name}."
        )

    cache["mlp_write_module_name"] = mlp_write_module_name
    return cache["layer_output"], cache["mlp_write_output"], mlp_write_module_name


@torch.no_grad()
def get_feature_read_write_dirs(sae, feature_idx: int, device):
    read_dir = _unit(sae.W_enc[:, feature_idx].detach().to(device=device))
    write_dir = _unit(sae.W_dec[feature_idx].detach().to(device=device))
    return read_dir.float(), write_dir.float()


@torch.no_grad()
def measure_prompt_feature_and_mlp_write(
    prompt: str,
    feature_idx: int,
    layer_idx: int,
    previous_block_index: int = -2,
    target_block_index: int = -1,
    expected_num_blocks: int = NUM_BLOCKS_TOTAL,
):
    """
    For one prompt variant, measure:
      - mean preactivation of feature_idx on source block;
      - mean preactivation of feature_idx on target block;
      - mean signed MLP residual-path write on target block along SAE read/write directions.
    """
    enc = tokenize_prompt_for_offsets(prompt)
    offset_mapping = enc["offset_mapping"][0].tolist()
    input_ids = enc["input_ids"].to(MODEL_INPUT_DEVICE)
    attention_mask = enc["attention_mask"].to(MODEL_INPUT_DEVICE)

    block_spans = get_example_block_token_spans(
        prompt=prompt,
        offset_mapping=offset_mapping,
        expected_num_blocks=expected_num_blocks,
    )

    prev_span = block_spans[previous_block_index]
    tgt_span = block_spans[target_block_index]

    layer_output, mlp_write_output, mlp_write_module_name = capture_layer_output_and_mlp_write_output(
        model=model,
        input_ids=input_ids,
        attention_mask=attention_mask,
        layer_idx=layer_idx,
    )

    pre_f = sae_single_feature_pre_activation(
        sae=sae,
        x=layer_output,
        feature_idx=feature_idx,
    )[0].detach().float()

    prev_idx = token_selection_to_torch_indices(prev_span, device=pre_f.device)
    tgt_idx = token_selection_to_torch_indices(tgt_span, device=pre_f.device)

    prev_feature_mean = float(pre_f.index_select(0, prev_idx).mean().detach().cpu())
    target_feature_mean = float(pre_f.index_select(0, tgt_idx).mean().detach().cpu())

    read_dir, write_dir = get_feature_read_write_dirs(
        sae=sae,
        feature_idx=feature_idx,
        device=mlp_write_output.device,
    )

    z = mlp_write_output[0].index_select(
        0,
        token_selection_to_torch_indices(tgt_span, device=mlp_write_output.device),
    ).float()

    if z.shape[-1] != read_dir.numel():
        raise ValueError(
            f"MLP write dim {z.shape[-1]} does not match SAE direction dim {read_dir.numel()}."
        )

    target_mlp_write_to_read = float((z @ read_dir).mean().detach().cpu())
    target_mlp_write_to_write = float((z @ write_dir).mean().detach().cpu())

    return {
        "prev_feature_mean": prev_feature_mean,
        "target_feature_mean": target_feature_mean,
        "target_mlp_write_to_read": target_mlp_write_to_read,
        "target_mlp_write_to_write": target_mlp_write_to_write,
        "prev_span": prev_span,
        "target_span": tgt_span,
        "mlp_write_module_name": mlp_write_module_name,
    }


def _summarize_mlp_write_test(prompt_records_df, mlp_read_vec, mlp_write_vec, layer_idx: int, feature_idx: int):
    """
    Summarize MLP-level evidence for:
      high source activation -> negative target-block MLP write.
    """
    x_all = prompt_records_df["prev_feature_mean"].to_numpy(dtype=np.float64)
    y_tgt_all = prompt_records_df["target_feature_mean"].to_numpy(dtype=np.float64)
    q_all = prompt_records_df["query_idx"].to_numpy()

    unique_queries = sorted(prompt_records_df["query_idx"].unique().tolist())

    feature_corr_pooled = _safe_pearson(x_all, y_tgt_all)

    feature_corr_by_query = []
    for q in unique_queries:
        mask = q_all == q
        feature_corr_by_query.append(_safe_pearson(x_all[mask], y_tgt_all[mask]))
    feature_corr_mean_query = _nanmean_safe(feature_corr_by_query)

    corr_prev_to_read_by_q = []
    corr_prev_to_write_by_q = []
    corr_read_to_target_by_q = []
    corr_write_to_target_by_q = []

    delta_read_by_q = []
    delta_write_by_q = []
    delta_read_z_by_q = []
    delta_write_z_by_q = []

    for q in unique_queries:
        mask = q_all == q
        x = x_all[mask]
        y_read = mlp_read_vec[mask]
        y_write = mlp_write_vec[mask]
        y_tgt = y_tgt_all[mask]

        corr_prev_to_read_by_q.append(_safe_pearson(x, y_read))
        corr_prev_to_write_by_q.append(_safe_pearson(x, y_write))
        corr_read_to_target_by_q.append(_safe_pearson(y_read, y_tgt))
        corr_write_to_target_by_q.append(_safe_pearson(y_write, y_tgt))

        finite = np.isfinite(x) & np.isfinite(y_read) & np.isfinite(y_write)
        if finite.sum() >= 4:
            xf = x[finite]
            yr = y_read[finite]
            yw = y_write[finite]

            q_low = np.quantile(xf, MLP_WRITE_TEST_QUANTILE)
            q_high = np.quantile(xf, 1.0 - MLP_WRITE_TEST_QUANTILE)
            low_mask = xf <= q_low
            high_mask = xf >= q_high

            if low_mask.sum() > 0 and high_mask.sum() > 0:
                dr = float(np.mean(yr[high_mask]) - np.mean(yr[low_mask]))
                dw = float(np.mean(yw[high_mask]) - np.mean(yw[low_mask]))
                delta_read_by_q.append(dr)
                delta_write_by_q.append(dw)
                delta_read_z_by_q.append(dr / (float(np.std(yr)) + EPS))
                delta_write_z_by_q.append(dw / (float(np.std(yw)) + EPS))
            else:
                delta_read_by_q.append(np.nan)
                delta_write_by_q.append(np.nan)
                delta_read_z_by_q.append(np.nan)
                delta_write_z_by_q.append(np.nan)
        else:
            delta_read_by_q.append(np.nan)
            delta_write_by_q.append(np.nan)
            delta_read_z_by_q.append(np.nan)
            delta_write_z_by_q.append(np.nan)

    corr_prev_to_read_mean = _nanmean_safe(corr_prev_to_read_by_q)
    corr_prev_to_write_mean = _nanmean_safe(corr_prev_to_write_by_q)
    corr_read_to_target_mean = _nanmean_safe(corr_read_to_target_by_q)
    corr_write_to_target_mean = _nanmean_safe(corr_write_to_target_by_q)

    delta_read_mean = _nanmean_safe(delta_read_by_q)
    delta_write_mean = _nanmean_safe(delta_write_by_q)
    delta_read_z_mean = _nanmean_safe(delta_read_z_by_q)
    delta_write_z_mean = _nanmean_safe(delta_write_z_by_q)

    anti_write_score_read = (
        max(-corr_prev_to_read_mean, 0.0)
        * max(-delta_read_z_mean, 0.0)
        * (0.5 + 0.5 * max(corr_read_to_target_mean, 0.0))
    )
    anti_write_score_write = (
        max(-corr_prev_to_write_mean, 0.0)
        * max(-delta_write_z_mean, 0.0)
        * (0.5 + 0.5 * max(corr_write_to_target_mean, 0.0))
    )

    row = {
        "layer": int(layer_idx),
        "component": "mlp",
        "feature_idx": int(feature_idx),

        "feature_prev_to_target_corr_pooled": feature_corr_pooled,
        "feature_prev_to_target_corr_mean_query": feature_corr_mean_query,

        "corr_prev_to_target_mlp_write_to_read_mean_query": corr_prev_to_read_mean,
        "corr_prev_to_target_mlp_write_to_write_mean_query": corr_prev_to_write_mean,

        "corr_prev_to_target_mlp_write_to_negative_read_mean_query": -corr_prev_to_read_mean,
        "corr_prev_to_target_mlp_write_to_negative_write_mean_query": -corr_prev_to_write_mean,

        "high_minus_low_mlp_write_to_read_mean": delta_read_mean,
        "high_minus_low_mlp_write_to_write_mean": delta_write_mean,
        "high_minus_low_mlp_write_to_read_z_mean": delta_read_z_mean,
        "high_minus_low_mlp_write_to_write_z_mean": delta_write_z_mean,

        "high_minus_low_mlp_write_to_negative_read_z_mean": -delta_read_z_mean,
        "high_minus_low_mlp_write_to_negative_write_z_mean": -delta_write_z_mean,

        "corr_mlp_write_to_read_with_target_feature_mean_query": corr_read_to_target_mean,
        "corr_mlp_write_to_write_with_target_feature_mean_query": corr_write_to_target_mean,

        "anti_write_score_read": anti_write_score_read,
        "anti_write_score_write": anti_write_score_write,
        "anti_write_score_max": max(anti_write_score_read, anti_write_score_write),
    }

    return pd.DataFrame([row])


@torch.no_grad()
def run_mlp_negative_write_test_for_feature(
    feature_idx: int,
    prompts_all_agnews,
    layer_idx: int = TARGET_LAYER,
    previous_block_index: int = -2,
    target_block_index: int = -1,
    max_queries=None,
    max_variants=None,
    verbose_every: int = 10,
):
    """
    Main MLP write test.

    For every selected prompt variant:
      x = mean feature preactivation on source block
      y = mean target-block MLP residual-path write along feature direction

    MLP support for the negative-write mechanism means:
      corr(x, y) < 0
      mean(y | x high) - mean(y | x low) < 0
    """
    num_queries_total = len(prompts_all_agnews)
    query_indices = list(range(num_queries_total if max_queries is None else min(max_queries, num_queries_total)))

    prompt_rows = []
    read_vals = []
    write_vals = []

    for query_idx in query_indices:
        variants = prompts_all_agnews[query_idx]
        if max_variants is not None:
            variants = variants[:max_variants]

        for variant_idx, prompt in enumerate(variants):
            m = measure_prompt_feature_and_mlp_write(
                prompt=prompt,
                feature_idx=feature_idx,
                layer_idx=layer_idx,
                previous_block_index=previous_block_index,
                target_block_index=target_block_index,
                expected_num_blocks=NUM_BLOCKS_TOTAL,
            )

            prompt_rows.append({
                "query_idx": int(query_idx),
                "variant_idx": int(variant_idx),
                "feature_idx": int(feature_idx),
                "layer": int(layer_idx),
                "prev_feature_mean": m["prev_feature_mean"],
                "target_feature_mean": m["target_feature_mean"],
                "prev_span_start": int(np.min(token_selection_to_torch_indices(m["prev_span"], device="cpu").numpy())),
                "prev_span_end": int(np.max(token_selection_to_torch_indices(m["prev_span"], device="cpu").numpy())) + 1,
                "target_span_start": int(np.min(token_selection_to_torch_indices(m["target_span"], device="cpu").numpy())),
                "target_span_end": int(np.max(token_selection_to_torch_indices(m["target_span"], device="cpu").numpy())) + 1,
                "mlp_write_module_name": m["mlp_write_module_name"],
            })
            read_vals.append(m["target_mlp_write_to_read"])
            write_vals.append(m["target_mlp_write_to_write"])

        if verbose_every is not None:
            if len(prompt_rows) % max(int(verbose_every), 1) == 0 or query_idx == query_indices[-1]:
                print(f"processed query {query_idx + 1}/{query_indices[-1] + 1}; prompt rows={len(prompt_rows)}")

    prompt_records_df = pd.DataFrame(prompt_rows)
    mlp_read_vec = np.asarray(read_vals, dtype=np.float64)
    mlp_write_vec = np.asarray(write_vals, dtype=np.float64)

    mlp_summary_df = _summarize_mlp_write_test(
        prompt_records_df=prompt_records_df,
        mlp_read_vec=mlp_read_vec,
        mlp_write_vec=mlp_write_vec,
        layer_idx=layer_idx,
        feature_idx=feature_idx,
    )

    print("=" * 120)
    print(f"Layer-{layer_idx} MLP negative-write test for feature {feature_idx}")
    print("Expected mechanism: high source-block feature activation -> negative target-block MLP write")
    print("Rows/prompts measured:", len(prompt_records_df))
    print("MLP write module:", prompt_records_df["mlp_write_module_name"].iloc[0])
    print("Feature prev->target corr, pooled:", mlp_summary_df["feature_prev_to_target_corr_pooled"].iloc[0])
    print("Feature prev->target corr, mean query-wise:", mlp_summary_df["feature_prev_to_target_corr_mean_query"].iloc[0])
    print("=" * 120)

    display(mlp_summary_df)

    return mlp_summary_df, prompt_records_df, mlp_read_vec, mlp_write_vec


def plot_mlp_write_scatter(
    prompt_records_df,
    mlp_vec,
    feature_idx: int,
    layer_idx: int,
    direction_name: str = "SAE read",
):
    """
    Scatter: source feature mean vs target-block MLP signed write.
    A negative slope supports the proposed MLP negative-write mechanism.
    """
    x = prompt_records_df["prev_feature_mean"].to_numpy(dtype=np.float64)
    y = np.asarray(mlp_vec, dtype=np.float64)
    q = prompt_records_df["query_idx"].to_numpy()

    r = _safe_pearson(x, y)
    if np.isfinite(r) and np.std(x) > 0 and np.std(y) > 0:
        slope, intercept = np.polyfit(x, y, 1)
        xx = np.linspace(np.min(x), np.max(x), 200)
        yy = slope * xx + intercept
    else:
        slope, intercept = np.nan, np.nan
        xx, yy = np.array([]), np.array([])

    df_plot = pd.DataFrame({
        "prev_feature_mean": x,
        "target_mlp_write": y,
        "query_idx": q.astype(str),
        "variant_idx": prompt_records_df["variant_idx"].to_numpy(),
        "target_feature_mean": prompt_records_df["target_feature_mean"].to_numpy(),
    })

    fig = px.scatter(
        df_plot,
        x="prev_feature_mean",
        y="target_mlp_write",
        color="query_idx",
        hover_data=["variant_idx", "target_feature_mean"],
        title=(
            f"Layer {layer_idx} MLP: source feature activation vs target-block MLP write<br>"
            f"feature={feature_idx}, direction={direction_name}, pooled r={r:.3f}, slope={slope:.3f}"
        ),
    )

    if len(xx) > 0:
        fig.add_trace(go.Scatter(
            x=xx,
            y=yy,
            mode="lines",
            name="pooled linear fit",
            line=dict(width=3),
        ))

    fig.add_hline(y=0, line_dash="dash", opacity=0.4)
    fig.update_layout(
        width=900,
        height=600,
        xaxis_title="Mean SAE pre-activation on source demonstration block",
        yaxis_title=f"Mean MLP write to {direction_name} direction on target block",
    )
    fig.show()
    return fig


# ------------------------------------------------------------
# Run the MLP write test
# ------------------------------------------------------------

mlp_summary_df, mlp_prompt_records_df, mlp_read_vec, mlp_write_vec = run_mlp_negative_write_test_for_feature(
    feature_idx=MLP_WRITE_TEST_FEATURE_IDX,
    prompts_all_agnews=prompts_all_agnews,
    layer_idx=MLP_WRITE_TEST_LAYER,
    previous_block_index=MLP_WRITE_TEST_PREVIOUS_BLOCK_INDEX,
    target_block_index=MLP_WRITE_TEST_TARGET_BLOCK_INDEX,
    max_queries=MLP_WRITE_TEST_MAX_QUERIES,
    max_variants=MLP_WRITE_TEST_MAX_VARIANTS,
    verbose_every=10,
)

_ = plot_mlp_write_scatter(
    prompt_records_df=mlp_prompt_records_df,
    mlp_vec=mlp_read_vec,
    feature_idx=MLP_WRITE_TEST_FEATURE_IDX,
    layer_idx=MLP_WRITE_TEST_LAYER,
    direction_name="SAE read",
)

# 13B. Compare attention heads and MLP write strength

In [ ]:
# ============================================================
# 13B. Compare attention-head negative writes vs MLP negative write
# ============================================================

def build_attention_mlp_comparison_df(head_summary_df, mlp_summary_df):
    """
    Compare:
      - top attention head by read-direction score
      - top attention head by write-direction score
      - top attention head by max score
      - MLP residual-path write
    """
    rows = []

    def add_head_row(label, row):
        rows.append({
            "component_label": label,
            "component_type": "attention_head",
            "layer": int(row["layer"]),
            "component_id": int(row["head"]),
            "feature_idx": int(row["feature_idx"]),

            "feature_prev_to_target_corr_mean_query": float(row["feature_prev_to_target_corr_mean_query"]),

            "neg_corr_to_read_mean_query": float(row["corr_prev_to_target_head_write_to_negative_read_mean_query"]),
            "neg_corr_to_write_mean_query": float(row["corr_prev_to_target_head_write_to_negative_write_mean_query"]),

            "neg_high_minus_low_read_z": float(row["high_minus_low_head_write_to_negative_read_z_mean"]),
            "neg_high_minus_low_write_z": float(row["high_minus_low_head_write_to_negative_write_z_mean"]),

            "write_to_read_target_alignment": float(row["corr_head_write_to_read_with_target_feature_mean_query"]),
            "write_to_write_target_alignment": float(row["corr_head_write_to_write_with_target_feature_mean_query"]),

            "anti_write_score_read": float(row["anti_write_score_read"]),
            "anti_write_score_write": float(row["anti_write_score_write"]),
            "anti_write_score_max": float(row["anti_write_score_max"]),
        })

    def add_mlp_row(row):
        rows.append({
            "component_label": "MLP residual-path write",
            "component_type": "mlp",
            "layer": int(row["layer"]),
            "component_id": -1,
            "feature_idx": int(row["feature_idx"]),

            "feature_prev_to_target_corr_mean_query": float(row["feature_prev_to_target_corr_mean_query"]),

            "neg_corr_to_read_mean_query": float(row["corr_prev_to_target_mlp_write_to_negative_read_mean_query"]),
            "neg_corr_to_write_mean_query": float(row["corr_prev_to_target_mlp_write_to_negative_write_mean_query"]),

            "neg_high_minus_low_read_z": float(row["high_minus_low_mlp_write_to_negative_read_z_mean"]),
            "neg_high_minus_low_write_z": float(row["high_minus_low_mlp_write_to_negative_write_z_mean"]),

            "write_to_read_target_alignment": float(row["corr_mlp_write_to_read_with_target_feature_mean_query"]),
            "write_to_write_target_alignment": float(row["corr_mlp_write_to_write_with_target_feature_mean_query"]),

            "anti_write_score_read": float(row["anti_write_score_read"]),
            "anti_write_score_write": float(row["anti_write_score_write"]),
            "anti_write_score_max": float(row["anti_write_score_max"]),
        })

    top_head_read = head_summary_df.sort_values("anti_write_score_read", ascending=False).iloc[0]
    top_head_write = head_summary_df.sort_values("anti_write_score_write", ascending=False).iloc[0]
    top_head_max = head_summary_df.sort_values("anti_write_score_max", ascending=False).iloc[0]
    mlp_row = mlp_summary_df.iloc[0]

    add_head_row(f"Top attention head by read score: H{int(top_head_read['head'])}", top_head_read)
    add_head_row(f"Top attention head by write score: H{int(top_head_write['head'])}", top_head_write)
    add_head_row(f"Top attention head by max score: H{int(top_head_max['head'])}", top_head_max)
    add_mlp_row(mlp_row)

    df = pd.DataFrame(rows)

    # Keep duplicate heads if a single head is top under multiple criteria;
    # this makes it clear which criterion selected it.
    return df


attention_mlp_comparison_df = build_attention_mlp_comparison_df(
    head_summary_df=head_summary_df,
    mlp_summary_df=mlp_summary_df,
)

display(attention_mlp_comparison_df)

# Plot the combined anti-write scores.
comparison_long_df = attention_mlp_comparison_df.melt(
    id_vars=["component_label", "component_type"],
    value_vars=["anti_write_score_read", "anti_write_score_write", "anti_write_score_max"],
    var_name="metric",
    value_name="score",
)

fig = px.bar(
    comparison_long_df,
    x="component_label",
    y="score",
    color="metric",
    barmode="group",
    title=(
        f"Attention vs MLP negative-write evidence<br>"
        f"feature={MLP_WRITE_TEST_FEATURE_IDX}, layer={MLP_WRITE_TEST_LAYER}"
    ),
)
fig.update_layout(
    width=1100,
    height=550,
    xaxis_title="Component",
    yaxis_title="Negative-write score",
)
fig.show()

# Optional: compare only the direct negative correlations.
corr_long_df = attention_mlp_comparison_df.melt(
    id_vars=["component_label", "component_type"],
    value_vars=["neg_corr_to_read_mean_query", "neg_corr_to_write_mean_query"],
    var_name="metric",
    value_name="negative_correlation_strength",
)

fig = px.bar(
    corr_long_df,
    x="component_label",
    y="negative_correlation_strength",
    color="metric",
    barmode="group",
    title=(
        f"Attention vs MLP: source feature activation -> negative target write<br>"
        f"feature={MLP_WRITE_TEST_FEATURE_IDX}, layer={MLP_WRITE_TEST_LAYER}"
    ),
)
fig.update_layout(
    width=1100,
    height=550,
    xaxis_title="Component",
    yaxis_title="-corr(source feature, target component write)",
)
fig.show()

# 14B. Causal MLP intervention test

In [ ]:
# ============================================================
# 14B. Causal MLP intervention test
# ============================================================

import numpy as np
import pandas as pd
import torch
import plotly.express as px

# ------------------------------------------------------------
# User-editable config
# ------------------------------------------------------------

MLP_CAUSAL_LAYER = TARGET_LAYER
MLP_CAUSAL_FEATURE_IDX = int(MLP_WRITE_TEST_FEATURE_IDX)

MLP_CAUSAL_PREVIOUS_BLOCK_INDEX = int(MLP_WRITE_TEST_PREVIOUS_BLOCK_INDEX)
MLP_CAUSAL_TARGET_BLOCK_INDEX = int(MLP_WRITE_TEST_TARGET_BLOCK_INDEX)

MLP_CAUSAL_MAX_QUERIES = min(10, len(prompts_all_agnews))
MLP_CAUSAL_MAX_VARIANTS = None

MLP_CAUSAL_ALPHAS = [-10.0, -5.0, -1.0, 0.0, 1.0, 5.0, 10.0]


@torch.no_grad()
def capture_layer_output_with_mlp_write_intervention(
    model,
    input_ids,
    attention_mask,
    layer_idx: int,
    target_token_selection,
    alpha: float,
):
    """
    Capture layer output while scaling the MLP residual-path write on selected target tokens.

    Effective multiplier on selected target-token MLP write:
      multiplier = 1 + alpha

    Therefore:
      alpha = 0  -> baseline
      alpha = -1 -> ablation
      alpha = 1  -> double the MLP write
    """
    cache = {}

    block = model.get_submodule(f"model.layers.{layer_idx}")
    mlp_write_module, mlp_write_module_name = get_mlp_residual_write_module(model, layer_idx)

    def mlp_write_intervention_hook(module, inputs, output):
        if isinstance(output, tuple):
            x = output[0]
            rest = output[1:]
        else:
            x = output
            rest = None

        x_new = x.clone()
        idx = token_selection_to_torch_indices(target_token_selection, device=x_new.device)

        if idx.numel() > 0:
            multiplier = 1.0 + float(alpha)
            x_new[0, idx, :] = x_new[0, idx, :] * multiplier

        if rest is not None:
            return (x_new,) + rest
        return x_new

    def block_hook(module, inputs, output):
        x = output[0] if isinstance(output, tuple) else output
        cache["layer_output"] = x.detach()
        return output

    h1 = mlp_write_module.register_forward_hook(mlp_write_intervention_hook)
    h2 = block.register_forward_hook(block_hook)

    try:
        _ = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
        )
    finally:
        h1.remove()
        h2.remove()

    if "layer_output" not in cache:
        raise RuntimeError(f"Failed to capture intervened layer output for layer {layer_idx}.")

    cache["mlp_write_module_name"] = mlp_write_module_name
    return cache["layer_output"], mlp_write_module_name


@torch.no_grad()
def measure_prompt_feature_with_mlp_write_alpha(
    prompt: str,
    feature_idx: int,
    layer_idx: int,
    alpha: float,
    previous_block_index: int = -2,
    target_block_index: int = -1,
    expected_num_blocks: int = NUM_BLOCKS_TOTAL,
):
    """
    Run one prompt with MLP write intervention and measure source/target feature means.
    """
    enc = tokenize_prompt_for_offsets(prompt)
    offset_mapping = enc["offset_mapping"][0].tolist()
    input_ids = enc["input_ids"].to(MODEL_INPUT_DEVICE)
    attention_mask = enc["attention_mask"].to(MODEL_INPUT_DEVICE)

    block_spans = get_example_block_token_spans(
        prompt=prompt,
        offset_mapping=offset_mapping,
        expected_num_blocks=expected_num_blocks,
    )

    prev_span = block_spans[previous_block_index]
    tgt_span = block_spans[target_block_index]

    layer_output, mlp_write_module_name = capture_layer_output_with_mlp_write_intervention(
        model=model,
        input_ids=input_ids,
        attention_mask=attention_mask,
        layer_idx=layer_idx,
        target_token_selection=tgt_span,
        alpha=float(alpha),
    )

    pre_f = sae_single_feature_pre_activation(
        sae=sae,
        x=layer_output,
        feature_idx=feature_idx,
    )[0].detach().float()

    prev_idx = token_selection_to_torch_indices(prev_span, device=pre_f.device)
    tgt_idx = token_selection_to_torch_indices(tgt_span, device=pre_f.device)

    prev_feature_mean = float(pre_f.index_select(0, prev_idx).mean().detach().cpu())
    target_feature_mean = float(pre_f.index_select(0, tgt_idx).mean().detach().cpu())

    return {
        "prev_feature_mean": prev_feature_mean,
        "target_feature_mean": target_feature_mean,
        "prev_span": prev_span,
        "target_span": tgt_span,
        "mlp_write_module_name": mlp_write_module_name,
    }


def summarize_mlp_causal_records(mlp_causal_records_df):
    rows = []

    for alpha, g in mlp_causal_records_df.groupby("alpha"):
        x = g["prev_feature_mean"].to_numpy(dtype=np.float64)
        y = g["target_feature_mean"].to_numpy(dtype=np.float64)
        q = g["query_idx"].to_numpy()

        pooled_r = _safe_pearson(x, y)

        query_rs = []
        for q_idx in sorted(g["query_idx"].unique().tolist()):
            mask = q == q_idx
            query_rs.append(_safe_pearson(x[mask], y[mask]))

        rows.append({
            "alpha": float(alpha),
            "effective_multiplier": 1.0 + float(alpha),
            "n_prompts": int(len(g)),
            "feature_prev_to_target_corr_pooled": pooled_r,
            "feature_prev_to_target_corr_mean_query": _nanmean_safe(query_rs),
            "feature_prev_to_target_corr_median_query": float(np.nanmedian(query_rs)),
            "target_feature_mean": float(np.nanmean(y)),
            "target_feature_std": float(np.nanstd(y)),
            "mlp_write_module_name": g["mlp_write_module_name"].iloc[0],
        })

    summary_df = pd.DataFrame(rows).sort_values("alpha").reset_index(drop=True)

    if (summary_df["alpha"] == 0.0).any():
        base = summary_df.loc[summary_df["alpha"] == 0.0].iloc[0]
        summary_df["delta_corr_mean_query_vs_alpha0"] = (
            summary_df["feature_prev_to_target_corr_mean_query"]
            - float(base["feature_prev_to_target_corr_mean_query"])
        )
        summary_df["delta_corr_pooled_vs_alpha0"] = (
            summary_df["feature_prev_to_target_corr_pooled"]
            - float(base["feature_prev_to_target_corr_pooled"])
        )
        summary_df["delta_target_feature_mean_vs_alpha0"] = (
            summary_df["target_feature_mean"]
            - float(base["target_feature_mean"])
        )

    return summary_df


@torch.no_grad()
def run_mlp_causal_intervention_test_for_feature(
    feature_idx: int,
    prompts_all_agnews,
    layer_idx: int = TARGET_LAYER,
    previous_block_index: int = -2,
    target_block_index: int = -1,
    alphas = (-1.0, 0.0, 1.0),
    max_queries=None,
    max_variants=None,
    verbose_every: int = 50,
):
    """
    Causal test for MLP write-side contribution to LFH.

    For each alpha:
      - scale the target-token MLP residual-path write by 1 + alpha;
      - remeasure corr(source feature, target feature).

    Expected pattern if MLP write helps implement LFH:
      alpha = -1 should weaken the negative source->target correlation;
      alpha > 0 should strengthen the negative source->target correlation.
    """
    num_queries_total = len(prompts_all_agnews)
    query_indices = list(range(num_queries_total if max_queries is None else min(max_queries, num_queries_total)))

    rows = []

    for alpha in alphas:
        print("=" * 100)
        print(f"Running MLP causal intervention: alpha={float(alpha):.3f}, multiplier={1.0 + float(alpha):.3f}")
        print("=" * 100)

        n_done = 0

        for query_idx in query_indices:
            variants = prompts_all_agnews[query_idx]
            if max_variants is not None:
                variants = variants[:max_variants]

            for variant_idx, prompt in enumerate(variants):
                m = measure_prompt_feature_with_mlp_write_alpha(
                    prompt=prompt,
                    feature_idx=feature_idx,
                    layer_idx=layer_idx,
                    alpha=float(alpha),
                    previous_block_index=previous_block_index,
                    target_block_index=target_block_index,
                    expected_num_blocks=NUM_BLOCKS_TOTAL,
                )

                rows.append({
                    "alpha": float(alpha),
                    "effective_multiplier": 1.0 + float(alpha),
                    "query_idx": int(query_idx),
                    "variant_idx": int(variant_idx),
                    "feature_idx": int(feature_idx),
                    "layer": int(layer_idx),
                    "prev_feature_mean": m["prev_feature_mean"],
                    "target_feature_mean": m["target_feature_mean"],
                    "mlp_write_module_name": m["mlp_write_module_name"],
                })

                n_done += 1
                if verbose_every is not None and n_done % max(int(verbose_every), 1) == 0:
                    print(f"alpha={alpha}: processed {n_done} prompt rows")

        print(f"alpha={alpha}: processed total {n_done} prompt rows")

    records_df = pd.DataFrame(rows)
    summary_df = summarize_mlp_causal_records(records_df)

    print("=" * 120)
    print(f"Layer-{layer_idx} MLP causal intervention test for feature {feature_idx}")
    print("Intervention: target-token MLP residual-path write *= 1 + alpha")
    print("=" * 120)
    display(summary_df)

    fig = px.line(
        summary_df,
        x="alpha",
        y="feature_prev_to_target_corr_mean_query",
        markers=True,
        title=(
            f"MLP causal intervention: source-to-target LFH correlation<br>"
            f"feature={feature_idx}, layer={layer_idx}"
        ),
    )
    fig.add_hline(y=0, line_dash="dash", opacity=0.4)
    fig.update_layout(
        font=dict(family="Times New Roman"),
        title_font_size=26,
        width=900,
        height=600,
        xaxis_title="alpha, where effective MLP multiplier = 1 + alpha",
        xaxis_title_font_size=22,
        yaxis_title="Mean query-wise corr(source feature, target feature)",
        yaxis_title_font_size=22,
    )

    # --- ADDED: 4x Resolution Config for Interactive 'Download Plot' Button ---
    config = {
        'toImageButtonOptions': {
            'format': 'png',         # one of png, svg, jpeg, webp
            'filename': 'mlp_causal_intervention',
            'height': 600,
            'width': 900,
            'scale': 4               # Multiply resolution by 4
        }
    }

    fig.show(config=config)


    return summary_df, records_df


# ------------------------------------------------------------
# Run the MLP causal test
# ------------------------------------------------------------

mlp_causal_summary_df, mlp_causal_records_df = run_mlp_causal_intervention_test_for_feature(
    feature_idx=MLP_CAUSAL_FEATURE_IDX,
    prompts_all_agnews=prompts_all_agnews,
    layer_idx=MLP_CAUSAL_LAYER,
    previous_block_index=MLP_CAUSAL_PREVIOUS_BLOCK_INDEX,
    target_block_index=MLP_CAUSAL_TARGET_BLOCK_INDEX,
    alphas=MLP_CAUSAL_ALPHAS,
    max_queries=MLP_CAUSAL_MAX_QUERIES,
    max_variants=MLP_CAUSAL_MAX_VARIANTS,
    verbose_every=50,
)